This file is just for data cleaning operations removing nana removing cols that dont make sense and so on this will be exracted from the file pessoa_seg.ipynb  but here i will do it a a cleaner and more better structured version 




# libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
import sys
import re
import seaborn as sns
import pprint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from datetime import date

from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import dendrogram

from sklearn.metrics import silhouette_score
import warnings

from scipy.cluster.hierarchy import linkage, cophenet, fcluster

import warnings
warnings.filterwarnings("ignore")


from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance


# Functions

In [2]:
def fillna_by_postcode_mode_vectorized(df, cols, postcode_col='codpost_tom'):
    df = df.copy()
    for col in cols:
        filled = pd.Series([False] * len(df), index=df.index)
        for n in [8, 4, 3, 2, 1]:
            prefix = df[postcode_col].astype(str).str[:n]
            mode_map = (
                df.loc[~df[col].isna()]
                .groupby(prefix)[col]
                .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
            )
            mask = df[col].isna() & ~filled
            fill_prefix = prefix[mask]
            fill_values = fill_prefix.map(mode_map)
            df.loc[mask, col] = fill_values
            filled = filled | mask & fill_values.notna()
    return df

# loading the lapses and score bd

In [3]:
lapses = pd.read_parquet("lapses1.parquet")
scores = pd.read_excel("SAUDE_WO_NIF_2_w_cv.xlsx")


In [4]:
#pass the cols to date format
date_cols = ["fefecto_poliza", "fvto_ac", "fbajaaseg", "fecha_cv1", "fecha_cv2"]
for col in date_cols:
    scores[col] = pd.to_datetime(scores[col], errors='coerce').dt.date

In [5]:
# Create lookup tables with unique codpost_tom index from lapses


# Get the first non-NaN value for each codpost_tom
concelho_map = lapses.groupby("codpost_tom")["Concelho"].apply(lambda x: x.dropna().iloc[0] if not x.dropna().empty else np.nan)
distrito_map = lapses.groupby("codpost_tom")["Distrito"].apply(lambda x: x.dropna().iloc[0] if not x.dropna().empty else np.nan)

# Mask for rows where Concelho or Distrito is NaN
mask_concelho = lapses["Concelho"].isna()
mask_distrito = lapses["Distrito"].isna()

# Fill NaNs in Concelho and Distrito using the lookup tables
lapses.loc[mask_concelho, "Concelho"] = lapses.loc[mask_concelho, "codpost_tom"].map(concelho_map)
lapses.loc[mask_distrito, "Distrito"] = lapses.loc[mask_distrito, "codpost_tom"].map(distrito_map)

In [6]:
lapses = fillna_by_postcode_mode_vectorized(lapses, ["Concelho", "Distrito"], postcode_col="codpost_tom")

# variables description


IN THIS MARKDOWN CHUNK I WILL DESCRIBE THE VARIABLES I HAVE IN THE LAPSES DF

"<mark>POLIZA</mark>": number of the POLICY is a surrogatory key for each apólice 

"<mark>NASEGURADO</mark>": tels me wich number the assegurado takes inside the policy ex 1,2,3,4 

"<mark>ASEGURADO</mark>": its a number that uniquelly identifies the pessoa segura in the company so it a surrogatory key 

"<mark>SEXO</mark>": tels me the sex of the pessoa segura H(HEMBRA): women, V(VARÓN) :men 

"<mark>PARENTESCO</mark>": its an encoded variable telling me the relationship with tomador im not fully aware of the encoding but so far i know 
0 is the tomador
1 is the conjuge
3 sun or doughter
9 is the only one i dont know it may be godson or something like that

"<mark>FALTAASEG</mark>": date when the coverage started its the same as fefecto_poliza but in diferent format dd/mm/yyyy

"<mark>FNACI8</mark>": birth date of the individual format dd/mm/yyyy

"<mark>POLIZAS_45</mark>":  binari variable im stil not sure what it does indicates 

"<mark>ESTATURA</mark>": hight of the person in cm  there is some data inconsistency

"<mark>PESO</mark>": the wheight of the person there is some inconsistencies as well 

"<mark>TRANSF_SEGURO_INT</mark>": binary variable indicating that this policy was old policy from the same company that was changed N: no , Y: yes

"<mark>TRANSF_SEGURO_EXT</mark>" :binary variable indicating that this policy was previous from another company N: no , Y: yes

"<mark>FECHA_PREEXISTENCIA</mark>": it indicates a date but im not sure what date maybe the date where the medical prof was made for the ones that are missing im ginna imput the value of the fefecto

"<mark>DESCGARA</mark>": TELS ME WHAT TYPE OF COVERAGE (PRODUCT) IS CONTRACTED IS A STR I SHALL PASS TO OBJ the nan im gona pass to the descgara of the 

"<mark>FEFECTO</mark>": it shall read fecha de efecto  but its not the same as fefecto-poliza

"<mark>ICORR</mark>": not sure what it is it shall be something like coverage index

"<mark>PRIANUALAC</mark>": premium  for the current year

"<mark>PRIANUALAA</mark>": premium of the previous year

"<mark>TOMADOR</mark>": internal number of the policy taker surrogatory key

"<mark>FP</mark>" : stands for forma de pagamento indicates the n opf instalements 1-6 from 1 to 6 anual payments

"<mark>FVTO_AC</mark>" : stands for fecha de vencimento indicates the date where the policy is renewed for the year of the row form(str) yyyy-mm-dd same as RENOVAÇÂO

"<mark>FCob</mark>" : this one im not sure it only has 3 classes "DACB" ,"fÃ­sico" , "f>sico" 

"<mark>fefecto_poliza</mark>" : date where the policy started same as FALTAASEG but in format yyyy-mm-dd

"<mark>prem_inicial</mark>" : its the same as  PRIANUALAC but fot the p_exp_year where the policy is canceled it has  no values

"<mark>RENOVACAO</mark>" : same as the FVTO_AC but in another format   (dd-mm-yyyy)

"<mark>fbajaaseg</mark>" : date when the policy is not renovated or canceled its one of the main variables

"<mark>INICIO</mark>" :  format ( dd/mm/yyyy) not the same as dt_start  but i dont know the diferences when there is no sinister ussualy the same

"<mark>FIM</mark>" :   format (dd/mm/yyyy) not the same as dt_end  but i dont know the diferences


"<mark>EXPOSICAO</mark>" :  a measure of exposition 1-  whole year form = (FIM-INICIO)/365

"<mark>min_faltaaseg</mark>" : i dont know its almoast always missing  it has nothing to do with lapses dd/mm/yyyy

"<mark>min_fecha</mark>" :  i dont know it has the same missing values as the above  format (dd/mm/yyyy)

"<mark>max_fbajaaseg</mark>" : dont know its also a date a lot of nan 99%  but is not on a date format

"<mark>FECHA_novo</mark>" :  its also  a date but i dont know what it means also 99% nan format (dd/mm/yyyy)

"<mark>fecha</mark>" : its a date format dd/mm/yyyy but its aleays 31/12 only the year changes the year is always equal to p_exp_year 

"<mark>p_exp_year</mark>" : exposition year of that row

"<mark>aux_prem_aa</mark>" : usually the same as PRIANUALAA but the first line of each year is the premium 2 years before so only in the first row of each year is diferent from PRIANUALAA

"<mark>aux_prem_ac</mark>" : same logic as the above so the first row of each year aux_prem_ac = PRIANUALAA then  its equal to PRIANUALAC

"<mark>aux_ren</mark>" : date col format( dd/mm/yyyy) same logic as the above so the first row of each year corresponds to the last year so its the same as FVTO_AC and RENOVAÇAO exept in the first row of each year form(dd/mm/yyyy)


"<mark>aux_gar</mark>" : same function aus aux it tels the garantia in code but its almoast redudant since many policies do not change. it is the last digits of the garantia_label  and the col DESCGARA has the same info but with the names aldought the DESCGARA as nan



"<mark>aux_ren_apol</mark>" : the same thing as the aux cols this one is the same as RENOVAÇAO


"<mark>prem_final</mark>" : ussually is equal to aux_prem_ac but its also equal to PRIANUALAA and PRIANUALAC i think in the first entries of each year its equalto the PRIANUALAC



"<mark>prem_exp</mark>" :  =PRIANUALAC* EXPOSIÇAO or prem_inicial*EXPOSIÇAO



"<mark>Ren2</mark>" : date format ( dd/mm/yyyy) almoast the same as ren_apolice2 dont know the diff it also matches a lot of values with RENOVAÇAO


"<mark>Ren_final</mark>" : i gues its the last renovation date form( dd/mm/yyyy) its the last date of renovation so its equal to FVTO_AC for the last lines

"<mark>ren_inicial</mark>" : its the same info od fefecho_poliza but in diferent format( dd/mm/yyyy)


"<mark>premio2</mark>" :  same info as  aux_prem_ac , prem_final


"<mark>IDADE</mark>" : the age of the pessoa segura 

"<mark>data_inicio2</mark>" :  its in number format i dont know what it is 

"<mark>dt_antiga</mark>" : dont know the meaning 

"<mark>data_pre_existencia</mark>" : sometimes is equal to fefecto_poliza but  in dif format still not always equal it must be the date when the pre existencias are evaluated or diagnosed 


"<mark>n_pessoas_ren2</mark>" :  number od pessoas asseguradas at that renovation date 

"<mark>aux_ren2</mark>" : date (dd/mm/yyyy) in 1421256 rows its equal to RENOVACAO -1 year


"<mark>dif_meses</mark>" : the tenure in months how many months does that policy has

"<mark>tenure</mark>" : n of yers the policy is active


"<mark>ant_pessoa</mark>" : n of years a person as policies in that insurer



"<mark>new_ant_pessoa</mark>" : same as above but with some nuances when  a pessoa segura enter the apólice it appears sometimes as -1


"<mark>codpost_tom</mark>" : postal code of tomador 

"<mark>codpost_tomador</mark>" : same as above but more complete


"<mark>codine_tomador</mark>" : no idea




"<mark>cp4</mark>" : the first 4 numbers of codpost tomador

"<mark>codine_tomador_n</mark>" : the first 4 digits of  codine_tomador



"<mark>cp7 </mark>" : the seven digits of codigo_postal but sll together 


"<mark>Avg_Number_Floors  </mark>" : average number of flors per huse in that concelho ! by the first 4 digits of codpost but its old data 
 
"<mark>Crime_rate_2011  </mark>" : crime rate per year


"<mark>Crime_rate_2012  </mark>" :crime rate per year



"<mark>Crime_rate_2013  </mark>" :crime rate per year



"<mark>Crime_rate_2014  </mark>" :crime rate per year


"<mark>Crime_rate_2015  </mark>" :crime rate per year
 


"<mark>Crime_rate_Avg  </mark>" : average of crime rates


"<mark>zip_code  </mark>" : primeiros digitos de cod_post


"<mark>households  </mark>" : number of families 

"<mark>inhabitants  </mark>" : number of people living in that location
 
"<mark>purch_power_Euro  </mark>" : purchase power for that location 


"<mark>Distrito  </mark>" : name of the district

"<mark>Concelho  </mark>" : concelho 


"<mark>CODINE   </mark>" : first four digits of the  codine_tomador


"<mark>N_Clientes  </mark>" : number of allianz clients in that region


"<mark>N_Prestadores  </mark>" : i dont know if its the number of caregiver belonging to the network or the number of agents that sell allianz insurance


"<mark>N_Domicilios  </mark>" : i have no idea its only 52 diferent values


"<mark>Concentracao  </mark>" : binary variable i have no idea what is the meaning


"<mark>N_pessoas_total  </mark>" : number of people living in that region but i dont know the diference with the inhabitants col from what i have reserched this one is more up to date for the concelho of barreiro  so maybe the other one is old data not suitable  =N_Homens + N_Mulheres


"<mark>N_Homens  </mark>" : number of men in that region 


"<mark>N_Mulheres  </mark>" : number of women in that regiom



"<mark>N_hospitais_total  </mark>" : total number of hospitals in the region 



"<mark>N_hospitais_Publico  </mark>" :  total number of public hospitals

"<mark>N_hospitais_Privado</mark>" : total number of private hospitals 



"<mark>N_hospitais_PP</mark>" : number of public and private " parceria publico privada" in that region 


"<mark>Chapter_6</mark>" : cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 



"<mark>Chapter_17</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 



"<mark>Chapter_13</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 



"<mark>Chapter_7</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 


"<mark>Chapter_10</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 



"<mark>Chapter_16</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 



"<mark>Chapter_9</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 



"<mark>Chapter_12</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 




"<mark>Chapter_8</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 



"<mark>Chapter_3</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 


"<mark>Chapter_2</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 


"<mark>Chapter_4</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 


"<mark>Chapter_14</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 


"<mark>Chapter_5</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 


"<mark>Chapter_1</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 


"<mark>Chapter_15</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 


"<mark>Chapter_11</mark>" :cat variable it contains info about  if the person as some kind of desiese but i dont know which desiese 


"<mark>bin_Pre</mark>" : boolean variable that tell me if the pessoa segura has or does not have a previous desiese



"<mark>Capital_Hosp_Cirurgia_e_Parto</mark>" : amount of capital insured for cirurgy and birth



"<mark>Bin_Hosp_Cirurgia_e_Parto</mark>" : boolena variable that tels me if that pessoa seguras has coverage for cirurgy and  birth  1 if it has 2 if it does not has



"<mark>Capital_Parto</mark>" : amount of capital insured for birth separated from the Capital_Hosp_Cirurgia_e_Parto i.e there are policies that don join cirurgia and parto ex for a man it does not makes scence



"<mark>Capital_Hosp_Cirurgia</mark>" : amount of capital insured for cirurgy



"<mark>Capital_Prot_n_Oculares</mark>" :amount of capital insured for  non ocular prosthesis 


"<mark>Capital_Prot_Oculares</mark>" : amount of capital for ocular prosthesis


"<mark>Bin_DENTAL</mark>" : boolean that indentifies if it has dental coverage or not 0 it does not 1 it has


"<mark>Bin_assist_port</mark>" : boolean that indentifies if it has prosthesis coverage or not



"<mark>Capital_Cob_Medica_Int</mark>" : amount of capital insured for internamentos


"<mark>Bin_Cob_Medica_Int</mark>" : bolean that identifies if it has capital for internamento


"<mark>Capital_Consultas</mark>" : amount of capital for consultas médicas


"<mark>Bin_Consultas</mark>" : bollean that indicates if it has capital for consultas médicas or not



"<mark>Bin_Medicamentos</mark>" : boolean that indicates if there is capital for medicines 



"<mark>Capital_Medicamentos</mark>" :Capital_Medicamentos amount of capital insured for medicines 


"<mark>Capital_Estomat_Consult_e_Trat</mark>" : amount of capital insured for estomatologia( destista) includes appointments and treatments


"<mark>Cap_Oncologia</mark>" : amont of capital insured for oncologia



"<mark>CodigoPostal</mark>" : zip code but in format (xxxxxxx)



"<mark>Freguesia_Final_Pos_RATF</mark>" : fregursia of the tomador 

all the data beloww respcts to freguesia ####

"<mark>POP_densidade_populacao</mark>" : population freguesia wise



"<mark>POP_relacao_masculinidade</mark>" : demographic indicator that tels the proportion of men , the number of mens per 100 women (= 100 means same n men and women
>100 means more men than women
< 100 means more women than men )



"<mark>POP_indice_envelhecimento</mark>" : demographic indicato tells  the amount of olds ( +65) per 100 yougs (0-14)


"<mark>POP_indice_dependencia_idosos</mark>" : number of olds by 100 working age individuals



"<mark>POP_durac_media_mov_pendular</mark>" : the amount of time one spends on average going from house to work and from work to house



"<mark>POP_taxa_desemprego</mark>" : unenmplloyment rate




"<mark>POP_ES_completo</mark>" : percentage of the population with colage diploma




"<mark>POP_residente_idade_media</mark>" : average age  




"<mark>POP_ADP_dim_media</mark>" : dimençao média do agregado familiar




"<mark>POP_AI_dim_media</mark>" : dont know


"<mark>POP_NclF_monoparentais</mark>" : number of monoparenting families


"<mark>POP_NclF_com_filhos</mark>" : number of couples with kids

"<mark>POP_meio_transp_total_abs</mark>" : total number of trasnportation means


"<mark>POP_PMT_index_trafego_auto</mark>" : trafic index




"<mark>POP_meio_transp_index_saudavel</mark>" : index that measures how healthy are the transportation eans

"<mark>POP_index_trafg_dens_auto</mark>" : index that measures the amount of cars per area


"<mark>POP_index_trafg_dens_temp_auto</mark>" : no idea



"<mark>POP_index_trafg_dens_saudavel</mark>" :



"<mark>POP_ix_trafg_dens_temp_saudavel</mark>" :



"<mark>POP_index_trafg_dens_motjov</mark>" :



"<mark>POP_index_trafg_dens_temp_motjov</mark>" :





"<mark>POP_idade01_0_14anos</mark>" :

"<mark>POP_idade02_15_19anos</mark>" :


"<mark>POP_idade03_20_24anos</mark>" :


"<mark>POP_idade04_25_29anos</mark>" :


"<mark>POP_idade05_30_34anos</mark>" :


"<mark>POP_idade06_35_39anos</mark>" :


"<mark>POP_idade07_40_44anos</mark>" :



"<mark>POP_idade08_45_49anos</mark>" :




"<mark>POP_idade09_50_54anos</mark>" :



"<mark>POP_idade10_55_59anos</mark>" :



"<mark>POP_idade11_60_64anos</mark>" :



"<mark>POP_idade12_65_69anos</mark>" :


<mark>POP_idade13_70_74anos</mark>" :


<mark>POP_idade14_75_mais_anos</mark>" :


<mark>POP_idade15_residente_total_abs</mark>" :total of residents per freguesia its the summ of all the previous cols 

######
<mark>POP_ativa_ecivil01_solteiro</mark>" : percentage of singles in the active population (18-64)



<mark>POP_ativa_ecivil02_casado</mark>" : percentage of maried in the active population



<mark>POP_ativa_ecivil03_viuvo</mark>" : percentage of widowers in the active population




<mark>POP_ativa_ecivil04_divorciado</mark>" : perceentage of divorced people in the active population 

######

<mark>POP_NclF_nfilhos00_0_filhosEnt</mark>" : percentage of people with 0 sons 




<mark>POP_NclF_nfilhos01_1_filhosEnt</mark>" : percentage of people with 1 son




<mark>POP_NclF_nfilhos02_2_filhosEnt</mark>" : percentage of people with 2 sons




<mark>POP_NclF_nfilhos03_3_filhosEnt</mark>" : percentage of people with 3 sons





<mark>POP_NclF_nfilhos04_4_m_filhosEnt</mark>" : percentage of people with 4 sons





<mark>POP_NclF_nfilhos05_tot_absoluto</mark>" : percentage of people with 5 sons

######



<mark>POL_ESAE01_educacao</mark>" : percentage of people working on educação




<mark>POL_ESAE02_artes_humanidades</mark>" :percentage of people working on artes and humanities



<mark>POL_ESAE03_cienc_soc_jornal_info</mark>" :percentage of people working on social sciences, information and jornalism



<mark>POL_ESAE04_cienc_empr_admn_dir</mark>" : percentage of people working on entrepernurial sciences and bussines admin




<mark>POL_ESAE05_cienc_natur_mat_esta</mark>" :percentage of people working on hard sciences math statistics etc.




<mark>POL_ESAE06_tecnolog_info_com</mark>" :percentage of people working on technologies of information




<mark>POL_ESAE07_engen_industri_const</mark>" :percentage of people working on engenierring, industry and construction




<mark>POL_ESAE08_agric_sivic_pesca_vet</mark>" :percentage of people working on agriculture , fishing veterinaries sivilculture etc..



<mark>POL_ESAE09_saude_protecao_social</mark>" :percentage of people working on health and social protection


<mark>POL_ESAE10_servicos</mark>" :percentage of people working on services not mentioned above

 ######## as seguintes estão em percentagem da pop total


<mark>POP_GSE01_emprs_prof_intelect_CT</mark>" :


<mark>POP_GSE02_quadros_intelectuais_C</mark>" :


<mark>POP_GSE03_forcas_armadas</mark>" :


<mark>POP_GSE04_trab_n_qualif_SP</mark>" :



<mark>POP_GSE05_operar_n_qualif</mark>" :



<mark>POP_GSE06_trab_adm_comerc_SNQ</mark>" :



<mark>POP_GSE07_assalariados_SP</mark>" :



<mark>POP_GSE08_operar_qualificados_SQ</mark>" :



<mark>POP_GSE09_emp_admin_comerc_serv</mark>" :



<mark>POP_GSE10_quadros_admin_interm</mark>" :



<mark>POP_GSE11_quadros_tecn_interm</mark>" :




<mark>POP_GSE12_diretor_quadros_DEMGE</mark>" :




<mark>POP_GSE13_emprs_ind_com_serv</mark>" :




<mark>POP_GSE14_trab_independent_SP</mark>" :






<mark>POP_GSE15_prestad_serv_CI</mark>" :




<mark>POP_GSE16_trab_industr_AI</mark>" :





<mark>POP_GSE17_prof_tecn_interm_I</mark>" :





<mark>POP_GSE18_prof_intelc_cientf</mark>" :







<mark>POP_GSE19_pequenos_patroes_SP</mark>" :







<mark>POP_GSE20_pequenos_patroes_CS</mark>" :




<mark>POP_GSE21_pequenos_patroes_I</mark>" :



<mark>POP_GSE22_pequenos_patroes_PTI</mark>" :




<mark>POP_GSE23_pequenos_patroes_PIC</mark>" :


<mark>POP_GSE24_emprs_SP</mark>" :



<mark>POP_GSE25_outras_pess_atv_NE</mark>" :




<mark>POP_GSE26_pessoas_inativas</mark>" :

# quebra aqui os indicadores socio demográficos




<mark>freg_ratf_github</mark>" : name of the freguesia it has 74740 missing and 295 distinct values


<mark>key_merge_comp</mark>" : no idea it must be a key created by the sistem



<mark>REFSTRO</mark>" : reference number of the sinester when a sinister has more than 1 payment both payments have a line and this number is equal in both lines



<mark>refstro2</mark>" : its the same as REFSTRO but more detailed ex in  the case above when there is more than one payment for the same sinister this col will be dif this number is  REFSTRO-xxxxxxx



<mark>FECHSTRO</mark>" : date of the sinister



<mark>SINISTRALIDADE</mark>" : cash flow generated by the sinister

### all this cols are the components of sinistralidasde by coverage

<mark>cst_Ambul</mark>" : from sinistralidade how much was from ambulatorio



<mark>cst_Cesariana</mark>" :from sinistralidade how much was from cesariana




<mark>cst_Consulta_Domicilio</mark>" :from sinistralidade how much was from consulta domicilio




<mark>cst_Consultas</mark>" :from sinistralidade how much was from consultas




<mark>cst_Consultas_Urg</mark>" :from sinistralidade how much was from consultas de urgencia

## all the EAD bellow i think it means exames analises e diagnóstico 


<mark>cst_EAD_ECO</mark>" : from sinistralidade how much was from ecografia




<mark>cst_EAD_TAC</mark>" :from sinistralidade how much was from TAC





<mark>cst_EADs_Analises</mark>" : from sinistralidade how much was from análises





<mark>cst_EADs_AnatomiaPat</mark>" : from sinistralidade how much was from 




<mark>cst_EADs_Endo</mark>" : from sinistralidade how much was from  endocrinologia





<mark>cst_EADs_MedNuclear</mark>" : from sinistralidade how much was from  medicina nuclear




<mark>cst_EADs_Outros</mark>" : from sinistralidade how much was from outros exames e analises




<mark>cst_EADs_RX</mark>" :from sinistralidade how much was from raio x




<mark>cst_ERRO</mark>" : i have no idea because there is only 2 distinct vals 0 and 130.54 and onlyone entry for 130.54


## the cst might mean cost

<mark>cst_Estomat</mark>" : cost from estomatologia the thing is the next cols summed are not the same as this one  so i do not know what it refers to 




<mark>cst_Estomat_Consult_e_Trat</mark>" : cost of consultas and tratamentos from estomatologia excluding proteses



<mark>cst_Fisio</mark>" : cost of fisioterapia from this sinister


<mark>cst_Hosp_Cirurgia</mark>" : cost of cirurgia hosptalar



<mark>cst_Hosp_Cirurgia_e_Parto</mark>" : cost of cirurgia hospitalar e parto only fot those who have the coverage cirurgia e parto the ones that dont have  are accounted  in cst_Hosp_Cirurgia



<mark>cst_Lesoes_Benig_Pele</mark>" : cost de tratamento lesoes benignas pele 



<mark>cst_Lesoes_Malig_Pele</mark>" : cost of tratamento lesoes malignas na pele




<mark>cst_Onc_Pacote_Total</mark>" : cost of oncologia treatments




<mark>cst_Medicamentos</mark>" : cost of medicamentos



<mark>cst_NOT_HOSP</mark>" : cost of not hospitalar treatments  the thing is i dont know what on hospitalar means


<mark>cst_Parto_Norm_e_IIG</mark>" : cost of parto normal, i dont know the other i suspet second pregnancy




<mark>cst_Prot_Aros</mark>" : custo de aros de óculos





<mark>cst_Prot_Lentes</mark>" : custo de lentes para óculos





<mark>cst_Prot_Lentes_contacto</mark>" : custo de lentes de contacto






<mark>cst_Prot_Oculares</mark>" : custo de proteses oculares, não sei a que se refere porque não é uma soma das acima mencionadas





<mark>cst_Prot_n_Oculares</mark>" : custo de protese não oculares






<mark>cst_Quimio_Radio</mark>" : custo de tratamentos quimio e radio 






<mark>cst_Subs_Desloc</mark>" : custo subsidio de deslocamento






<mark>cst_Subs_Hosp</mark>" : custo subsidio de hospitalizção 


<mark>cst_Tratamentos</mark>" : custo teatamentos


# todas as cont são variavéis binárias tem info sobre se o sinistro foi a cada tipo de occorencia 0-negativo 1- positvo 
por vezes pode haver mais do que uma classe em para parto e cesariana se for 0 não foi  a eesa cobertura se foi 1 foi a parto se for 2 foi  a cesariana

<mark>cont_Ambul</mark>" : indica se houve custo em ambulatorio 


<mark>cont_Cesariana</mark>" :indica se houve custo em cesariana


<mark>cont_Consulta_Domicilio</mark>" :indica se houve custo em Consulta_Domicilio



<mark>cont_Consultas</mark>" :indica se houve custo em Consultas



<mark>cont_Consultas_Urg</mark>" :indica se houve custo em Consultas_Urg



<mark>cont_EAD_ECO</mark>" :indica se houve custo em EAD_ECO 



<mark>cont_EAD_RM</mark>" :indica se houve custo em EAD_RM



<mark>cont_EAD_TAC</mark>" :indica se houve custo em EAD_TAC 




<mark>cont_EADs_Analises</mark>" :indica se houve custo em Analises



<mark>cont_EADs_AnatomiaPat</mark>" :indica se houve custo em anatomia e patologias




<mark>cst_Subcont_EADs_Endos_Hosp</mark>" :indica se houve custo em endoscopia hospitalar




<mark>cont_EADs_MedNuclear</mark>" :indica se houve custo em medicina nuclear





<mark>cont_EADs_Outros</mark>" :indica se houve custo em exames analises e diagnóstcos não listados






<mark>cont_EADs_RX</mark>" :indica se houve custo em raio x




<mark>cont_ERRO</mark>" : não sei


<mark>cont_Estomat</mark>" :indica se houve custo em estomatologia


<mark>cont_Estomat_Consult_e_Trat</mark>" :indica se houve custo em consultas e tratamentos de estomatologia



<mark>cont_Estomat_Prot</mark>" :indica se houve custo em proteses dentárias



<mark>cont_Fisio</mark>" :indica se houve custo em fisioterapia esta tem 42 classes cada classe deve ser para um tipo de fisio terapia diferente (ou tambem pode ser o numero de cessoes de fisio terapia )


<mark>cont_Hosp_Cirurgia</mark>" :indica se houve custo em cirurgia hospitalar



<mark>cont_Hosp_Cirurgia_e_Parto</mark>" :indica se houve custo em cirurgia e parto mais uma vez para aqueles que tem cobertura dos dois




<mark>cont_Lesoes_Benig_Pele</mark>" :indica se houve custo em lesoes benignas na pele



<mark>cont_Lesoes_Malig_Pele</mark>" :indica se houve custo em lesoes malignas na pele


<mark>cont_Medicamentos</mark>" :indica se houve custo em medicamentos



<mark>cont_NOT_HOSP</mark>" :indica se houve custo em coberturas não hospitalares problema eu não sei a que se refere o não hospitalar



<mark>cont_Parto_Norm_e_IIG</mark>" :indica se houve custo em parto normal ou (cesariana nao tenho  a certeza)



<mark>cont_Prot_Aros</mark>" :indica se houve custo em aros para óculos 


<mark>cont_Prot_Lentes</mark>" :indica se houve custo em lentes para óculos

<mark>cont_Prot_Lentes_contacto</mark>" :indica se houve custo em lentes de cntacto


<mark>cont_Prot_Oculares</mark>" :indica se houve custo em proteses oculares





<mark>cont_Prot_n_Oculares</mark>" :indica se houve custo em proteses não oculares


<mark>cont_Quimio_Radio</mark>" :indica se houve custo em tratamentos de quimio ou radio terapia


<mark>cont_Subs_Desloc</mark>" :indica se houve custo em subsidios de deslocação


<mark>cont_Subs_Hosp</mark>" :indica se houve custo em subsidios de hospitalização



<mark>cont_Tratamentos</mark>" :indica se houve custo em tratamentos


<mark>n_claims_dia</mark>" : indica o numero de pagamentos por um sinistro ex se um sinistro tiver x rows i.e o mesmo sinistro gera x pagamentos o n_clais dia vai a x



<mark>nclaim_total</mark>" : esta coluna é a soma de todas as colunas de cont por row ou seja é o numero de coberturas ativas para aquele pagamento problema para os cont que tem mais de 2 classes

<mark>dt_start</mark>" :data de inicio 

<mark>dt_end</mark>" : data de fim 


<mark>n_claims_dia2</mark>" : mesma info que n_claims_dia mas em formato diferente , quando n_claims_dia is nan n_claims_dia2 is 1 otherwhise equal to n_claims_dia2 mais ainda esté int e a n_claims_dia é float




<mark>EXPOSICAO_v2</mark>" : n dias entre (dt_end e dt_start)/365

<mark>exp_DENTAL</mark>" : se a cobertura estiver incluida no pacote = EXPOSICAO_v2 else = 0

<mark>exp_assist_port</mark>" :se a cobertura estiver incluida no pacote = EXPOSICAO_v2 else = 0


<mark>exp_Cob_Medica_Int</mark>" :se a cobertura estiver incluida no pacote = EXPOSICAO_v2 else = 0


<mark>exp_Consultas</mark>" :se a cobertura estiver incluida no pacote = EXPOSICAO_v2 else = 0


<mark>exp_Medicamentos</mark>" :se a cobertura estiver incluida no pacote = EXPOSICAO_v2 else = 0


<mark>Tp_cliente</mark>" : str indica o tipo de cliente se é individual ,emprea( uma empreza a contratar o seguro de saude individual) ou missing
 


<mark>final_premium</mark>" : = pre_final* EXPOSICAO_v2



<mark>new_concentracao</mark>" :não sei




<mark>new_estatura</mark>" : mesmo que estatura mas em metros




<mark>IMC</mark>" :indice de massa corporal





<mark>excess_Hosp_Cirurgia</mark>" : montante gasto acima do limite de cobertura  para hospitalização e cirurgia





<mark>cst_Hosp_Cirurgia_v2</mark>" : não sei deve ser algum montante que a seguradora deve ter ido buscar a reseguro ou assim





<mark>cont_LL_Hosp</mark>" : variável binaria não se o significado de ll




<mark>cst_Estomat_Prot_total</mark>" :custo de proteses totais estomatologia




<mark>cst_Estomat_Prot_partial</mark>" :custo de proteses parciais estomatologia




<mark>cont_total_est_prot</mark>" :indica se houve gasto em protese total em estomatologia





<mark>cont_partial_est_prot</mark>" : indica se houve gasto em protese parcial em estomatologia





<mark>Nr_Line</mark>" : surrogatory key  uniquelly identifies each line




<mark>garantia_label</mark>" : codigo primeiros 3 digit ( ramo) ultimos digitos (produto) xxx-xx
 
#### the next cols are almoast the same as the ones without the 2 sufix all the mean diferences are 0 the only fif i spoted is that one is int the other is float


<mark>N_hospitais_PP2</mark>" : numero de hospitais publico privado 


<mark>N_hospitais_Privado2</mark>" : numero de hospitais privados 


<mark>N_hospitais_Publico2</mark>" : numero de hospitais publicos 


<mark>N_hospitais_total2</mark>" : total de hospitais


<mark>N_Homens2</mark>" :  numero de homens


<mark>N_Mulheres2</mark>" : numero de molheres



<mark>N_pessoas_total2</mark>" :  total de pessoas
##############################################
maybe the above cols where imputed to calculate the proportion of men and women

<mark>Prop_homens</mark>" : proporção de homens 



<mark>Prop_mulheres</mark>" : proporção de homens 
#########################################

the next  cols have the same n of missing values so they are in some way corretated moreover i think its some kind of demografic info


<mark>C_FVI</mark>" :  float it only has values ([ 1.,  3.,  5.,  2.,  8.,  6.,  4., nan,  7.]) so it must represent some type of category



<mark>Exposure</mark>" : type float it only has values 1,2,3,4 so it must represent some type of category 


<mark>Physical_Susceptibility </mark>" : cat var stored as float its like a physical_hazard bin 


<mark>Precipitation</mark>" :  cat var stored as float it mesures how much does it rains


<mark>Social_susceptibility</mark>" : cat var stored as float it measures the social hazard i.e if people are to old and por 


<mark>POLIZASNP</mark>" : indicates if its a new bussines or not


<mark>key</mark>" : key that consist on POLIZA/aplica/NASEGURADO


<mark>marca_inicio_new</mark>" :  bollean var stored as int indicates if that row corresponds to the first row of the year 0 if not 1 if yes


<mark>marca_fim_new</mark>" :bollean var stored as int indicates if that row corresponds to the last row of the year 0 if not 1 if yes


<mark>new_nif_tomador</mark>" : its the numero de identificação fiscal of the tomador 


<mark>score_final</mark>" : the client score calculated based on the client value for the insurer its calculated based on the tomador nif


<mark>ICART_MES</mark>" : its the month of the renovation date 


<mark>ICART_DIA </mark>" : its the day of the renovation date 



<mark>is_lapse </mark>" : boolean variable indicates whetrer the policy was canceled or not renovated or its still at risk 1- if it was midterm or lapse, 0 if its still at risk



<mark>APLICA </mark>" : always 0 



<mark>midterm</mark>" : bollean variable indicates if the policy was canceled midterm or not 1 it was a midterm cancelation 0 if it was not a midterm  










"<mark>dates</mark>" : those are the cols i need to change to date format
FALTAASEG  FECHA_PREEXISTENCIA FEFECTO   FVTO_AC  fefecto_poliza  RENOVACAO   fbajaaseg  INICIO  FIM   min_faltaaseg  min_fecha   max_fbajaaseg   FECHA_novo   fecha  aux_ren    aux_ren_apol data_inicio2   dt_antiga data_pre_existencia  dt_start dt_end   aux_ren2 FNACI8 Ren2 ren_apolice2 Ren_final ren_inicial  FECHSTRO



 "<mark>columns with premium info</mark>" : PRIANUALAA PRIANUALAC   prem_inicial  aux_prem_aa aux_prem_ac prem_final   premio2

# dealing with the nan values 

In [7]:
nan_percent = (lapses.isna().mean() * 100).reset_index()
nan_percent.columns = ['column', 'percent_nan']
nan_percent['num_nan'] = lapses.isna().sum().values

In [8]:
# Convert columns to datetime
lapses['RENOVACAO'] = pd.to_datetime(lapses['RENOVACAO'], format='%d/%m/%Y').dt.date
lapses['aux_ren2'] = pd.to_datetime(lapses['aux_ren2'], format='%d/%m/%Y').dt.date

bar_vars= ["FALTAASEG","FECHA_PREEXISTENCIA","FEFECTO","INICIO","FIM","min_faltaaseg","min_fecha","FECHA_novo","fecha","aux_ren","aux_ren_apol","data_pre_existencia","FNACI8","Ren2","ren_apolice2","Ren_final","ren_inicial","FECHSTRO"]
sep_vars =["FVTO_AC","fefecto_poliza","fbajaaseg","dt_start","dt_end"] 

for col in bar_vars:
    lapses[col] = pd.to_datetime(lapses[col], format='%d/%m/%Y').dt.date

for col in sep_vars:
    lapses[col]=pd.to_datetime(lapses[col],format = "%Y-%m-%d").dt.date


In [9]:
# eleminate cols due to hig occurance of nan
cols_to_drop = ["min_faltaaseg","min_fecha","max_fbajaaseg", "FECHA_novo"]

lapses=lapses.drop(columns=cols_to_drop)

In [10]:
##changing the IMC peso and estatura
# Ensure columns are float for imputation
lapses['PESO'] = lapses['PESO'].astype(float)
lapses['new_estatura'] = lapses['new_estatura'].astype(float)

# Drop the old 'ESTATURA' column if it exists
if 'ESTATURA' in lapses.columns:
    lapses = lapses.drop(columns=['ESTATURA'])

# Define bins and labels
bins = [0, 1, 3, 5, 6, 10, 15, 20, 30, 60, 100]
labels = ['0-1', '2-3', '4-5', '5-6', '7-10', '11-15', '16-20', '21-30', '31-60', '61-100']

# Assign bins using the correct column name 'IDADE'
lapses['idade_bin'] = pd.cut(lapses['IDADE'], bins=bins, labels=labels, right=True, include_lowest=True)

def get_bin_means(df):
    valid = df[
        (df['PESO'].notna()) & (df['PESO'] > 0) & (df['PESO'] <= 150) &
        (df['new_estatura'].notna()) & (df['new_estatura'] > 0) & (df['new_estatura'] <= 2.2)
    ]
    return valid.groupby('idade_bin').agg({'PESO': 'mean', 'new_estatura': 'mean'})

bin_means = get_bin_means(lapses)

# Impute PESO
for bin_label in labels:
    mask = (
        (lapses['idade_bin'] == bin_label) &
        (
            lapses['PESO'].isna() |
            (lapses['PESO'] == 0) |
            (lapses['PESO'] > 150)
        )
    )
    mean_peso = bin_means.loc[bin_label, 'PESO'] if bin_label in bin_means.index else np.nan
    lapses.loc[mask, 'PESO'] = mean_peso

# Impute new_estatura
for bin_label in labels:
    mask = (
        (lapses['idade_bin'] == bin_label) &
        (
            lapses['new_estatura'].isna() |
            (lapses['new_estatura'] == 0) |
            (lapses['new_estatura'] > 2.2)
        )
    )
    mean_estatura = bin_means.loc[bin_label, 'new_estatura'] if bin_label in bin_means.index else np.nan
    lapses.loc[mask, 'new_estatura'] = mean_estatura

# Calculate IMC
lapses["IMC"] = lapses["PESO"] / (lapses["new_estatura"] ** 2)

# Fill NaN IMC with the mean IMC
lapses["IMC"] = lapses["IMC"].fillna(lapses["IMC"].mean())

# changing the way i flag lapses

In [11]:
# here im gonna eleminate the column is_lapse and midterm and i will flag them in another way 
cols_to_remove= ["is_lapse","midterm"]
lapses.drop(columns=cols_to_remove,inplace=True)

lapses["canceled"]= (lapses["fbajaaseg"].notna().astype(int))



# For is_lapse
cond_lapse = (
    (lapses.groupby(['POLIZA', 'NASEGURADO'])['canceled'].transform('sum') >= 1) &
    (lapses.groupby(['POLIZA', 'NASEGURADO'])['fbajaaseg'].transform(lambda x: (x == lapses.loc[x.index, 'FVTO_AC']).any()))
)
lapses['is_lapse'] = cond_lapse.astype(int)

# For is_midterm
cond_midterm = (
    (lapses.groupby(['POLIZA', 'NASEGURADO'])['canceled'].transform('sum') >= 1) &
    (~lapses.groupby(['POLIZA', 'NASEGURADO'])['fbajaaseg'].transform(lambda x: (x == lapses.loc[x.index, 'FVTO_AC']).any()))
)
lapses['is_midterm'] = cond_midterm.astype(int)

# now lets sort the renovation perioud

In [12]:
renov_period = [pd.to_datetime("2025-04-01").date(), pd.to_datetime("2025-04-30").date()]
print(renov_period)

[datetime.date(2025, 4, 1), datetime.date(2025, 4, 30)]


In [13]:
lapses_in_renov_period = lapses[(lapses["FVTO_AC"] >= renov_period[0]) & (lapses["FVTO_AC"] <= renov_period[1])]

## see the number of this apolice with the product health team 
205796892

In [14]:
lapses_in_renov_period[lapses_in_renov_period["POLIZA"] == 205796892][["POLIZA","NASEGURADO","ASEGURADO","Ren2","fefecto_poliza","FALTAASEG","aux_ren_apol","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","is_midterm"]]

,POLIZA,NASEGURADO,ASEGURADO,Ren2,fefecto_poliza,FALTAASEG,aux_ren_apol,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,is_midterm


In [15]:
mask_lp_n_p = (lapses_in_renov_period["is_lapse"] == 1) & (lapses_in_renov_period["PRIANUALAC"].isna())
filtered = lapses_in_renov_period.loc[mask_lp_n_p]
unique_combinations = filtered.drop_duplicates(subset=["POLIZA", "NASEGURADO"])
count = unique_combinations.shape[0]
print("Number of unique combinations:", count)

Number of unique combinations: 4


In [16]:
lapses_in_renov_period[mask_lp_n_p].head(60)[["POLIZA","NASEGURADO","ASEGURADO","Ren2","fefecto_poliza","FALTAASEG","aux_ren_apol","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","is_midterm"]]

,POLIZA,NASEGURADO,ASEGURADO,Ren2,fefecto_poliza,FALTAASEG,aux_ren_apol,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,is_midterm
410634,205674445,2,5206162,2024-04-01,2020-04-01,2020-04-01,2024-04-01,2025-04-01,4,25,2025-04-01,2025-01-01,2025-03-31,2025,NaN,NaN,520.98,14,1,0
422612,205674800,2,11374347,2024-04-01,2020-04-01,2020-04-01,2024-04-01,2025-04-01,4,18,2025-04-01,2025-01-01,2025-01-14,2025,NaN,NaN,424.54,12,1,0
422613,205674800,2,11374347,2024-04-01,2020-04-01,2020-04-01,2024-04-01,2025-04-01,4,18,2025-04-01,2025-01-14,2025-01-14,2025,NaN,NaN,424.54,12,1,0
422614,205674800,2,11374347,2024-04-01,2020-04-01,2020-04-01,2024-04-01,2025-04-01,4,18,2025-04-01,2025-01-15,2025-02-10,2025,NaN,NaN,424.54,12,1,0
422615,205674800,2,11374347,2024-04-01,2020-04-01,2020-04-01,2024-04-01,2025-04-01,4,18,2025-04-01,2025-02-10,2025-03-31,2025,NaN,NaN,424.54,12,1,0
437875,205675284,2,3049987,2024-04-01,2020-04-01,2020-04-01,2024-04-01,2025-04-01,4,72,2025-04-01,2025-01-01,2025-03-31,2025,NaN,NaN,899.03,17,1,0
1494474,206834996,2,123353721,2024-04-01,2023-04-01,2023-04-01,2024-04-01,2025-04-01,1,62,2025-04-01,2025-01-01,2025-03-31,2025,NaN,NaN,1960.36,15,1,0


In [17]:
lapses_in_renov_period[lapses_in_renov_period["POLIZA"] == 204832137][["POLIZA","NASEGURADO","ASEGURADO","PARENTESCO","fefecto_poliza","FALTAASEG","aux_ren_apol","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","is_midterm"]]

,POLIZA,NASEGURADO,ASEGURADO,PARENTESCO,fefecto_poliza,FALTAASEG,aux_ren_apol,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,is_midterm


In [18]:
lapses[(lapses["POLIZA"] == 204832137) & (lapses["NASEGURADO"] == 2)][["POLIZA","NASEGURADO","ASEGURADO","PARENTESCO","fefecto_poliza","FALTAASEG","aux_ren_apol","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","p_exp_year"]]

,POLIZA,NASEGURADO,ASEGURADO,PARENTESCO,fefecto_poliza,FALTAASEG,aux_ren_apol,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,p_exp_year
37907,204832137,2,120870451,1,2018-04-26,2018-04-26,2021-05-01,2022-05-01,3,32,2025-05-01,2022-01-01,2022-01-13,2022,397.38,427.69,397.38,3,1,2022
37908,204832137,2,120870451,1,2018-04-26,2018-04-26,2021-05-01,2022-05-01,3,32,2025-05-01,2022-01-13,2022-01-13,2022,397.38,427.69,397.38,3,1,2022
37909,204832137,2,120870451,1,2018-04-26,2018-04-26,2021-05-01,2022-05-01,3,32,2025-05-01,2022-01-13,2022-04-30,2022,397.38,427.69,397.38,3,1,2022
37910,204832137,2,120870451,1,2018-04-26,2018-04-26,2022-05-01,2022-05-01,4,33,2025-05-01,2022-05-01,2022-12-31,2022,397.38,427.69,427.69,4,1,2022
37911,204832137,2,120870451,1,2018-04-26,2018-04-26,2022-05-01,2023-05-01,4,33,2025-05-01,2023-01-01,2023-04-30,2023,427.69,431.04,427.69,4,1,2023
37912,204832137,2,120870451,1,2018-04-26,2018-04-26,2023-05-01,2023-05-01,5,34,2025-05-01,2023-05-01,2023-09-25,2023,427.69,431.04,431.04,5,1,2023
37913,204832137,2,120870451,1,2018-04-26,2018-04-26,2023-05-01,2023-05-01,5,34,2025-05-01,2023-09-26,2023-10-04,2023,427.69,431.04,431.04,5,1,2023
37914,204832137,2,120870451,1,2018-04-26,2018-04-26,2023-05-01,2023-05-01,5,34,2025-05-01,2023-10-05,2023-11-10,2023,427.69,431.04,431.04,5,1,2023
37915,204832137,2,120870451,1,2018-04-26,2018-04-26,2023-05-01,2023-05-01,5,34,2025-05-01,2023-11-10,2023-11-10,2023,427.69,431.04,431.04,5,1,2023
37916,204832137,2,120870451,1,2018-04-26,2018-04-26,2023-05-01,2023-05-01,5,34,2025-05-01,2023-11-10,2023-12-31,2023,427.69,431.04,431.04,5,1,2023


In [19]:
# here the aging is also not done and there is no premio proposed
lapses[(lapses["POLIZA"] == 205796892) & (lapses["NASEGURADO"] == 3)][["POLIZA","NASEGURADO","ASEGURADO","PARENTESCO","fefecto_poliza","FALTAASEG","aux_ren_apol","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","p_exp_year"]]

,POLIZA,NASEGURADO,ASEGURADO,PARENTESCO,fefecto_poliza,FALTAASEG,aux_ren_apol,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,p_exp_year
713835,205796892,3,11019496,3,2020-08-01,2020-08-01,2021-08-01,2022-08-01,1,20,2025-08-01,2022-01-01,2022-01-18,2022,323.72,418.49,323.72,8,1,2022
713836,205796892,3,11019496,3,2020-08-01,2020-08-01,2021-08-01,2022-08-01,1,20,2025-08-01,2022-01-18,2022-07-31,2022,323.72,418.49,323.72,8,1,2022
713837,205796892,3,11019496,3,2020-08-01,2020-08-01,2022-08-01,2022-08-01,2,21,2025-08-01,2022-08-01,2022-12-31,2022,323.72,418.49,418.49,9,1,2022
713838,205796892,3,11019496,3,2020-08-01,2020-08-01,2022-08-01,2023-08-01,2,21,2025-08-01,2023-01-01,2023-07-31,2023,418.49,465.69,418.49,9,1,2023
713839,205796892,3,11019496,3,2020-08-01,2020-08-01,2023-08-01,2023-08-01,3,22,2025-08-01,2023-08-01,2023-12-31,2023,418.49,465.69,465.69,10,1,2023
713840,205796892,3,11019496,3,2020-08-01,2020-08-01,2023-08-01,2024-08-01,3,22,2025-08-01,2024-01-01,2024-01-26,2024,465.69,510.47,465.69,10,1,2024
713841,205796892,3,11019496,3,2020-08-01,2020-08-01,2023-08-01,2024-08-01,3,22,2025-08-01,2024-01-27,2024-07-31,2024,465.69,510.47,465.69,10,1,2024
713842,205796892,3,11019496,3,2020-08-01,2020-08-01,2024-08-01,2024-08-01,4,23,2025-08-01,2024-08-01,2024-12-31,2024,465.69,510.47,510.47,11,1,2024
713843,205796892,3,11761687,3,2020-08-01,2020-08-01,2024-08-01,2025-08-01,4,23,2025-08-01,2025-01-01,2025-04-22,2025,NaN,NaN,510.47,11,1,2025
713844,205796892,3,11761687,3,2020-08-01,2020-08-01,2024-08-01,2025-08-01,4,23,2025-08-01,2025-04-23,2025-04-26,2025,NaN,NaN,510.47,11,1,2025


In [20]:
mask_np = (lapses_in_renov_period["PRIANUALAC"].isna())
filtered = lapses_in_renov_period.loc[mask_np]
unique_combinations = filtered.drop_duplicates(subset=["POLIZA", "NASEGURADO"])
count = unique_combinations.shape[0]
print("Number of unique combinations:", count)

Number of unique combinations: 23


In [21]:
mask_mi_np = (lapses_in_renov_period["is_lapse"] == 0) & (lapses_in_renov_period["PRIANUALAC"].isna() &(lapses_in_renov_period["is_midterm"]==1))
filtered = lapses_in_renov_period.loc[mask_mi_np]
unique_combinations = filtered.drop_duplicates(subset=["POLIZA", "NASEGURADO"])
count = unique_combinations.shape[0]
print("Number of unique combinations:", count)

Number of unique combinations: 19


### i have a total of 214 cases without PRIANUALAC and PRIANUALAA from those 53 are lapses the cases where premiums where not even proposed, the other cases are midterm cancelaions  so all the cases where there is no premio proposed are either for lapses or midterm cancelations. im left with 2 choises either i drop all the cases where there is no proposed premium or i can also use some imputation technique for the cases where is indeed a lapse ( say i imput a value based on the last known premium and aply  a rate thats the average of the premium increase rates of the cases that are lapse )

In [22]:
#for now lets just eleminate the cases where we do not have premium and latter i will think if i should or should not substitute them 
lapses_in_renov_period =lapses_in_renov_period.dropna(subset=["PRIANUALAC", "PRIANUALAA"])

In [23]:
#i also have cases where the Prianuala is 0  and i do not even have info about it in the lapses df so i will remove this cases because for those i cant compute or use tre premium increase 
#for now im gona drop them latter i might think of a way to substitute them 
mask = lapses_in_renov_period["PRIANUALAA"] == 0
lapses_in_renov_period = lapses_in_renov_period[~mask]

In [24]:
# Group by POLIZA and ASEGURADO, and count unique NASEGURADO values
grouped = lapses_in_renov_period.groupby(['POLIZA', 'ASEGURADO'])['NASEGURADO'].nunique().reset_index()

# Filter groups where there is more than one unique NASEGURADO
cases = grouped[grouped['NASEGURADO'] > 1]

print(cases)

Empty DataFrame
Columns: [POLIZA, ASEGURADO, NASEGURADO]
Index: []


In [25]:
lapses_in_renov_period[lapses_in_renov_period["POLIZA"] == 207229302      ][["POLIZA","NASEGURADO","ASEGURADO","PARENTESCO","fefecto_poliza","FALTAASEG","Ren2","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","is_midterm"]]
# in this case the conjuge has a midterm and do not reenter the policy but the son also has a midterm in ht esame date but reenters the policy in 2025-10-20	 the thing is the FVTO_AC of the new entry is 2025-05-01  before the reentry so using that criteria i will have a renewal before the entry wich does not make sense thus its better do also filter the db by removing the cases here 
#FVTO_AC <= FALTASEG and it should be okey

,POLIZA,NASEGURADO,ASEGURADO,PARENTESCO,fefecto_poliza,FALTAASEG,Ren2,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,is_midterm


# removing the cases here 
# FVTO_AC <= FALTASEG 

In [26]:
# removing the cases here 
#FVTO_AC <= FALTASEG 
mask = lapses_in_renov_period["FVTO_AC"] <= lapses_in_renov_period["FALTAASEG"]
lapses_in_renov_period = lapses_in_renov_period[~mask]

In [27]:
renov_period

[datetime.date(2025, 4, 1), datetime.date(2025, 4, 30)]

In [28]:
lapses[lapses["POLIZA"] == 204834801    ][["POLIZA","NASEGURADO","ASEGURADO","Ren2","fefecto_poliza","FALTAASEG","aux_ren_apol","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","is_midterm","DESCGARA"]]

,POLIZA,NASEGURADO,ASEGURADO,Ren2,fefecto_poliza,FALTAASEG,aux_ren_apol,FVTO_AC,tenure,IDADE,...,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,is_midterm,DESCGARA
39359,204834801,1,5422445,2021-05-01,2018-04-27,2018-04-27,2021-05-01,2022-05-01,3,62,...,2022-01-01,2022-04-30,2022,354.96,382.03,354.96,5,1,0,HOSPITALIZACAO BASE
39360,204834801,1,5422445,2022-05-01,2018-04-27,2018-04-27,2022-05-01,2022-05-01,4,63,...,2022-05-01,2022-12-31,2022,354.96,382.03,382.03,6,1,0,HOSPITALIZACAO BASE
39361,204834801,1,5422445,2022-05-01,2018-04-27,2018-04-27,2022-05-01,2023-05-01,4,63,...,2023-01-01,2023-04-30,2023,382.03,359.64,382.03,6,1,0,HOSPITALIZACAO BASE
39362,204834801,1,5422445,2023-05-01,2018-04-27,2018-04-27,2023-05-01,2023-05-01,5,64,...,2023-05-01,2023-12-31,2023,382.03,359.64,359.64,7,1,0,HOSPITALIZACAO BASE
39363,204834801,1,5422445,2023-05-01,2018-04-27,2018-04-27,2023-05-01,2024-05-01,5,64,...,2024-01-01,2024-04-30,2024,359.64,409.98,359.64,7,1,0,HOSPITALIZACAO BASE
39364,204834801,1,5422445,2024-05-01,2018-04-27,2018-04-27,2024-05-01,2024-05-01,6,65,...,2024-05-01,2024-12-31,2024,359.64,409.98,409.98,8,1,0,HOSPITALIZACAO BASE
39365,204834801,1,5422445,2024-05-01,2018-04-27,2018-04-27,2024-05-01,2025-05-01,6,65,...,2025-01-01,2025-04-30,2025,NaN,NaN,409.98,8,1,0,NaN
39366,204834801,2,5422445,2025-05-01,2018-04-27,2025-05-01,2025-05-01,2025-05-01,0,66,...,2025-05-01,2025-12-31,2025,409.74,404.95,404.95,9,0,0,Standard [NOVO]


In [29]:
# checking how many fiferent pessoas seguras in this perioud of renovations 
lapses_in_renov_period[["POLIZA","ASEGURADO","NASEGURADO"]].drop_duplicates()

,POLIZA,ASEGURADO,NASEGURADO
5039,204755164,11862752,1
5073,204755164,120807249,2
5090,204755164,120830196,3
5122,204755164,121548806,4
6419,204758776,4019200,1
...,...,...,...
1581044,207269988,122836365,2
1581254,207271673,122741315,1
1584826,207303149,120246147,1
1584834,207303149,122731167,2


In [30]:
lapses_in_renov_period[["POLIZA","ASEGURADO"]].drop_duplicates()

,POLIZA,ASEGURADO
5039,204755164,11862752
5073,204755164,120807249
5090,204755164,120830196
5122,204755164,121548806
6419,204758776,4019200
...,...,...
1581044,207269988,122836365
1581254,207271673,122741315
1584826,207303149,120246147
1584834,207303149,122731167


In [31]:
lapses_in_renov_period[["POLIZA","ASEGURADO","NASEGURADO","FVTO_AC"]].drop_duplicates()

,POLIZA,ASEGURADO,NASEGURADO,FVTO_AC
5039,204755164,11862752,1,2025-04-01
5073,204755164,120807249,2,2025-04-01
5090,204755164,120830196,3,2025-04-01
5122,204755164,121548806,4,2025-04-01
6419,204758776,4019200,1,2025-04-01
...,...,...,...,...
1581044,207269988,122836365,2,2025-04-01
1581254,207271673,122741315,1,2025-04-01
1584826,207303149,120246147,1,2025-04-01
1584834,207303149,122731167,2,2025-04-01


In [32]:
#i have another strange case the FVTO_AC has 2 diferent values for the same pessoa segura 
# Find groups with more than one unique FVTO_AC
duplicates = (
    lapses_in_renov_period
    .groupby(['POLIZA', 'ASEGURADO', 'NASEGURADO'])['FVTO_AC']
    .nunique()
    .reset_index()
)
cases_with_multiple_fvto_ac = duplicates[duplicates['FVTO_AC'] > 1]
print(cases_with_multiple_fvto_ac)

Empty DataFrame
Columns: [POLIZA, ASEGURADO, NASEGURADO, FVTO_AC]
Index: []


In [33]:
lapses_in_renov_period[lapses_in_renov_period["POLIZA"] == 207333704        ][["POLIZA","NASEGURADO","ASEGURADO","PARENTESCO","fefecto_poliza","FALTAASEG","Ren2","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","is_midterm"]]

,POLIZA,NASEGURADO,ASEGURADO,PARENTESCO,fefecto_poliza,FALTAASEG,Ren2,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,is_midterm


In [34]:
#checking if the filtration by the last date of renovação was correct

lapses_in_renov_period.FVTO_AC.max() # it is correct 

datetime.date(2025, 4, 30)

In [35]:
# Get the last row for each unique combination using drop_duplicates
lapses_pp_renov_period = lapses_in_renov_period.drop_duplicates(
    subset=['POLIZA', 'ASEGURADO', 'NASEGURADO'],
    keep='last'
).reset_index(drop=True)

In [36]:
# total number of lapses 
total_lapses= lapses_pp_renov_period['is_lapse'].sum()
total_policies= lapses_pp_renov_period.shape[0]
lapse_rate = (total_lapses / total_policies) * 100
print(f"Total Lapses: {total_lapses}")
print(f"Total Policies: {total_policies}")  
print(f"Lapse Rate: {lapse_rate:.2f}%")

Total Lapses: 320
Total Policies: 5402
Lapse Rate: 5.92%


In [37]:
#now lets esplore the aging in the lapses cases
mask = lapses_pp_renov_period["is_lapse"] == 1
filtered = lapses_pp_renov_period.loc[mask]
filtered

,POLIZA,NASEGURADO,ASEGURADO,SEXO,PARENTESCO,FALTAASEG,FNACI8,POLIZAS_45,PESO,TRANSF_SEGURO_INT,...,marca_fim_new,new_nif_tomador,score_final,ICART_MES,ICART_DIA,APLICA,idade_bin,canceled,is_lapse,is_midterm
216,204794292,1,120842465,H,0,2018-03-23,1959-02-10,0,67.0,N,...,1,104954140,10.0,4,1,0,61-100,1,1,0
381,204807901,1,120837574,H,0,2018-04-04,1983-04-12,0,68.0,N,...,1,223904503,6.0,4,1,0,31-60,1,1,0
382,204808105,1,3505616,V,0,2018-04-05,1957-09-01,0,73.0,N,...,1,177082410,10.0,4,1,0,61-100,1,1,0
383,204808105,2,3505617,H,1,2018-04-05,1961-12-03,0,53.0,N,...,1,177082410,10.0,4,1,0,61-100,1,1,0
384,204808105,3,120858059,H,3,2018-04-05,1998-02-22,0,55.0,N,...,1,177082410,10.0,4,1,0,21-30,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5320,207218453,1,122235822,V,0,2024-04-10,1985-03-03,0,90.0,N,...,1,318143348,5.0,4,1,0,31-60,1,1,0
5321,207218453,2,122751199,H,1,2024-04-10,1986-12-11,0,65.0,N,...,1,318143348,5.0,4,1,0,31-60,1,1,0
5362,207221429,1,122081912,H,0,2024-04-12,1996-08-24,0,72.0,N,...,1,315336021,5.0,4,1,0,21-30,1,1,0
5391,207242348,1,122792071,V,0,2024-04-30,2023-04-14,0,7.0,N,...,1,254684173,3.0,4,1,0,0-1,1,1,0


In [38]:
# lets take on poliza i the filtered with tenure == 0

lapses[(lapses["POLIZA"] == 204870750 )& (lapses["ASEGURADO"] == 123040967) ][["POLIZA","NASEGURADO","ASEGURADO","fefecto_poliza","FALTAASEG","aux_ren_apol","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse"]]
# since the time passed betwenn the date he entered and the date he left was 10 month the tenure was not updated to 1 year so i need to update (do the aging of this cases)

,POLIZA,NASEGURADO,ASEGURADO,fefecto_poliza,FALTAASEG,aux_ren_apol,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse
54812,204870750,4,123040967,2018-06-15,2024-08-27,2024-07-01,2024-07-01,0,0,2025-07-01,2024-08-27,2024-11-12,2024,0.00,403.45,403.45,0,1
54813,204870750,4,123040967,2018-06-15,2024-08-27,2024-07-01,2024-07-01,0,0,2025-07-01,2024-11-13,2024-11-22,2024,0.00,403.45,403.45,0,1
54814,204870750,4,123040967,2018-06-15,2024-08-27,2024-07-01,2024-07-01,0,0,2025-07-01,2024-11-23,2024-12-31,2024,0.00,403.45,403.45,0,1
54815,204870750,4,123040967,2018-06-15,2024-08-27,2024-07-01,2025-07-01,0,0,2025-07-01,2025-01-01,2025-01-27,2025,403.45,427.66,403.45,0,1
54816,204870750,4,123040967,2018-06-15,2024-08-27,2024-07-01,2025-07-01,0,0,2025-07-01,2025-01-28,2025-02-26,2025,403.45,427.66,403.45,0,1
54817,204870750,4,123040967,2018-06-15,2024-08-27,2024-07-01,2025-07-01,0,0,2025-07-01,2025-02-27,2025-03-07,2025,403.45,427.66,403.45,0,1
54818,204870750,4,123040967,2018-06-15,2024-08-27,2024-07-01,2025-07-01,0,0,2025-07-01,2025-03-08,2025-03-31,2025,403.45,427.66,403.45,0,1
54819,204870750,4,123040967,2018-06-15,2024-08-27,2024-07-01,2025-07-01,0,0,2025-07-01,2025-04-01,2025-06-30,2025,403.45,427.66,403.45,0,1


In [39]:
#lets see another case
lapses[lapses["POLIZA"] == 207401668  ][["POLIZA","NASEGURADO","ASEGURADO","fefecto_poliza","FALTAASEG","aux_ren_apol","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse"]]
# same as above the FALTAASEG is diferent from the day  of FVTO_AC so the tenure is not updated now i need to see if this only happens in the cases where FVTO_AC!= FALTAASEG

,POLIZA,NASEGURADO,ASEGURADO,fefecto_poliza,FALTAASEG,aux_ren_apol,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse
1596554,207401668,1,122336086,2024-09-10,2024-09-10,2024-09-10,2025-09-01,0,54,2025-09-01,2024-09-10,2024-12-31,2024,0.00,348.12,348.12,0,1
1596555,207401668,1,122336086,2024-09-10,2024-09-10,2024-09-10,2025-09-01,0,54,2025-09-01,2025-01-01,2025-08-31,2025,348.12,358.56,348.12,0,1


In [40]:
filtered.FVTO_AC.dtype

dtype('O')

In [41]:
# lets see the cases where i have lapses and the FVTO_AC is equal to FALTAASEG to asses if the aging is done
# Compare day and month
# Ensure both columns are datetime
filtered["FVTO_AC"] = pd.to_datetime(filtered["FVTO_AC"])
filtered["FALTAASEG"] = pd.to_datetime(filtered["FALTAASEG"])

# Compare day and month
mask = (filtered["FVTO_AC"].dt.day == filtered["FALTAASEG"].dt.day) & \
       (filtered["FVTO_AC"].dt.month == filtered["FALTAASEG"].dt.month)

result = filtered[mask][["POLIZA","NASEGURADO","ASEGURADO","fefecto_poliza","FALTAASEG","aux_ren_apol","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse"]]
result

,POLIZA,NASEGURADO,ASEGURADO,fefecto_poliza,FALTAASEG,aux_ren_apol,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse
974,205674488,1,10974018,2020-04-01,2020-04-01,2024-04-01,2025-04-01,4,71,2025-04-01,2025-03-28,2025-03-31,2025,3370.73,3874.51,3874.51,14,1
979,205674498,1,3495982,2020-04-01,2020-04-01,2024-04-01,2025-04-01,4,46,2025-04-01,2025-01-01,2025-03-31,2025,801.63,840.17,801.63,14,1
1047,205674562,1,5213932,2020-04-01,2020-04-01,2024-04-01,2025-04-01,4,40,2025-04-01,2025-01-01,2025-03-31,2025,1326.53,1353.75,1326.53,13,1
1091,205674614,1,3242093,2020-04-01,2020-04-01,2024-04-01,2025-04-01,4,45,2025-04-01,2025-01-01,2025-03-31,2025,831.82,874.10,831.82,13,1
1147,205674674,1,3621313,2020-04-01,2020-04-01,2024-04-01,2025-04-01,4,71,2025-04-01,2025-01-01,2025-03-31,2025,2403.66,2505.95,2403.66,12,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5126,207205857,1,122720675,2024-04-01,2024-04-01,2024-04-01,2025-04-01,0,34,2025-04-01,2025-01-01,2025-03-31,2025,184.17,187.85,184.17,0,1
5127,207205857,2,121960916,2024-04-01,2024-04-01,2024-04-01,2025-04-01,0,6,2025-04-01,2025-01-01,2025-03-31,2025,123.82,126.30,123.82,2,1
5128,207205857,3,121960917,2024-04-01,2024-04-01,2024-04-01,2025-04-01,0,5,2025-04-01,2025-01-01,2025-03-31,2025,120.63,122.58,120.63,2,1
5142,207207693,1,5487752,2024-04-01,2024-04-01,2024-04-01,2025-04-01,0,41,2025-04-01,2025-01-01,2025-03-31,2025,248.64,256.10,248.64,10,1


In [42]:
lapses[lapses["POLIZA"] == 204856802  ][["POLIZA","NASEGURADO","ASEGURADO","fefecto_poliza","FALTAASEG","aux_ren_apol","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","FNACI8"]]
#in fact for the lapse cases no aging is done in the cols (tenure, idade and new_ant_pessoa  thus i need to update that 

,POLIZA,NASEGURADO,ASEGURADO,fefecto_poliza,FALTAASEG,aux_ren_apol,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,FNACI8
50451,204856802,1,120827824,2018-06-01,2018-06-01,2021-06-01,2022-06-01,3,34,2025-06-01,2022-01-01,2022-05-31,2022,131.09,145.34,131.09,3,1,1987-03-15
50452,204856802,1,120827824,2018-06-01,2018-06-01,2022-06-01,2022-06-01,4,35,2025-06-01,2022-06-01,2022-12-31,2022,131.09,145.34,145.34,4,1,1987-03-15
50453,204856802,1,120827824,2018-06-01,2018-06-01,2022-06-01,2023-06-01,4,35,2025-06-01,2023-01-01,2023-05-31,2023,145.34,166.10,145.34,4,1,1987-03-15
50454,204856802,1,120827824,2018-06-01,2018-06-01,2023-06-01,2023-06-01,5,36,2025-06-01,2023-06-01,2023-12-31,2023,145.34,166.10,166.10,5,1,1987-03-15
50455,204856802,1,120827824,2018-06-01,2018-06-01,2023-06-01,2024-06-01,5,36,2025-06-01,2024-01-01,2024-05-31,2024,166.10,177.68,166.10,5,1,1987-03-15
50456,204856802,1,120827824,2018-06-01,2018-06-01,2024-06-01,2024-06-01,6,37,2025-06-01,2024-06-01,2024-12-31,2024,166.10,177.68,177.68,6,1,1987-03-15
50457,204856802,1,120827824,2018-06-01,2018-06-01,2024-06-01,2025-06-01,6,37,2025-06-01,2025-01-01,2025-05-31,2025,177.68,185.63,177.68,6,1,1987-03-15
50458,204856802,2,120870061,2018-06-01,2018-06-01,2021-06-01,2022-06-01,3,28,2025-06-01,2022-01-01,2022-05-31,2022,109.50,121.41,109.50,3,1,1992-12-15
50459,204856802,2,120870061,2018-06-01,2018-06-01,2022-06-01,2022-06-01,4,29,2025-06-01,2022-06-01,2022-12-31,2022,109.50,121.41,121.41,4,1,1992-12-15
50460,204856802,2,120870061,2018-06-01,2018-06-01,2022-06-01,2023-06-01,4,29,2025-06-01,2023-01-01,2023-05-31,2023,121.41,135.80,121.41,4,1,1992-12-15


### lets see if the aging is also not done for the midterms 


In [43]:
mask = lapses_pp_renov_period["is_midterm"] == 1
filtered = lapses_pp_renov_period.loc[mask]
filtered

,POLIZA,NASEGURADO,ASEGURADO,SEXO,PARENTESCO,FALTAASEG,FNACI8,POLIZAS_45,PESO,TRANSF_SEGURO_INT,...,marca_fim_new,new_nif_tomador,score_final,ICART_MES,ICART_DIA,APLICA,idade_bin,canceled,is_lapse,is_midterm
31,204776194,1,120774774,H,0,2018-04-01,1964-02-19,0,69.0,N,...,1,230537588,10.0,4,1,0,61-100,1,0,1
32,204776194,2,120841683,V,1,2018-04-01,1964-10-16,0,85.0,N,...,1,230537588,10.0,4,1,0,31-60,1,0,1
38,204778402,1,120842688,V,0,2018-04-01,1979-02-07,0,73.0,N,...,1,223619647,9.0,4,1,0,31-60,1,0,1
39,204778402,2,120509560,H,3,2018-04-01,2011-01-12,0,30.0,N,...,1,223619647,9.0,4,1,0,11-15,1,0,1
40,204778402,3,121091283,H,3,2019-02-01,2018-09-20,0,3.0,N,...,1,223619647,9.0,4,1,0,5-6,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5361,207221417,1,122225770,H,0,2024-04-12,1991-01-12,0,54.0,N,...,1,317396862,5.0,4,1,0,31-60,1,0,1
5364,207221538,1,121532262,H,0,2024-04-12,2011-08-26,0,45.0,N,...,1,228684048,5.0,4,1,0,11-15,1,0,1
5369,207221615,1,120411384,H,0,2024-04-12,1976-11-22,0,60.0,N,...,1,223838020,4.0,4,1,0,31-60,1,0,1
5371,207221697,1,122707650,V,0,2024-04-12,1960-08-01,0,70.0,N,...,1,317766520,6.0,4,1,0,61-100,1,0,1


In [ ]:
lapses[lapses["POLIZA"]== 207416632][["POLIZA","NASEGURADO","ASEGURADO","fefecto_poliza","FALTAASEG","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_midterm","FNACI8","new_ant_pessoa"]]
#in fact for the lapse cases no aging is done in the cols (tenure, idade and new_ant_pessoa  thus i need to update that col for those casesbecause in internet transacional the first policy was activated  for this pesoa segura in 8-1-2000 so in 2022 it should be 22 years but its 14 in addition it also has a policy starting in 2007 so the 14 would be correct

,POLIZA,NASEGURADO,ASEGURADO,fefecto_poliza,FALTAASEG,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_midterm,FNACI8,new_ant_pessoa
1599170,207416632,1,121186656,2024-09-01,2024-09-01,2025-09-01,0,16,2025-03-01,2024-09-01,2024-12-31,2024,0.0,125.03,125.03,5,1,2008-05-14,0
1599171,207416632,1,121186656,2024-09-01,2024-09-01,2025-09-01,0,16,2025-03-01,2025-01-01,2025-02-28,2025,0.0,125.03,125.03,5,1,2008-05-14,0


In [45]:
#lets see how many cases in the lapses_pp_renov_period have ant_pessoa dif from new_ant_pessoa
mask = (lapses_pp_renov_period['ant_pessoa'] != lapses_pp_renov_period['new_ant_pessoa'])
filtered = lapses_pp_renov_period.loc[mask] 
filtered

,POLIZA,NASEGURADO,ASEGURADO,SEXO,PARENTESCO,FALTAASEG,FNACI8,POLIZAS_45,PESO,TRANSF_SEGURO_INT,...,marca_fim_new,new_nif_tomador,score_final,ICART_MES,ICART_DIA,APLICA,idade_bin,canceled,is_lapse,is_midterm
6,204765363,1,11865386,V,0,2018-03-22,2002-01-25,1,80.0,N,...,1,211302708,9.0,4,1,0,21-30,0,0,0
12,204768623,1,11852689,H,0,2018-04-01,2004-03-08,1,50.0,N,...,1,209982330,8.0,4,1,0,21-30,0,0,0
13,204769026,1,2464674,V,0,2018-04-01,1946-11-04,1,77.0,S,...,1,128936916,6.0,4,1,0,61-100,0,0,0
25,204772654,1,2217621,V,0,2018-03-27,1968-07-24,1,72.0,N,...,1,215210913,10.0,4,1,0,31-60,0,0,0
26,204772654,2,11700823,H,1,2018-04-01,1971-05-07,1,54.0,N,...,1,215210913,10.0,4,1,0,31-60,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5392,207244892,1,122792050,H,0,2024-03-25,1995-05-28,0,75.0,N,...,1,259409944,5.0,4,1,0,21-30,1,0,1
5393,207250650,1,2848689,V,0,2024-03-18,1973-10-15,1,70.0,S,...,1,208441727,7.0,4,1,0,31-60,0,0,0
5396,207269988,1,120846441,H,0,2024-04-01,1990-03-17,1,50.0,N,...,1,244251487,9.0,4,1,0,31-60,0,0,0
5398,207271673,1,122741315,V,0,2024-03-15,2008-10-26,1,68.0,N,...,1,235122920,4.0,4,1,0,16-20,0,0,0


In [46]:
lapses[lapses["POLIZA"]== 207437575][["POLIZA","NASEGURADO","ASEGURADO","fefecto_poliza","FALTAASEG","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","is_midterm","FNACI8","new_ant_pessoa"]]

,POLIZA,NASEGURADO,ASEGURADO,fefecto_poliza,FALTAASEG,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,is_midterm,FNACI8,new_ant_pessoa
1602750,207437575,1,123075393,2024-06-16,2024-06-16,2025-07-01,0,65,NaT,2024-06-16,2024-12-31,2024,0.00,703.05,703.05,0,0,0,1959-06-08,20
1602751,207437575,1,123075393,2024-06-16,2024-06-16,2025-07-01,0,65,NaT,2025-01-01,2025-06-30,2025,703.05,737.49,703.05,0,0,0,1959-06-08,20
1602752,207437575,1,123075393,2024-06-16,2024-06-16,2025-07-01,1,66,NaT,2025-07-01,2025-12-31,2025,703.05,737.49,737.49,1,0,0,1959-06-08,21


In [47]:
lapses[lapses["POLIZA"]== 204769026][["POLIZA","NASEGURADO","ASEGURADO","fefecto_poliza","FALTAASEG","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","is_midterm","FNACI8","new_ant_pessoa"]]

,POLIZA,NASEGURADO,ASEGURADO,fefecto_poliza,FALTAASEG,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,is_midterm,FNACI8,new_ant_pessoa
12323,204769026,1,2464674,2018-04-01,2018-04-01,2022-04-01,3,74,NaT,2022-01-01,2022-03-31,2022,1760.95,1895.23,1760.95,10,0,0,1946-11-04,14
12324,204769026,1,2464674,2018-04-01,2018-04-01,2022-04-01,4,75,NaT,2022-04-01,2022-05-17,2022,1760.95,1895.23,1895.23,11,0,0,1946-11-04,15
12325,204769026,1,2464674,2018-04-01,2018-04-01,2022-04-01,4,75,NaT,2022-05-18,2022-12-31,2022,1760.95,1895.23,1895.23,11,0,0,1946-11-04,15
12326,204769026,1,2464674,2018-04-01,2018-04-01,2023-04-01,4,75,NaT,2023-01-01,2023-01-20,2023,1895.23,2056.73,1895.23,11,0,0,1946-11-04,15
12327,204769026,1,2464674,2018-04-01,2018-04-01,2023-04-01,4,75,NaT,2023-01-21,2023-02-02,2023,1895.23,2056.73,1895.23,11,0,0,1946-11-04,15
12328,204769026,1,2464674,2018-04-01,2018-04-01,2023-04-01,4,75,NaT,2023-02-03,2023-02-06,2023,1895.23,2056.73,1895.23,11,0,0,1946-11-04,15
12329,204769026,1,2464674,2018-04-01,2018-04-01,2023-04-01,4,75,NaT,2023-02-06,2023-02-06,2023,1895.23,2056.73,1895.23,11,0,0,1946-11-04,15
12330,204769026,1,2464674,2018-04-01,2018-04-01,2023-04-01,4,75,NaT,2023-02-07,2023-02-24,2023,1895.23,2056.73,1895.23,11,0,0,1946-11-04,15
12331,204769026,1,2464674,2018-04-01,2018-04-01,2023-04-01,4,75,NaT,2023-02-25,2023-03-06,2023,1895.23,2056.73,1895.23,11,0,0,1946-11-04,15
12332,204769026,1,2464674,2018-04-01,2018-04-01,2023-04-01,4,75,NaT,2023-03-06,2023-03-06,2023,1895.23,2056.73,1895.23,11,0,0,1946-11-04,15


In [48]:
#the col new_ant_pessoa is not correct in the cases of lapses and midterm since the aging is not done in those cases so i need to update that col for those casesbecause in internet transacional the first policy was activated  for this pesoa segura in 8-1-2000 so in 2022 it should be 22 years but its 14 in addition it also has a policy starting in 2007 so the 14 would be correct
# # moreover the ant_pessoa strts at 10 fir exp_year 2022 that would be correct if the first policy considered was in fact the one from 2011 so wich one is correct?? i dont know lets count the cases where the two are difeent and see from those how many are lapses
#  

mask = (lapses_pp_renov_period['ant_pessoa'] != lapses_pp_renov_period['new_ant_pessoa'])
filtered = lapses_pp_renov_period.loc[mask] 
cases= filtered.shape[0]
cases_w_lapses = filtered[filtered["is_lapse"] == 1].shape[0]
print(f"Total cases with different ant_pessoa and new_ant_pessoa: {cases}")
print(f"Total cases with different ant_pessoa and new_ant_pessoa that are lapses: {cases_w_lapses}")

Total cases with different ant_pessoa and new_ant_pessoa: 1365
Total cases with different ant_pessoa and new_ant_pessoa that are lapses: 51


In [49]:
filtered[filtered["is_lapse"] == 1]

,POLIZA,NASEGURADO,ASEGURADO,SEXO,PARENTESCO,FALTAASEG,FNACI8,POLIZAS_45,PESO,TRANSF_SEGURO_INT,...,marca_fim_new,new_nif_tomador,score_final,ICART_MES,ICART_DIA,APLICA,idade_bin,canceled,is_lapse,is_midterm
1147,205674674,1,3621313,H,0,2020-04-01,1952-09-10,0,60.000000,S,...,0,107067501,10.0,4,1,0,61-100,1,1,0
1277,205674809,1,175438,H,0,2020-04-01,1950-09-23,0,67.000000,S,...,1,122346041,7.0,4,1,0,61-100,1,1,0
1705,205675227,1,11933096,V,0,2020-04-01,1975-08-05,0,82.000000,S,...,1,200004786,9.0,4,1,0,31-60,1,1,0
1852,205675384,1,3802110,H,0,2020-04-01,1980-11-27,0,54.000000,S,...,1,243404255,10.0,4,1,0,31-60,1,1,0
2043,205675600,1,790307,V,0,2020-04-01,1949-06-02,0,65.000000,S,...,1,129471925,10.0,4,1,0,61-100,1,1,0
2132,205675696,1,2426282,H,0,2020-04-01,1968-06-12,0,50.000000,S,...,1,181630990,10.0,4,1,0,31-60,1,1,0
2181,205675757,1,3821830,H,0,2020-04-01,1978-07-15,0,64.000000,S,...,1,203219953,3.0,4,1,0,31-60,1,1,0
2269,205675878,5,3500053,V,1,2021-02-04,1972-04-04,0,73.000000,S,...,1,202628981,10.0,4,4,0,31-60,1,1,0
2361,205675975,1,587455,V,0,2020-04-02,1965-03-31,0,69.756078,S,...,1,170841383,6.0,4,2,0,31-60,1,1,0
2362,205675975,2,793137,H,1,2020-04-02,1965-08-23,0,69.756078,S,...,1,170841383,6.0,4,2,0,31-60,1,1,0


In [50]:
lapses[lapses["POLIZA"]== 207385903][["POLIZA","NASEGURADO","ASEGURADO","fefecto_poliza","FALTAASEG","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","is_midterm","FNACI8","new_ant_pessoa"]]

#in this case the ant_pessoa is correct but the new_ant_pessoa is not correct since the first policy for this pessoa segura was in 2015 so in 2024 it should be 8 yo like the ant pesoa

,POLIZA,NASEGURADO,ASEGURADO,fefecto_poliza,FALTAASEG,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,is_midterm,FNACI8,new_ant_pessoa
1594317,207385903,1,11890937,2024-09-01,2024-09-01,2025-09-01,0,31,2025-09-01,2024-09-01,2024-12-31,2024,0.00,554.82,554.82,8,1,0,1993-06-23,0
1594318,207385903,1,123098957,2024-09-01,2024-09-01,2025-09-01,0,31,2025-09-01,2025-01-01,2025-08-31,2025,554.82,577.01,554.82,8,1,0,1993-06-23,0


In [51]:
# lets see another case 
lapses[lapses["POLIZA"]== 204855634][["POLIZA","NASEGURADO","ASEGURADO","fefecto_poliza","FALTAASEG","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","is_midterm","FNACI8","new_ant_pessoa"]]
# okey now i got it the ant_pessoa measures the time it was client and it was not the tomador , and the ant_pessoa measures the time it has been in the company  as tomador

,POLIZA,NASEGURADO,ASEGURADO,fefecto_poliza,FALTAASEG,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,is_midterm,FNACI8,new_ant_pessoa
49743,204855634,1,11863404,2018-06-01,2018-06-01,2022-06-01,3,24,2025-06-01,2022-01-01,2022-05-31,2022,351.74,389.93,351.74,7,1,0,1997-04-22,3
49744,204855634,1,11863404,2018-06-01,2018-06-01,2022-06-01,4,25,2025-06-01,2022-06-01,2022-06-13,2022,351.74,389.93,389.93,8,1,0,1997-04-22,4
49745,204855634,1,11863404,2018-06-01,2018-06-01,2022-06-01,4,25,2025-06-01,2022-06-14,2022-06-28,2022,351.74,389.93,389.93,8,1,0,1997-04-22,4
49746,204855634,1,11863404,2018-06-01,2018-06-01,2022-06-01,4,25,2025-06-01,2022-06-28,2022-06-28,2022,351.74,389.93,389.93,8,1,0,1997-04-22,4
49747,204855634,1,11863404,2018-06-01,2018-06-01,2022-06-01,4,25,2025-06-01,2022-06-28,2022-06-28,2022,351.74,389.93,389.93,8,1,0,1997-04-22,4
49748,204855634,1,11863404,2018-06-01,2018-06-01,2022-06-01,4,25,2025-06-01,2022-06-29,2022-10-28,2022,351.74,389.93,389.93,8,1,0,1997-04-22,4
49749,204855634,1,11863404,2018-06-01,2018-06-01,2022-06-01,4,25,2025-06-01,2022-10-29,2022-11-02,2022,351.74,389.93,389.93,8,1,0,1997-04-22,4
49750,204855634,1,11863404,2018-06-01,2018-06-01,2022-06-01,4,25,2025-06-01,2022-11-03,2022-12-31,2022,351.74,389.93,389.93,8,1,0,1997-04-22,4
49751,204855634,1,11863404,2018-06-01,2018-06-01,2023-06-01,4,25,2025-06-01,2023-01-01,2023-02-07,2023,389.93,440.28,389.93,8,1,0,1997-04-22,4
49752,204855634,1,11863404,2018-06-01,2018-06-01,2023-06-01,4,25,2025-06-01,2023-02-08,2023-03-12,2023,389.93,440.28,389.93,8,1,0,1997-04-22,4


### i got  it now the diference between the ant_pessoa and the new_ant_pessoa is that the ant pessoa is the time he is the client counting from the moment he was first tomador  and the new_ant pessoa is the years counting from the date we was first a segurado but not a tomador, but one question remains how about the cases where both the new_ant_pessoa and ant_pessoa are equal does that means that they have two policies in the same year  one as tomador and another as pessoa segura ? lets check those cases

In [52]:
mask = (lapses_pp_renov_period['ant_pessoa'] != lapses_pp_renov_period['new_ant_pessoa'])
filtered = lapses_pp_renov_period.loc[~mask] 
filtered[filtered["is_lapse"] == 1]

,POLIZA,NASEGURADO,ASEGURADO,SEXO,PARENTESCO,FALTAASEG,FNACI8,POLIZAS_45,PESO,TRANSF_SEGURO_INT,...,marca_fim_new,new_nif_tomador,score_final,ICART_MES,ICART_DIA,APLICA,idade_bin,canceled,is_lapse,is_midterm
216,204794292,1,120842465,H,0,2018-03-23,1959-02-10,0,67.0,N,...,1,104954140,10.0,4,1,0,61-100,1,1,0
381,204807901,1,120837574,H,0,2018-04-04,1983-04-12,0,68.0,N,...,1,223904503,6.0,4,1,0,31-60,1,1,0
382,204808105,1,3505616,V,0,2018-04-05,1957-09-01,0,73.0,N,...,1,177082410,10.0,4,1,0,61-100,1,1,0
383,204808105,2,3505617,H,1,2018-04-05,1961-12-03,0,53.0,N,...,1,177082410,10.0,4,1,0,61-100,1,1,0
384,204808105,3,120858059,H,3,2018-04-05,1998-02-22,0,55.0,N,...,1,177082410,10.0,4,1,0,21-30,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5310,207218294,5,122750892,V,3,2024-04-11,2023-09-12,0,9.0,N,...,1,323250963,8.0,4,1,0,0-1,1,1,0
5320,207218453,1,122235822,V,0,2024-04-10,1985-03-03,0,90.0,N,...,1,318143348,5.0,4,1,0,31-60,1,1,0
5321,207218453,2,122751199,H,1,2024-04-10,1986-12-11,0,65.0,N,...,1,318143348,5.0,4,1,0,31-60,1,1,0
5391,207242348,1,122792071,V,0,2024-04-30,2023-04-14,0,7.0,N,...,1,254684173,3.0,4,1,0,0-1,1,1,0


In [53]:
lapses[lapses["POLIZA"]== 204808105	
       ][["POLIZA","NASEGURADO","ASEGURADO","fefecto_poliza","FALTAASEG","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","is_midterm","FNACI8","new_ant_pessoa"]]


,POLIZA,NASEGURADO,ASEGURADO,fefecto_poliza,FALTAASEG,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,is_midterm,FNACI8,new_ant_pessoa
27076,204808105,1,3505616,2018-04-05,2018-04-05,2022-04-01,3,63,2025-04-01,2022-01-01,2022-03-31,2022,463.44,498.80,463.44,3,1,0,1957-09-01,3
27077,204808105,1,3505616,2018-04-05,2018-04-05,2022-04-01,4,64,2025-04-01,2022-04-01,2022-12-31,2022,463.44,498.80,498.80,4,1,0,1957-09-01,4
27078,204808105,1,3505616,2018-04-05,2018-04-05,2023-04-01,4,64,2025-04-01,2023-01-01,2023-03-31,2023,498.80,547.66,498.80,4,1,0,1957-09-01,4
27079,204808105,1,3505616,2018-04-05,2018-04-05,2023-04-01,5,65,2025-04-01,2023-04-01,2023-12-31,2023,498.80,547.66,547.66,5,1,0,1957-09-01,5
27080,204808105,1,3505616,2018-04-05,2018-04-05,2024-04-01,5,65,2025-04-01,2024-01-01,2024-03-31,2024,619.21,623.88,547.66,5,1,0,1957-09-01,5
27081,204808105,1,3505616,2018-04-05,2018-04-05,2024-04-01,6,66,2025-04-01,2024-04-01,2024-12-31,2024,619.21,623.88,623.88,6,1,0,1957-09-01,6
27082,204808105,1,3505616,2018-04-05,2018-04-05,2025-04-01,6,66,2025-04-01,2025-01-01,2025-03-31,2025,623.88,638.06,623.88,6,1,0,1957-09-01,6
27083,204808105,2,3505617,2018-04-05,2018-04-05,2022-04-01,3,59,2025-04-01,2022-01-01,2022-03-31,2022,328.15,353.18,328.15,3,1,0,1961-12-03,3
27084,204808105,2,3505617,2018-04-05,2018-04-05,2022-04-01,4,60,2025-04-01,2022-04-01,2022-12-31,2022,328.15,353.18,353.18,4,1,0,1961-12-03,4
27085,204808105,2,3505617,2018-04-05,2018-04-05,2023-04-01,4,60,2025-04-01,2023-01-01,2023-03-31,2023,353.18,379.14,353.18,4,1,0,1961-12-03,4


In [54]:
#checking if the filtration by the last date of renovação was correct

lapses_in_renov_period.FVTO_AC.max() # it is correct 

datetime.date(2025, 4, 30)

In [ ]:
#  creating a col that tels if its the tomador or not 
lapses_pp_renov_period["is_tomador"] = (lapses_pp_renov_period["ASEGURADO"] == lapses_pp_renov_period["TOMADOR"]).astype(int)
# so far from what i know of the data this is the best condition to identify if its indeed the tomador or not 
#from cases below i serched some in the internet and the nifs of the assegurados do not corespond to the new_nif_tomador , that makes sence when the Tp_cliente is empreza but 
#in the case whre Tp cliente is individual it does not make sence, maybe its another person payng the policy, which ever  the NAsegurado==1 is not the one who pays so the col is_tomador can be interpreted as is this pessoa segura that pays the policy ?

In [56]:
# now  ineed to see if lapses are well flagged  to do that i will:
#check if the number of rows where fbajaaseg exits == n cases is_lapse=1 
#i have 62 rows where the fbajaaseg is not null and where is_lapse is 0, so it seems that the flag is  not correct
# but sometimes  there is fbajaaseg and its not flagged as a lapse neither a midterm  because the cancelation only happens after the desired renovation perioud so i need to chef if all these cases have that nuance  
#to do that i will selec all these cases 
lapses_pp_renov_period[lapses_pp_renov_period["is_lapse"]==0]["POLIZA"].shape[0]

5082

In [57]:
# checking if there is any case where there is fbajaaseg and no lapse is marked
lapses_pp_renov_period[(lapses_pp_renov_period["fbajaaseg"].notna()) & (lapses_pp_renov_period["is_lapse"]==0) & (lapses_pp_renov_period["is_midterm"]==0)].shape[0]
# this cases correspond to cases where either the lapse or the midterm takes place after the last date of renovation so they need to exist

0

In [ ]:
mask = (lapses_pp_renov_period["fbajaaseg"].notna()) & \
       (lapses_pp_renov_period["is_lapse"] == 0) & \
       (lapses_pp_renov_period["is_midterm"] == 0)

lapses_pp_renov_period = lapses_pp_renov_period[~mask]

,POLIZA,NASEGURADO,ASEGURADO,SEXO,PARENTESCO,FALTAASEG,FNACI8,POLIZAS_45,PESO,TRANSF_SEGURO_INT,...,new_nif_tomador,score_final,ICART_MES,ICART_DIA,APLICA,idade_bin,canceled,is_lapse,is_midterm,is_tomador


In [59]:
lapses_pp_renov_period[lapses_pp_renov_period  ["POLIZA"] == 207401801      ][["POLIZA","NASEGURADO","ASEGURADO","PARENTESCO","fefecto_poliza","FALTAASEG","Ren2","FVTO_AC","tenure","IDADE","fbajaaseg","dt_start","dt_end","p_exp_year","PRIANUALAA","PRIANUALAC","premio2","ant_pessoa","is_lapse","is_midterm"]]

,POLIZA,NASEGURADO,ASEGURADO,PARENTESCO,fefecto_poliza,FALTAASEG,Ren2,FVTO_AC,tenure,IDADE,fbajaaseg,dt_start,dt_end,p_exp_year,PRIANUALAA,PRIANUALAC,premio2,ant_pessoa,is_lapse,is_midterm


In [60]:
#good so far i do not have a single case where there is fbajaaseg and i dont have a lapse or a midterm flagged
#but now i need to see the cases that are in fact a lapse but the lapse or the midter only happens after the last date of renovation 

lapses_pp_renov_period[ (lapses_pp_renov_period['fbajaaseg'] > lapses_pp_renov_period['FVTO_AC']) & (lapses_pp_renov_period['is_lapse']==1)][["POLIZA","NASEGURADO","fbajaaseg","FVTO_AC","is_lapse","is_midterm"]]

# i dont have any case where the fbajaaseg is bigger than the FVTO_AC and the lapse is flagged as 1  but this is just in this time partition for other time partitions the contrary may happen  so i ned to clean it further
# i increassed the time period of FVTO_AC and now there are cases whrer fbajaseg= FVTO_AC + 1 year 

,POLIZA,NASEGURADO,fbajaaseg,FVTO_AC,is_lapse,is_midterm


In [61]:
lapses[lapses["POLIZA"] == 207082936      ][["POLIZA","NASEGURADO","fbajaaseg","FVTO_AC","is_lapse","is_midterm"]]

,POLIZA,NASEGURADO,fbajaaseg,FVTO_AC,is_lapse,is_midterm
1546580,207082936,1,2025-12-01,2024-12-01,1,0
1546581,207082936,1,2025-12-01,2024-12-01,1,0
1546582,207082936,1,2025-12-01,2024-12-01,1,0
1546583,207082936,1,2025-12-01,2025-12-01,1,0
1546584,207082936,2,2025-12-01,2024-12-01,1,0
1546585,207082936,2,2025-12-01,2024-12-01,1,0
1546586,207082936,2,2025-12-01,2024-12-01,1,0
1546587,207082936,2,2025-12-01,2025-12-01,1,0


In [62]:
lapses_pp_renov_period['fbajaaseg'][~lapses_pp_renov_period['fbajaaseg'].isna()].max()

datetime.date(2025, 12, 11)

In [63]:
# as can be seen above the max fbajaseg is 2026, 6, 1 thus i have cancelation dates after the last date of renovation  and  that cant happen  so what i will do is to force every case where fbajaaseg  bigger than the max FVTO_AC to have is_lapse= 0 and midterm =0
#by doing that i will automatize this for wichever time band i chose
mask = (~lapses_pp_renov_period['fbajaaseg'].isna()) & (lapses_pp_renov_period['fbajaaseg'] > lapses_pp_renov_period['FVTO_AC'])
lapses_pp_renov_period.loc[mask, ['is_lapse', 'is_midterm']] = 0

In [64]:
# now checking if there are still any cases where fbajjaseg is above the max date of renovation and its flagged as a lapse or a midterm
lapses_pp_renov_period[ (lapses_pp_renov_period['fbajaaseg'] > lapses_pp_renov_period['FVTO_AC']) & ((lapses_pp_renov_period['is_lapse']==1) | (lapses_pp_renov_period['is_midterm']==1))][["POLIZA","NASEGURADO","fbajaaseg","FVTO_AC","is_lapse","is_midterm"]]
# it returned an emptu df so its good

,POLIZA,NASEGURADO,fbajaaseg,FVTO_AC,is_lapse,is_midterm


# lets do the aging of the cols 

In [65]:
mask = (lapses_pp_renov_period['is_lapse'] == 1) & (lapses_pp_renov_period['is_midterm'] == 0)
cols_to_increment = ["tenure", "IDADE", "ant_pessoa", "new_ant_pessoa"]
lapses_pp_renov_period.loc[mask, cols_to_increment] = lapses_pp_renov_period.loc[mask, cols_to_increment] + 1

# now lets see the cases where tenure == 0

In [66]:
mask = lapses_pp_renov_period["tenure"] == 0
lapses_pp_renov_period[mask]["POLIZA"]

4896    207194615
4928    207196013
4967    207197987
4968    207197994
4969    207197997
5066    207203129
5093    207204188
5107    207204859
5188    207210407
5250    207214817
5253    207215057
5254    207215057
5260    207215350
5261    207215350
5344    207220034
5358    207221153
5369    207221615
5371    207221697
5392    207244892
Name: POLIZA, dtype: int64

In [67]:
# now from the cases that are left lets see how many have tenure = 0 
lapses_pp_renov_period[(lapses_pp_renov_period["fbajaaseg"].notna()) & (lapses_pp_renov_period["is_lapse"]==0) & (lapses_pp_renov_period["tenure"]==0)&( lapses_pp_renov_period["fbajaaseg"] < lapses_pp_renov_period["FVTO_AC"])].shape[0]
#this cases correspond to the cases  of the midterm cancelations where the cancelation happens before the policy  has completed a single year 
#does it make sense to include these cases in the data???
# i dont think so because if the model is used for lapses then the tenure should be at least 1 year this will only introduce noise moreover since thiss classes are not 1 for the lapse i will me removing instances from the majority class
# its not random undersampling but it is rule-based filtering

19

In [68]:
mask = (lapses_pp_renov_period["fbajaaseg"].notna()) & \
       (lapses_pp_renov_period["is_lapse"] == 0) & \
       (lapses_pp_renov_period["tenure"] == 0) & \
       (lapses_pp_renov_period["fbajaaseg"] < lapses_pp_renov_period["FVTO_AC"])

lapses_pp_renov_period = lapses_pp_renov_period[~mask]

In [69]:
lapses_pp_renov_period[lapses_pp_renov_period["score_final"].isna()].shape[0]

1

In [70]:
lapses_pp_renov_period.shape

(5383, 355)

In [ ]:

import numpy as np

# Create the imputed score column in scores
scores["score_final_imputed"] = np.where(
    ~scores["score_final_cv1"].isna(),
    scores["score_final_cv1"],
    scores["score_final_cv2"]
)

# Ensure unique index by grouping and taking the first imputed value
impute_map = scores.groupby("new_nif_tomador")["score_final_imputed"].first()

# Fill NaNs in lapses_pp_renov_period["score_final"] using the mapping
lapses_pp_renov_period["score"] = lapses_pp_renov_period["score_final"].fillna(
    lapses_pp_renov_period["new_nif_tomador"].map(impute_map)
)


In [72]:
lapses_pp_renov_period[lapses_pp_renov_period["score_final"].isna()].shape[0]


0

In [ ]:
# Fill remaining NaN values in score_final with 5
lapses_pp_renov_period["score"] = lapses_pp_renov_period["score"].fillna(5)

In [74]:
nan_percent_pp = (lapses_pp_renov_period.isna().mean() * 100).reset_index()
nan_percent_pp.columns = ['column', 'percent_nan']
nan_percent_pp['num_nan'] = lapses_pp_renov_period.isna().sum().values

In [75]:
lapses.codpost_tomador

0          2830-054
1          2830-054
2          2830-054
3          2830-054
4          4575-019
             ...   
1638220    2955-202
1638221    4430-945
1638222    2100-651
1638223    3040-692
1638224    3040-692
Name: codpost_tomador, Length: 1638225, dtype: str

In [76]:
lapses_pp_renov_period.columns.get_loc("Avg_Number_Floors")

58

In [77]:
lapses_pp_renov_period.columns.get_loc("Concentracao")

75

In [78]:
lapses_pp_renov_period.columns[54:69]

Index(['codine_tomador', 'cp4', 'codine_tomador_n', 'cp7', 'Avg_Number_Floors',
       'Crime_rate_2011', 'Crime_rate_2012', 'Crime_rate_2013',
       'Crime_rate_2014', 'Crime_rate_2015', 'Crime_rate_Avg', 'zip_code',
       'households', 'inhabitants', 'purch_power_Euro'],
      dtype='str')

In [79]:
# Example usage:
# Suppose you want to fill NaNs in columns 'col1' and 'col2' using the postcode column 'codpost_tom'

cols_to_fill = ['Avg_Number_Floors', 'Crime_rate_2011', 'Crime_rate_2012',
       'Crime_rate_2013', 'Crime_rate_2014', 'Crime_rate_2015',
       'Crime_rate_Avg', 'households', 'inhabitants', 'purch_power_Euro',
       'Distrito', 'Concelho', 'N_Clientes', 'N_Prestadores', 'N_Domicilios']
lapses_pp_renov_period = fillna_by_postcode_mode_vectorized(lapses_pp_renov_period, cols_to_fill, postcode_col='codpost_tom')

In [80]:
nan_percent_pp = (lapses_pp_renov_period.isna().mean() * 100).reset_index()
nan_percent_pp.columns = ['column', 'percent_nan']
nan_percent_pp['num_nan'] = lapses_pp_renov_period.isna().sum().values

In [81]:
lapses_pp_renov_period.columns.get_loc("Freguesia_Final_Pos_RATF")

118

In [82]:
lapses_pp_renov_period.columns.get_loc("conc_ratf_github")

224

In [83]:
lapses_pp_renov_period.columns[104:160]

Index(['Capital_Hosp_Cirurgia', 'Capital_Prot_n_Oculares',
       'Capital_Prot_Oculares', 'Bin_DENTAL', 'Bin_assist_port',
       'Capital_Cob_Medica_Int', 'Bin_Cob_Medica_Int', 'Capital_Consultas',
       'Bin_Consultas', 'Bin_Medicamentos', 'Capital_Medicamentos',
       'Capital_Estomat_Consult_e_Trat', 'Cap_Oncologia', 'CodigoPostal',
       'Freguesia_Final_Pos_RATF', 'POP_densidade_populacao',
       'POP_relacao_masculinidade', 'POP_indice_envelhecimento',
       'POP_indice_dependencia_idosos', 'POP_durac_media_mov_pendular',
       'POP_taxa_desemprego', 'POP_ES_completo', 'POP_residente_idade_media',
       'POP_ADP_dim_media', 'POP_AI_dim_media', 'POP_NclF_monoparentais',
       'POP_NclF_com_filhos', 'POP_meio_transp_total_abs',
       'POP_PMT_index_trafego_auto', 'POP_meio_transp_index_saudavel',
       'POP_index_trafg_dens_auto', 'POP_index_trafg_dens_temp_auto',
       'POP_index_trafg_dens_saudavel', 'POP_ix_trafg_dens_temp_saudavel',
       'POP_index_trafg_dens_mot

In [84]:
lapses_pp_renov_period.columns[160:210]

Index(['POP_NclF_nfilhos01_1_filhosEnt', 'POP_NclF_nfilhos02_2_filhosEnt',
       'POP_NclF_nfilhos03_3_filhosEnt', 'POP_NclF_nfilhos04_4_m_filhosEnt',
       'POP_NclF_nfilhos05_tot_absoluto', 'POL_ESAE01_educacao',
       'POL_ESAE02_artes_humanidades', 'POL_ESAE03_cienc_soc_jornal_info',
       'POL_ESAE04_cienc_empr_admn_dir', 'POL_ESAE05_cienc_natur_mat_esta',
       'POL_ESAE06_tecnolog_info_com', 'POL_ESAE07_engen_industri_const',
       'POL_ESAE08_agric_sivic_pesca_vet', 'POL_ESAE09_saude_protecao_social',
       'POL_ESAE10_servicos', 'POP_emp_CAE24_Total_absoluto',
       'POP_emp_CAE01_Agric_PA_C_Fl_Pesc', 'POP_emp_CAE02_industr_extrativas',
       'POP_emp_CAE03_indust_transformad', 'POP_emp_CAE04_eletr_G_V_AQF_AF',
       'POP_emp_CAE05_CTD_agua_SGRD', 'POP_emp_CAE06_construcao',
       'POP_emp_CAE07_comrc_GR_repar_VAM', 'POP_emp_CAE08_transpo_armazenag',
       'POP_emp_CAE09_aloj_resta_simil', 'POP_emp_CAE10_actv_info_comunic',
       'POP_emp_CAE11_actv_finan_seguros'

In [85]:
# Example usage:
# Suppose you want to fill NaNs in columns 'col1' and 'col2' using the postcode column 'codpost_tom'

cols_to_fill = ['Freguesia_Final_Pos_RATF', 'POP_densidade_populacao',
       'POP_relacao_masculinidade', 'POP_indice_envelhecimento',
       'POP_indice_dependencia_idosos', 'POP_durac_media_mov_pendular',
       'POP_taxa_desemprego', 'POP_ES_completo', 'POP_residente_idade_media',
       'POP_ADP_dim_media', 'POP_AI_dim_media', 'POP_NclF_monoparentais',
       'POP_NclF_com_filhos', 'POP_meio_transp_total_abs',
       'POP_PMT_index_trafego_auto', 'POP_meio_transp_index_saudavel',
       'POP_index_trafg_dens_auto', 'POP_index_trafg_dens_temp_auto',
       'POP_index_trafg_dens_saudavel', 'POP_ix_trafg_dens_temp_saudavel',
       'POP_index_trafg_dens_motjov', 'POP_index_trafg_dens_temp_motjov',
       'POP_idade01_0_14anos', 'POP_idade02_15_19anos',
       'POP_idade03_20_24anos', 'POP_idade04_25_29anos',
       'POP_idade05_30_34anos', 'POP_idade06_35_39anos',
       'POP_idade07_40_44anos', 'POP_idade08_45_49anos',
       'POP_idade09_50_54anos', 'POP_idade10_55_59anos',
       'POP_idade11_60_64anos', 'POP_idade12_65_69anos',
       'POP_idade13_70_74anos', 'POP_idade14_75_mais_anos',
       'POP_idade15_residente_total_abs', 'POP_ativa_ecivil01_solteiro',
       'POP_ativa_ecivil02_casado', 'POP_ativa_ecivil03_viuvo',
       'POP_ativa_ecivil04_divorciado', 'POP_NclF_nfilhos00_0_filhosEnt',
       'POP_NclF_nfilhos01_1_filhosEnt', 'POP_NclF_nfilhos02_2_filhosEnt',
       'POP_NclF_nfilhos03_3_filhosEnt', 'POP_NclF_nfilhos04_4_m_filhosEnt',
       'POP_NclF_nfilhos05_tot_absoluto', 'POL_ESAE01_educacao',
       'POL_ESAE02_artes_humanidades', 'POL_ESAE03_cienc_soc_jornal_info',
       'POL_ESAE04_cienc_empr_admn_dir', 'POL_ESAE05_cienc_natur_mat_esta',
       'POL_ESAE06_tecnolog_info_com', 'POL_ESAE07_engen_industri_const',
       'POL_ESAE08_agric_sivic_pesca_vet', 'POL_ESAE09_saude_protecao_social','POL_ESAE10_servicos', 'POP_emp_CAE24_Total_absoluto',
       'POP_emp_CAE01_Agric_PA_C_Fl_Pesc', 'POP_emp_CAE02_industr_extrativas',
       'POP_emp_CAE03_indust_transformad', 'POP_emp_CAE04_eletr_G_V_AQF_AF',
       'POP_emp_CAE05_CTD_agua_SGRD', 'POP_emp_CAE06_construcao',
       'POP_emp_CAE07_comrc_GR_repar_VAM', 'POP_emp_CAE08_transpo_armazenag',
       'POP_emp_CAE09_aloj_resta_simil', 'POP_emp_CAE10_actv_info_comunic',
       'POP_emp_CAE11_actv_finan_seguros', 'POP_emp_CAE12_actv_imabiliarias',
       'POP_emp_CAE13_actv_consulto_CTS', 'POP_emp_CAE14_act_adm_serv_apoio',
       'POP_emp_CAE15_adm_publ_def_SSO', 'POP_emp_CAE16_educacao',
       'POP_emp_CAE17_actv_saud_hum_AS', 'POP_emp_CAE18_actv_artist_EDR',
       'POP_emp_CAE19_outr_act_servc', 'POP_emp_CAE20_ac_fam_emp_PDAPFUP',
       'POP_emp_CAE21_act_org_int_OIET', 'POP_GSE01_emprs_prof_intelect_CT',
       'POP_GSE02_quadros_intelectuais_C', 'POP_GSE03_forcas_armadas',
       'POP_GSE04_trab_n_qualif_SP', 'POP_GSE05_operar_n_qualif',
       'POP_GSE06_trab_adm_comerc_SNQ', 'POP_GSE07_assalariados_SP',
       'POP_GSE08_operar_qualificados_SQ', 'POP_GSE09_emp_admin_comerc_serv',
       'POP_GSE10_quadros_admin_interm', 'POP_GSE11_quadros_tecn_interm',
       'POP_GSE12_diretor_quadros_DEMGE', 'POP_GSE13_emprs_ind_com_serv',
       'POP_GSE14_trab_independent_SP', 'POP_GSE15_prestad_serv_CI',
       'POP_GSE16_trab_industr_AI', 'POP_GSE17_prof_tecn_interm_I',
       'POP_GSE18_prof_intelc_cientf', 'POP_GSE19_pequenos_patroes_SP',
       'POP_GSE20_pequenos_patroes_CS', 'POP_GSE21_pequenos_patroes_I',
       'POP_GSE22_pequenos_patroes_PTI', 'POP_GSE23_pequenos_patroes_PIC',
       'POP_GSE24_emprs_SP', 'POP_GSE25_outras_pess_atv_NE',
       'POP_GSE26_pessoas_inativas', 'freg_ratf_github']
lapses_pp_renov_period = fillna_by_postcode_mode_vectorized(lapses_pp_renov_period, cols_to_fill, postcode_col='codpost_tom')

In [ ]:
lapses_pp_renov_period["FECHA_PREEXISTENCIA"] = lapses_pp_renov_period["FECHA_PREEXISTENCIA"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)
lapses_pp_renov_period["data_pre_existencia"] = lapses_pp_renov_period["data_pre_existencia"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)


In [ ]:
lapses_pp_renov_period["PRIANUALAA"] = lapses_pp_renov_period["PRIANUALAA"].fillna(lapses_pp_renov_period["aux_prem_aa"])
lapses_pp_renov_period["PRIANUALAC"] = lapses_pp_renov_period["PRIANUALAC"].fillna(lapses_pp_renov_period["aux_prem_ac"])


In [ ]:
mean_value = lapses_pp_renov_period["new_concentracao"].mean(skipna=True)
mask = lapses_pp_renov_period["new_concentracao"].isna() | (lapses_pp_renov_period["new_concentracao"] > 0.18)
lapses_pp_renov_period.loc[mask, "new_concentracao"] = mean_value

In [ ]:
comparison = lapses_pp_renov_period["premio2"] == lapses_pp_renov_period["PRIANUALAC"]
comparison

In [ ]:
mask = lapses_pp_renov_period["FALTAASEG"].apply(lambda x: pd.to_datetime(x) > pd.to_datetime("2024-12-31"))
a=lapses_pp_renov_period[mask][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year","FALTAASEG","tenure","dt_start","dt_end","is_lapse","premio2"]]

In [ ]:
renov_period

In [ ]:
#i still have some cases i need to deal with namelly the cases where the FALTASEG is in 2025 for this FVTOAC it does not make sense because they will not even have a total 9 months exposure (12-3) so i need to exclude the cases where (   FVTOAC - FALTASEG ) is less than 9 months 
# this is a better condition than just removing cases when Faltaseg is before 2024 because this automates the process when im chosing the renovation perioud 
mask = (pd.to_datetime(lapses_pp_renov_period["FVTO_AC"]) - pd.to_datetime(lapses_pp_renov_period["FALTAASEG"])).dt.days < 270
print(lapses_pp_renov_period[mask].shape)
lapses_pp_renov_period = lapses_pp_renov_period[~mask]


In [ ]:
lapses_pp_renov_period.to_parquet("lapses_pp_renov_period_before_removing_cols.parquet")

In [ ]:
#lets keep removing cols 
cols_to_remove = ["FALTAASEG","FNACI8","POLIZAS_45","FECHA_PREEXISTENCIA","FCob","fefecto_poliza","RENOVACAO","INICIO","FIM","EXPOSICAO","fecha","aux_prem_aa","aux_prem_ac","aux_ren","aux_gar","aux_ren_apol","prem_final","prem_exp","Ren2","ren_apolice2","Ren_final",
                  "ren_inicial","premio2","data_inicio2","dt_antiga","data_pre_existencia","aux_ren2","dif_meses","dt_start","dt_end","n_claims_dia2","EXPOSICAO_v2","exp_Hosp_Cirurgia_e_Parto","exp_DENTAL","exp_assist_port","exp_Cob_Medica_Int","exp_Consultas","exp_Medicamentos","final_premium","new_concentracao",
                  "Nr_Line","key","marca_inicio_new","marca_fim_new","ICART_DIA","APLICA","idade_bin","canceled"]

lapses_pp_renov_period = lapses_pp_renov_period.drop(columns=cols_to_remove)


In [ ]:
lapses_pp_renov_period["FECHA_PREEXISTENCIA"] = lapses_pp_renov_period["FECHA_PREEXISTENCIA"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)
lapses_pp_renov_period["data_pre_existencia"] = lapses_pp_renov_period["data_pre_existencia"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)


In [ ]:
lapses_pp_renov_period["PRIANUALAA"] = lapses_pp_renov_period["PRIANUALAA"].fillna(lapses_pp_renov_period["aux_prem_aa"])
lapses_pp_renov_period["PRIANUALAC"] = lapses_pp_renov_period["PRIANUALAC"].fillna(lapses_pp_renov_period["aux_prem_ac"])


In [ ]:
mean_value = lapses_pp_renov_period["new_concentracao"].mean(skipna=True)
mask = lapses_pp_renov_period["new_concentracao"].isna() | (lapses_pp_renov_period["new_concentracao"] > 0.18)
lapses_pp_renov_period.loc[mask, "new_concentracao"] = mean_value

In [ ]:
comparison = lapses_pp_renov_period["premio2"] == lapses_pp_renov_period["PRIANUALAC"]
comparison

In [ ]:
mask = lapses_pp_renov_period["FALTAASEG"].apply(lambda x: pd.to_datetime(x) > pd.to_datetime("2024-12-31"))
a=lapses_pp_renov_period[mask][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year","FALTAASEG","tenure","dt_start","dt_end","is_lapse","premio2"]]

In [ ]:
renov_period

In [ ]:
#i still have some cases i need to deal with namelly the cases where the FALTASEG is in 2025 for this FVTOAC it does not make sense because they will not even have a total 9 months exposure (12-3) so i need to exclude the cases where (   FVTOAC - FALTASEG ) is less than 9 months 
# this is a better condition than just removing cases when Faltaseg is before 2024 because this automates the process when im chosing the renovation perioud 
mask = (pd.to_datetime(lapses_pp_renov_period["FVTO_AC"]) - pd.to_datetime(lapses_pp_renov_period["FALTAASEG"])).dt.days < 270
print(lapses_pp_renov_period[mask].shape)
lapses_pp_renov_period = lapses_pp_renov_period[~mask]


In [ ]:
lapses_pp_renov_period.to_parquet("lapses_pp_renov_period_before_removing_cols.parquet")

In [ ]:
#lets keep removing cols 
cols_to_remove = ["FALTAASEG","FNACI8","POLIZAS_45","FECHA_PREEXISTENCIA","FCob","fefecto_poliza","RENOVACAO","INICIO","FIM","EXPOSICAO","fecha","aux_prem_aa","aux_prem_ac","aux_ren","aux_gar","aux_ren_apol","prem_final","prem_exp","Ren2","ren_apolice2","Ren_final",
                  "ren_inicial","premio2","data_inicio2","dt_antiga","data_pre_existencia","aux_ren2","dif_meses","dt_start","dt_end","n_claims_dia2","EXPOSICAO_v2","exp_Hosp_Cirurgia_e_Parto","exp_DENTAL","exp_assist_port","exp_Cob_Medica_Int","exp_Consultas","exp_Medicamentos","final_premium","new_concentracao",
                  "Nr_Line","key","marca_inicio_new","marca_fim_new","ICART_DIA","APLICA","idade_bin","canceled"]

lapses_pp_renov_period = lapses_pp_renov_period.drop(columns=cols_to_remove)


In [ ]:
lapses_pp_renov_period["FECHA_PREEXISTENCIA"] = lapses_pp_renov_period["FECHA_PREEXISTENCIA"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)
lapses_pp_renov_period["data_pre_existencia"] = lapses_pp_renov_period["data_pre_existencia"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)


In [ ]:
lapses_pp_renov_period["PRIANUALAA"] = lapses_pp_renov_period["PRIANUALAA"].fillna(lapses_pp_renov_period["aux_prem_aa"])
lapses_pp_renov_period["PRIANUALAC"] = lapses_pp_renov_period["PRIANUALAC"].fillna(lapses_pp_renov_period["aux_prem_ac"])


In [ ]:
mean_value = lapses_pp_renov_period["new_concentracao"].mean(skipna=True)
mask = lapses_pp_renov_period["new_concentracao"].isna() | (lapses_pp_renov_period["new_concentracao"] > 0.18)
lapses_pp_renov_period.loc[mask, "new_concentracao"] = mean_value

In [ ]:
comparison = lapses_pp_renov_period["premio2"] == lapses_pp_renov_period["PRIANUALAC"]
comparison

In [ ]:
mask = lapses_pp_renov_period["FALTAASEG"].apply(lambda x: pd.to_datetime(x) > pd.to_datetime("2024-12-31"))
a=lapses_pp_renov_period[mask][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year","FALTAASEG","tenure","dt_start","dt_end","is_lapse","premio2"]]

In [ ]:
renov_period

In [ ]:
#i still have some cases i need to deal with namelly the cases where the FALTASEG is in 2025 for this FVTOAC it does not make sense because they will not even have a total 9 months exposure (12-3) so i need to exclude the cases where (   FVTOAC - FALTASEG ) is less than 9 months 
# this is a better condition than just removing cases when Faltaseg is before 2024 because this automates the process when im chosing the renovation perioud 
mask = (pd.to_datetime(lapses_pp_renov_period["FVTO_AC"]) - pd.to_datetime(lapses_pp_renov_period["FALTAASEG"])).dt.days < 270
print(lapses_pp_renov_period[mask].shape)
lapses_pp_renov_period = lapses_pp_renov_period[~mask]


In [ ]:
lapses_pp_renov_period.to_parquet("lapses_pp_renov_period_before_removing_cols.parquet")

In [ ]:
#lets keep removing cols 
cols_to_remove = ["FALTAASEG","FNACI8","POLIZAS_45","FECHA_PREEXISTENCIA","FCob","fefecto_poliza","RENOVACAO","INICIO","FIM","EXPOSICAO","fecha","aux_prem_aa","aux_prem_ac","aux_ren","aux_gar","aux_ren_apol","prem_final","prem_exp","Ren2","ren_apolice2","Ren_final",
                  "ren_inicial","premio2","data_inicio2","dt_antiga","data_pre_existencia","aux_ren2","dif_meses","dt_start","dt_end","n_claims_dia2","EXPOSICAO_v2","exp_Hosp_Cirurgia_e_Parto","exp_DENTAL","exp_assist_port","exp_Cob_Medica_Int","exp_Consultas","exp_Medicamentos","final_premium","new_concentracao",
                  "Nr_Line","key","marca_inicio_new","marca_fim_new","ICART_DIA","APLICA","idade_bin","canceled"]

lapses_pp_renov_period = lapses_pp_renov_period.drop(columns=cols_to_remove)


In [ ]:
lapses_pp_renov_period["FECHA_PREEXISTENCIA"] = lapses_pp_renov_period["FECHA_PREEXISTENCIA"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)
lapses_pp_renov_period["data_pre_existencia"] = lapses_pp_renov_period["data_pre_existencia"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)


In [ ]:
lapses_pp_renov_period["PRIANUALAA"] = lapses_pp_renov_period["PRIANUALAA"].fillna(lapses_pp_renov_period["aux_prem_aa"])
lapses_pp_renov_period["PRIANUALAC"] = lapses_pp_renov_period["PRIANUALAC"].fillna(lapses_pp_renov_period["aux_prem_ac"])


In [ ]:
mean_value = lapses_pp_renov_period["new_concentracao"].mean(skipna=True)
mask = lapses_pp_renov_period["new_concentracao"].isna() | (lapses_pp_renov_period["new_concentracao"] > 0.18)
lapses_pp_renov_period.loc[mask, "new_concentracao"] = mean_value

In [ ]:
comparison = lapses_pp_renov_period["premio2"] == lapses_pp_renov_period["PRIANUALAC"]
comparison

In [ ]:
mask = lapses_pp_renov_period["FALTAASEG"].apply(lambda x: pd.to_datetime(x) > pd.to_datetime("2024-12-31"))
a=lapses_pp_renov_period[mask][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year","FALTAASEG","tenure","dt_start","dt_end","is_lapse","premio2"]]

In [ ]:
renov_period

In [ ]:
#i still have some cases i need to deal with namelly the cases where the FALTASEG is in 2025 for this FVTOAC it does not make sense because they will not even have a total 9 months exposure (12-3) so i need to exclude the cases where (   FVTOAC - FALTASEG ) is less than 9 months 
# this is a better condition than just removing cases when Faltaseg is before 2024 because this automates the process when im chosing the renovation perioud 
mask = (pd.to_datetime(lapses_pp_renov_period["FVTO_AC"]) - pd.to_datetime(lapses_pp_renov_period["FALTAASEG"])).dt.days < 270
print(lapses_pp_renov_period[mask].shape)
lapses_pp_renov_period = lapses_pp_renov_period[~mask]


In [ ]:
lapses_pp_renov_period.to_parquet("lapses_pp_renov_period_before_removing_cols.parquet")

In [ ]:
#lets keep removing cols 
cols_to_remove = ["FALTAASEG","FNACI8","POLIZAS_45","FECHA_PREEXISTENCIA","FCob","fefecto_poliza","RENOVACAO","INICIO","FIM","EXPOSICAO","fecha","aux_prem_aa","aux_prem_ac","aux_ren","aux_gar","aux_ren_apol","prem_final","prem_exp","Ren2","ren_apolice2","Ren_final",
                  "ren_inicial","premio2","data_inicio2","dt_antiga","data_pre_existencia","aux_ren2","dif_meses","dt_start","dt_end","n_claims_dia2","EXPOSICAO_v2","exp_Hosp_Cirurgia_e_Parto","exp_DENTAL","exp_assist_port","exp_Cob_Medica_Int","exp_Consultas","exp_Medicamentos","final_premium","new_concentracao",
                  "Nr_Line","key","marca_inicio_new","marca_fim_new","ICART_DIA","APLICA","idade_bin","canceled"]

lapses_pp_renov_period = lapses_pp_renov_period.drop(columns=cols_to_remove)


In [ ]:
lapses_pp_renov_period["FECHA_PREEXISTENCIA"] = lapses_pp_renov_period["FECHA_PREEXISTENCIA"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)
lapses_pp_renov_period["data_pre_existencia"] = lapses_pp_renov_period["data_pre_existencia"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)


In [ ]:
lapses_pp_renov_period["PRIANUALAA"] = lapses_pp_renov_period["PRIANUALAA"].fillna(lapses_pp_renov_period["aux_prem_aa"])
lapses_pp_renov_period["PRIANUALAC"] = lapses_pp_renov_period["PRIANUALAC"].fillna(lapses_pp_renov_period["aux_prem_ac"])


In [ ]:
mean_value = lapses_pp_renov_period["new_concentracao"].mean(skipna=True)
mask = lapses_pp_renov_period["new_concentracao"].isna() | (lapses_pp_renov_period["new_concentracao"] > 0.18)
lapses_pp_renov_period.loc[mask, "new_concentracao"] = mean_value

In [ ]:
comparison = lapses_pp_renov_period["premio2"] == lapses_pp_renov_period["PRIANUALAC"]
comparison

In [ ]:
mask = lapses_pp_renov_period["FALTAASEG"].apply(lambda x: pd.to_datetime(x) > pd.to_datetime("2024-12-31"))
a=lapses_pp_renov_period[mask][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year","FALTAASEG","tenure","dt_start","dt_end","is_lapse","premio2"]]

In [ ]:
renov_period

In [ ]:
#i still have some cases i need to deal with namelly the cases where the FALTASEG is in 2025 for this FVTOAC it does not make sense because they will not even have a total 9 months exposure (12-3) so i need to exclude the cases where (   FVTOAC - FALTASEG ) is less than 9 months 
# this is a better condition than just removing cases when Faltaseg is before 2024 because this automates the process when im chosing the renovation perioud 
mask = (pd.to_datetime(lapses_pp_renov_period["FVTO_AC"]) - pd.to_datetime(lapses_pp_renov_period["FALTAASEG"])).dt.days < 270
print(lapses_pp_renov_period[mask].shape)
lapses_pp_renov_period = lapses_pp_renov_period[~mask]


In [ ]:
lapses_pp_renov_period.to_parquet("lapses_pp_renov_period_before_removing_cols.parquet")

In [ ]:
#lets keep removing cols 
cols_to_remove = ["FALTAASEG","FNACI8","POLIZAS_45","FECHA_PREEXISTENCIA","FCob","fefecto_poliza","RENOVACAO","INICIO","FIM","EXPOSICAO","fecha","aux_prem_aa","aux_prem_ac","aux_ren","aux_gar","aux_ren_apol","prem_final","prem_exp","Ren2","ren_apolice2","Ren_final",
                  "ren_inicial","premio2","data_inicio2","dt_antiga","data_pre_existencia","aux_ren2","dif_meses","dt_start","dt_end","n_claims_dia2","EXPOSICAO_v2","exp_Hosp_Cirurgia_e_Parto","exp_DENTAL","exp_assist_port","exp_Cob_Medica_Int","exp_Consultas","exp_Medicamentos","final_premium","new_concentracao",
                  "Nr_Line","key","marca_inicio_new","marca_fim_new","ICART_DIA","APLICA","idade_bin","canceled"]

lapses_pp_renov_period = lapses_pp_renov_period.drop(columns=cols_to_remove)


In [ ]:
lapses_pp_renov_period["FECHA_PREEXISTENCIA"] = lapses_pp_renov_period["FECHA_PREEXISTENCIA"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)
lapses_pp_renov_period["data_pre_existencia"] = lapses_pp_renov_period["data_pre_existencia"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)


In [ ]:
lapses_pp_renov_period["PRIANUALAA"] = lapses_pp_renov_period["PRIANUALAA"].fillna(lapses_pp_renov_period["aux_prem_aa"])
lapses_pp_renov_period["PRIANUALAC"] = lapses_pp_renov_period["PRIANUALAC"].fillna(lapses_pp_renov_period["aux_prem_ac"])


In [ ]:
mean_value = lapses_pp_renov_period["new_concentracao"].mean(skipna=True)
mask = lapses_pp_renov_period["new_concentracao"].isna() | (lapses_pp_renov_period["new_concentracao"] > 0.18)
lapses_pp_renov_period.loc[mask, "new_concentracao"] = mean_value

In [ ]:
comparison = lapses_pp_renov_period["premio2"] == lapses_pp_renov_period["PRIANUALAC"]
comparison

In [ ]:
mask = lapses_pp_renov_period["FALTAASEG"].apply(lambda x: pd.to_datetime(x) > pd.to_datetime("2024-12-31"))
a=lapses_pp_renov_period[mask][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year","FALTAASEG","tenure","dt_start","dt_end","is_lapse","premio2"]]

In [ ]:
renov_period

In [ ]:
#i still have some cases i need to deal with namelly the cases where the FALTASEG is in 2025 for this FVTOAC it does not make sense because they will not even have a total 9 months exposure (12-3) so i need to exclude the cases where (   FVTOAC - FALTASEG ) is less than 9 months 
# this is a better condition than just removing cases when Faltaseg is before 2024 because this automates the process when im chosing the renovation perioud 
mask = (pd.to_datetime(lapses_pp_renov_period["FVTO_AC"]) - pd.to_datetime(lapses_pp_renov_period["FALTAASEG"])).dt.days < 270
print(lapses_pp_renov_period[mask].shape)
lapses_pp_renov_period = lapses_pp_renov_period[~mask]


In [ ]:
lapses_pp_renov_period.to_parquet("lapses_pp_renov_period_before_removing_cols.parquet")

In [ ]:
#lets keep removing cols 
cols_to_remove = ["FALTAASEG","FNACI8","POLIZAS_45","FECHA_PREEXISTENCIA","FCob","fefecto_poliza","RENOVACAO","INICIO","FIM","EXPOSICAO","fecha","aux_prem_aa","aux_prem_ac","aux_ren","aux_gar","aux_ren_apol","prem_final","prem_exp","Ren2","ren_apolice2","Ren_final",
                  "ren_inicial","premio2","data_inicio2","dt_antiga","data_pre_existencia","aux_ren2","dif_meses","dt_start","dt_end","n_claims_dia2","EXPOSICAO_v2","exp_Hosp_Cirurgia_e_Parto","exp_DENTAL","exp_assist_port","exp_Cob_Medica_Int","exp_Consultas","exp_Medicamentos","final_premium","new_concentracao",
                  "Nr_Line","key","marca_inicio_new","marca_fim_new","ICART_DIA","APLICA","idade_bin","canceled"]

lapses_pp_renov_period = lapses_pp_renov_period.drop(columns=cols_to_remove)


In [ ]:
lapses_pp_renov_period["FECHA_PREEXISTENCIA"] = lapses_pp_renov_period["FECHA_PREEXISTENCIA"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)
lapses_pp_renov_period["data_pre_existencia"] = lapses_pp_renov_period["data_pre_existencia"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)


In [ ]:
lapses_pp_renov_period["PRIANUALAA"] = lapses_pp_renov_period["PRIANUALAA"].fillna(lapses_pp_renov_period["aux_prem_aa"])
lapses_pp_renov_period["PRIANUALAC"] = lapses_pp_renov_period["PRIANUALAC"].fillna(lapses_pp_renov_period["aux_prem_ac"])


In [ ]:
mean_value = lapses_pp_renov_period["new_concentracao"].mean(skipna=True)
mask = lapses_pp_renov_period["new_concentracao"].isna() | (lapses_pp_renov_period["new_concentracao"] > 0.18)
lapses_pp_renov_period.loc[mask, "new_concentracao"] = mean_value

In [ ]:
comparison = lapses_pp_renov_period["premio2"] == lapses_pp_renov_period["PRIANUALAC"]
comparison

In [ ]:
mask = lapses_pp_renov_period["FALTAASEG"].apply(lambda x: pd.to_datetime(x) > pd.to_datetime("2024-12-31"))
a=lapses_pp_renov_period[mask][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year","FALTAASEG","tenure","dt_start","dt_end","is_lapse","premio2"]]

In [ ]:
renov_period

In [ ]:
#i still have some cases i need to deal with namelly the cases where the FALTASEG is in 2025 for this FVTOAC it does not make sense because they will not even have a total 9 months exposure (12-3) so i need to exclude the cases where (   FVTOAC - FALTASEG ) is less than 9 months 
# this is a better condition than just removing cases when Faltaseg is before 2024 because this automates the process when im chosing the renovation perioud 
mask = (pd.to_datetime(lapses_pp_renov_period["FVTO_AC"]) - pd.to_datetime(lapses_pp_renov_period["FALTAASEG"])).dt.days < 270
print(lapses_pp_renov_period[mask].shape)
lapses_pp_renov_period = lapses_pp_renov_period[~mask]


In [ ]:
lapses_pp_renov_period.to_parquet("lapses_pp_renov_period_before_removing_cols.parquet")

In [ ]:
#lets keep removing cols 
cols_to_remove = ["FALTAASEG","FNACI8","POLIZAS_45","FECHA_PREEXISTENCIA","FCob","fefecto_poliza","RENOVACAO","INICIO","FIM","EXPOSICAO","fecha","aux_prem_aa","aux_prem_ac","aux_ren","aux_gar","aux_ren_apol","prem_final","prem_exp","Ren2","ren_apolice2","Ren_final",
                  "ren_inicial","premio2","data_inicio2","dt_antiga","data_pre_existencia","aux_ren2","dif_meses","dt_start","dt_end","n_claims_dia2","EXPOSICAO_v2","exp_Hosp_Cirurgia_e_Parto","exp_DENTAL","exp_assist_port","exp_Cob_Medica_Int","exp_Consultas","exp_Medicamentos","final_premium","new_concentracao",
                  "Nr_Line","key","marca_inicio_new","marca_fim_new","ICART_DIA","APLICA","idade_bin","canceled"]

lapses_pp_renov_period = lapses_pp_renov_period.drop(columns=cols_to_remove)


In [ ]:
lapses_pp_renov_period["FECHA_PREEXISTENCIA"] = lapses_pp_renov_period["FECHA_PREEXISTENCIA"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)
lapses_pp_renov_period["data_pre_existencia"] = lapses_pp_renov_period["data_pre_existencia"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)


In [ ]:
lapses_pp_renov_period["PRIANUALAA"] = lapses_pp_renov_period["PRIANUALAA"].fillna(lapses_pp_renov_period["aux_prem_aa"])
lapses_pp_renov_period["PRIANUALAC"] = lapses_pp_renov_period["PRIANUALAC"].fillna(lapses_pp_renov_period["aux_prem_ac"])


In [ ]:
mean_value = lapses_pp_renov_period["new_concentracao"].mean(skipna=True)
mask = lapses_pp_renov_period["new_concentracao"].isna() | (lapses_pp_renov_period["new_concentracao"] > 0.18)
lapses_pp_renov_period.loc[mask, "new_concentracao"] = mean_value

In [ ]:
comparison = lapses_pp_renov_period["premio2"] == lapses_pp_renov_period["PRIANUALAC"]
comparison

In [ ]:
mask = lapses_pp_renov_period["FALTAASEG"].apply(lambda x: pd.to_datetime(x) > pd.to_datetime("2024-12-31"))
a=lapses_pp_renov_period[mask][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year","FALTAASEG","tenure","dt_start","dt_end","is_lapse","premio2"]]

In [ ]:
renov_period

In [ ]:
#i still have some cases i need to deal with namelly the cases where the FALTASEG is in 2025 for this FVTOAC it does not make sense because they will not even have a total 9 months exposure (12-3) so i need to exclude the cases where (   FVTOAC - FALTASEG ) is less than 9 months 
# this is a better condition than just removing cases when Faltaseg is before 2024 because this automates the process when im chosing the renovation perioud 
mask = (pd.to_datetime(lapses_pp_renov_period["FVTO_AC"]) - pd.to_datetime(lapses_pp_renov_period["FALTAASEG"])).dt.days < 270
print(lapses_pp_renov_period[mask].shape)
lapses_pp_renov_period = lapses_pp_renov_period[~mask]


In [ ]:
lapses_pp_renov_period.to_parquet("lapses_pp_renov_period_before_removing_cols.parquet")

In [ ]:
#lets keep removing cols 
cols_to_remove = ["FALTAASEG","FNACI8","POLIZAS_45","FECHA_PREEXISTENCIA","FCob","fefecto_poliza","RENOVACAO","INICIO","FIM","EXPOSICAO","fecha","aux_prem_aa","aux_prem_ac","aux_ren","aux_gar","aux_ren_apol","prem_final","prem_exp","Ren2","ren_apolice2","Ren_final",
                  "ren_inicial","premio2","data_inicio2","dt_antiga","data_pre_existencia","aux_ren2","dif_meses","dt_start","dt_end","n_claims_dia2","EXPOSICAO_v2","exp_Hosp_Cirurgia_e_Parto","exp_DENTAL","exp_assist_port","exp_Cob_Medica_Int","exp_Consultas","exp_Medicamentos","final_premium","new_concentracao",
                  "Nr_Line","key","marca_inicio_new","marca_fim_new","ICART_DIA","APLICA","idade_bin","canceled"]

lapses_pp_renov_period = lapses_pp_renov_period.drop(columns=cols_to_remove)


In [ ]:
lapses_pp_renov_period["FECHA_PREEXISTENCIA"] = lapses_pp_renov_period["FECHA_PREEXISTENCIA"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)
lapses_pp_renov_period["data_pre_existencia"] = lapses_pp_renov_period["data_pre_existencia"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)


In [ ]:
lapses_pp_renov_period["PRIANUALAA"] = lapses_pp_renov_period["PRIANUALAA"].fillna(lapses_pp_renov_period["aux_prem_aa"])
lapses_pp_renov_period["PRIANUALAC"] = lapses_pp_renov_period["PRIANUALAC"].fillna(lapses_pp_renov_period["aux_prem_ac"])


In [ ]:
mean_value = lapses_pp_renov_period["new_concentracao"].mean(skipna=True)
mask = lapses_pp_renov_period["new_concentracao"].isna() | (lapses_pp_renov_period["new_concentracao"] > 0.18)
lapses_pp_renov_period.loc[mask, "new_concentracao"] = mean_value

In [ ]:
comparison = lapses_pp_renov_period["premio2"] == lapses_pp_renov_period["PRIANUALAC"]
comparison

In [ ]:
mask = lapses_pp_renov_period["FALTAASEG"].apply(lambda x: pd.to_datetime(x) > pd.to_datetime("2024-12-31"))
a=lapses_pp_renov_period[mask][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year","FALTAASEG","tenure","dt_start","dt_end","is_lapse","premio2"]]

In [ ]:
renov_period

In [ ]:
#i still have some cases i need to deal with namelly the cases where the FALTASEG is in 2025 for this FVTOAC it does not make sense because they will not even have a total 9 months exposure (12-3) so i need to exclude the cases where (   FVTOAC - FALTASEG ) is less than 9 months 
# this is a better condition than just removing cases when Faltaseg is before 2024 because this automates the process when im chosing the renovation perioud 
mask = (pd.to_datetime(lapses_pp_renov_period["FVTO_AC"]) - pd.to_datetime(lapses_pp_renov_period["FALTAASEG"])).dt.days < 270
print(lapses_pp_renov_period[mask].shape)
lapses_pp_renov_period = lapses_pp_renov_period[~mask]


In [ ]:
lapses_pp_renov_period.to_parquet("lapses_pp_renov_period_before_removing_cols.parquet")

In [ ]:
#lets keep removing cols 
cols_to_remove = ["FALTAASEG","FNACI8","POLIZAS_45","FECHA_PREEXISTENCIA","FCob","fefecto_poliza","RENOVACAO","INICIO","FIM","EXPOSICAO","fecha","aux_prem_aa","aux_prem_ac","aux_ren","aux_gar","aux_ren_apol","prem_final","prem_exp","Ren2","ren_apolice2","Ren_final",
                  "ren_inicial","premio2","data_inicio2","dt_antiga","data_pre_existencia","aux_ren2","dif_meses","dt_start","dt_end","n_claims_dia2","EXPOSICAO_v2","exp_Hosp_Cirurgia_e_Parto","exp_DENTAL","exp_assist_port","exp_Cob_Medica_Int","exp_Consultas","exp_Medicamentos","final_premium","new_concentracao",
                  "Nr_Line","key","marca_inicio_new","marca_fim_new","ICART_DIA","APLICA","idade_bin","canceled"]

lapses_pp_renov_period = lapses_pp_renov_period.drop(columns=cols_to_remove)


In [ ]:
lapses_pp_renov_period["FECHA_PREEXISTENCIA"] = lapses_pp_renov_period["FECHA_PREEXISTENCIA"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)
lapses_pp_renov_period["data_pre_existencia"] = lapses_pp_renov_period["data_pre_existencia"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)


In [ ]:
lapses_pp_renov_period["PRIANUALAA"] = lapses_pp_renov_period["PRIANUALAA"].fillna(lapses_pp_renov_period["aux_prem_aa"])
lapses_pp_renov_period["PRIANUALAC"] = lapses_pp_renov_period["PRIANUALAC"].fillna(lapses_pp_renov_period["aux_prem_ac"])


In [ ]:
mean_value = lapses_pp_renov_period["new_concentracao"].mean(skipna=True)
mask = lapses_pp_renov_period["new_concentracao"].isna() | (lapses_pp_renov_period["new_concentracao"] > 0.18)
lapses_pp_renov_period.loc[mask, "new_concentracao"] = mean_value

In [ ]:
comparison = lapses_pp_renov_period["premio2"] == lapses_pp_renov_period["PRIANUALAC"]
comparison

In [ ]:
mask = lapses_pp_renov_period["FALTAASEG"].apply(lambda x: pd.to_datetime(x) > pd.to_datetime("2024-12-31"))
a=lapses_pp_renov_period[mask][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year","FALTAASEG","tenure","dt_start","dt_end","is_lapse","premio2"]]

In [ ]:
renov_period

In [ ]:
#i still have some cases i need to deal with namelly the cases where the FALTASEG is in 2025 for this FVTOAC it does not make sense because they will not even have a total 9 months exposure (12-3) so i need to exclude the cases where (   FVTOAC - FALTASEG ) is less than 9 months 
# this is a better condition than just removing cases when Faltaseg is before 2024 because this automates the process when im chosing the renovation perioud 
mask = (pd.to_datetime(lapses_pp_renov_period["FVTO_AC"]) - pd.to_datetime(lapses_pp_renov_period["FALTAASEG"])).dt.days < 270
print(lapses_pp_renov_period[mask].shape)
lapses_pp_renov_period = lapses_pp_renov_period[~mask]


In [ ]:
lapses_pp_renov_period.to_parquet("lapses_pp_renov_period_before_removing_cols.parquet")

In [ ]:
#lets keep removing cols 
cols_to_remove = ["FALTAASEG","FNACI8","POLIZAS_45","FECHA_PREEXISTENCIA","FCob","fefecto_poliza","RENOVACAO","INICIO","FIM","EXPOSICAO","fecha","aux_prem_aa","aux_prem_ac","aux_ren","aux_gar","aux_ren_apol","prem_final","prem_exp","Ren2","ren_apolice2","Ren_final",
                  "ren_inicial","premio2","data_inicio2","dt_antiga","data_pre_existencia","aux_ren2","dif_meses","dt_start","dt_end","n_claims_dia2","EXPOSICAO_v2","exp_Hosp_Cirurgia_e_Parto","exp_DENTAL","exp_assist_port","exp_Cob_Medica_Int","exp_Consultas","exp_Medicamentos","final_premium","new_concentracao",
                  "Nr_Line","key","marca_inicio_new","marca_fim_new","ICART_DIA","APLICA","idade_bin","canceled"]

lapses_pp_renov_period = lapses_pp_renov_period.drop(columns=cols_to_remove)


In [ ]:
lapses_pp_renov_period["FECHA_PREEXISTENCIA"] = lapses_pp_renov_period["FECHA_PREEXISTENCIA"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)
lapses_pp_renov_period["data_pre_existencia"] = lapses_pp_renov_period["data_pre_existencia"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)


In [ ]:
lapses_pp_renov_period["PRIANUALAA"] = lapses_pp_renov_period["PRIANUALAA"].fillna(lapses_pp_renov_period["aux_prem_aa"])
lapses_pp_renov_period["PRIANUALAC"] = lapses_pp_renov_period["PRIANUALAC"].fillna(lapses_pp_renov_period["aux_prem_ac"])


In [ ]:
mean_value = lapses_pp_renov_period["new_concentracao"].mean(skipna=True)
mask = lapses_pp_renov_period["new_concentracao"].isna() | (lapses_pp_renov_period["new_concentracao"] > 0.18)
lapses_pp_renov_period.loc[mask, "new_concentracao"] = mean_value

In [ ]:
comparison = lapses_pp_renov_period["premio2"] == lapses_pp_renov_period["PRIANUALAC"]
comparison

In [ ]:
mask = lapses_pp_renov_period["FALTAASEG"].apply(lambda x: pd.to_datetime(x) > pd.to_datetime("2024-12-31"))
a=lapses_pp_renov_period[mask][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year","FALTAASEG","tenure","dt_start","dt_end","is_lapse","premio2"]]

In [ ]:
renov_period

In [ ]:
#i still have some cases i need to deal with namelly the cases where the FALTASEG is in 2025 for this FVTOAC it does not make sense because they will not even have a total 9 months exposure (12-3) so i need to exclude the cases where (   FVTOAC - FALTASEG ) is less than 9 months 
# this is a better condition than just removing cases when Faltaseg is before 2024 because this automates the process when im chosing the renovation perioud 
mask = (pd.to_datetime(lapses_pp_renov_period["FVTO_AC"]) - pd.to_datetime(lapses_pp_renov_period["FALTAASEG"])).dt.days < 270
print(lapses_pp_renov_period[mask].shape)
lapses_pp_renov_period = lapses_pp_renov_period[~mask]


In [ ]:
lapses_pp_renov_period.to_parquet("lapses_pp_renov_period_before_removing_cols.parquet")

In [86]:
nan_percent_pp = (lapses_pp_renov_period.isna().mean() * 100).reset_index()
nan_percent_pp.columns = ['column', 'percent_nan']
nan_percent_pp['num_nan'] = lapses_pp_renov_period.isna().sum().values

In [87]:
a=lapses_pp_renov_period.loc[lapses_pp_renov_period["Concelho"].isna(), "codpost_tom"].unique()

In [88]:
import numpy as np
np.set_printoptions(threshold=np.inf)
print(a)

<ArrowStringArray>
[]
Length: 0, dtype: str


In [89]:
print(list(a))

[]


In [90]:
nan_percent_pp = (lapses_pp_renov_period.isna().mean() * 100).reset_index()
nan_percent_pp.columns = ['column', 'percent_nan']
nan_percent_pp['num_nan'] = lapses_pp_renov_period.isna().sum().values

In [91]:
# Value counts for is_lapse
print(lapses_pp_renov_period.loc[lapses_pp_renov_period["Concelho"].isna(), "is_lapse"].value_counts())

# Value counts for is_midterm
print(lapses_pp_renov_period.loc[lapses_pp_renov_period["Concelho"].isna(), "is_midterm"].value_counts())

Series([], Name: count, dtype: int64)
Series([], Name: count, dtype: int64)


In [92]:
cols_to_fill = ['Freguesia_Final_Pos_RATF', 'POP_densidade_populacao',
       'POP_relacao_masculinidade', 'POP_indice_envelhecimento',
       'POP_indice_dependencia_idosos', 'POP_durac_media_mov_pendular',
       'POP_taxa_desemprego', 'POP_ES_completo', 'POP_residente_idade_media',
       'POP_ADP_dim_media', 'POP_AI_dim_media', 'POP_NclF_monoparentais',
       'POP_NclF_com_filhos', 'POP_meio_transp_total_abs',
       'POP_PMT_index_trafego_auto', 'POP_meio_transp_index_saudavel',
       'POP_index_trafg_dens_auto', 'POP_index_trafg_dens_temp_auto',
       'POP_index_trafg_dens_saudavel', 'POP_ix_trafg_dens_temp_saudavel',
       'POP_index_trafg_dens_motjov', 'POP_index_trafg_dens_temp_motjov',
       'POP_idade01_0_14anos', 'POP_idade02_15_19anos',
       'POP_idade03_20_24anos', 'POP_idade04_25_29anos',
       'POP_idade05_30_34anos', 'POP_idade06_35_39anos',
       'POP_idade07_40_44anos', 'POP_idade08_45_49anos',
       'POP_idade09_50_54anos', 'POP_idade10_55_59anos',
       'POP_idade11_60_64anos', 'POP_idade12_65_69anos',
       'POP_idade13_70_74anos', 'POP_idade14_75_mais_anos',
       'POP_idade15_residente_total_abs', 'POP_ativa_ecivil01_solteiro',
       'POP_ativa_ecivil02_casado', 'POP_ativa_ecivil03_viuvo',
       'POP_ativa_ecivil04_divorciado', 'POP_NclF_nfilhos00_0_filhosEnt',
       'POP_NclF_nfilhos01_1_filhosEnt', 'POP_NclF_nfilhos02_2_filhosEnt',
       'POP_NclF_nfilhos03_3_filhosEnt', 'POP_NclF_nfilhos04_4_m_filhosEnt',
       'POP_NclF_nfilhos05_tot_absoluto', 'POL_ESAE01_educacao',
       'POL_ESAE02_artes_humanidades', 'POL_ESAE03_cienc_soc_jornal_info',
       'POL_ESAE04_cienc_empr_admn_dir', 'POL_ESAE05_cienc_natur_mat_esta',
       'POL_ESAE06_tecnolog_info_com', 'POL_ESAE07_engen_industri_const',
       'POL_ESAE08_agric_sivic_pesca_vet', 'POL_ESAE09_saude_protecao_social','POL_ESAE10_servicos', 'POP_emp_CAE24_Total_absoluto',
       'POP_emp_CAE01_Agric_PA_C_Fl_Pesc', 'POP_emp_CAE02_industr_extrativas',
       'POP_emp_CAE03_indust_transformad', 'POP_emp_CAE04_eletr_G_V_AQF_AF',
       'POP_emp_CAE05_CTD_agua_SGRD', 'POP_emp_CAE06_construcao',
       'POP_emp_CAE07_comrc_GR_repar_VAM', 'POP_emp_CAE08_transpo_armazenag',
       'POP_emp_CAE09_aloj_resta_simil', 'POP_emp_CAE10_actv_info_comunic',
       'POP_emp_CAE11_actv_finan_seguros', 'POP_emp_CAE12_actv_imabiliarias',
       'POP_emp_CAE13_actv_consulto_CTS', 'POP_emp_CAE14_act_adm_serv_apoio',
       'POP_emp_CAE15_adm_publ_def_SSO', 'POP_emp_CAE16_educacao',
       'POP_emp_CAE17_actv_saud_hum_AS', 'POP_emp_CAE18_actv_artist_EDR',
       'POP_emp_CAE19_outr_act_servc', 'POP_emp_CAE20_ac_fam_emp_PDAPFUP',
       'POP_emp_CAE21_act_org_int_OIET', 'POP_GSE01_emprs_prof_intelect_CT',
       'POP_GSE02_quadros_intelectuais_C', 'POP_GSE03_forcas_armadas',
       'POP_GSE04_trab_n_qualif_SP', 'POP_GSE05_operar_n_qualif',
       'POP_GSE06_trab_adm_comerc_SNQ', 'POP_GSE07_assalariados_SP',
       'POP_GSE08_operar_qualificados_SQ', 'POP_GSE09_emp_admin_comerc_serv',
       'POP_GSE10_quadros_admin_interm', 'POP_GSE11_quadros_tecn_interm',
       'POP_GSE12_diretor_quadros_DEMGE', 'POP_GSE13_emprs_ind_com_serv',
       'POP_GSE14_trab_independent_SP', 'POP_GSE15_prestad_serv_CI',
       'POP_GSE16_trab_industr_AI', 'POP_GSE17_prof_tecn_interm_I',
       'POP_GSE18_prof_intelc_cientf', 'POP_GSE19_pequenos_patroes_SP',
       'POP_GSE20_pequenos_patroes_CS', 'POP_GSE21_pequenos_patroes_I',
       'POP_GSE22_pequenos_patroes_PTI', 'POP_GSE23_pequenos_patroes_PIC',
       'POP_GSE24_emprs_SP', 'POP_GSE25_outras_pess_atv_NE',
       'POP_GSE26_pessoas_inativas', 'freg_ratf_github']  

for col in cols_to_fill:
    # Build a mapping from Concelho to the first non-NaN value in col
    value_map = lapses_pp_renov_period.groupby("Concelho")[col].apply(lambda x: x.dropna().iloc[0] if not x.dropna().empty else np.nan)
    # Fill NaNs in col using the mapping
    mask = lapses_pp_renov_period[col].isna() & lapses_pp_renov_period["Concelho"].notna()
    lapses_pp_renov_period.loc[mask, col] = lapses_pp_renov_period.loc[mask, "Concelho"].map(value_map)

In [93]:
# Dropping the rows where Concelho is still NaN after the imputation by the mode of the Concelho group
lapses_pp_renov_period.dropna(subset=["Concelho"], inplace=True)

In [94]:
nan_percent_pp = (lapses_pp_renov_period.isna().mean() * 100).reset_index()
nan_percent_pp.columns = ['column', 'percent_nan']
nan_percent_pp['num_nan'] = lapses_pp_renov_period.isna().sum().values

In [95]:
lapses_pp_renov_period.columns.get_loc("Avg_Number_Floors")

58

In [96]:
lapses_pp_renov_period.columns.get_loc("Concentracao")

75

In [97]:
lapses_pp_renov_period.columns[54:70]

Index(['codine_tomador', 'cp4', 'codine_tomador_n', 'cp7', 'Avg_Number_Floors',
       'Crime_rate_2011', 'Crime_rate_2012', 'Crime_rate_2013',
       'Crime_rate_2014', 'Crime_rate_2015', 'Crime_rate_Avg', 'zip_code',
       'households', 'inhabitants', 'purch_power_Euro', 'Distrito'],
      dtype='str')

In [98]:
cols_to_fill = ['Avg_Number_Floors', 'Crime_rate_2011', 'Crime_rate_2012',
       'Crime_rate_2013', 'Crime_rate_2014', 'Crime_rate_2015',
       'Crime_rate_Avg', 'households', 'inhabitants', 'purch_power_Euro',
       'Distrito', 'Concelho', 'N_Clientes', 'N_Prestadores', 'N_Domicilios',
       'Concentracao']  # Replace with your column names

for col in cols_to_fill:
    # Build a mapping from Concelho to the first non-NaN value in col
    value_map = lapses_pp_renov_period.groupby("Concelho")[col].apply(lambda x: x.dropna().iloc[0] if not x.dropna().empty else np.nan)
    # Fill NaNs in col using the mapping
    mask = lapses_pp_renov_period[col].isna() & lapses_pp_renov_period["Concelho"].notna()
    lapses_pp_renov_period.loc[mask, col] = lapses_pp_renov_period.loc[mask, "Concelho"].map(value_map)

In [99]:
# Get Concelho values where POP_emp_CAE16_educacao is NaN in lapses_pp_renov_period
nan_concelho = lapses_pp_renov_period.loc[
    lapses_pp_renov_period["POP_emp_CAE16_educacao"].isna(), "Concelho"
]

# Build a mapping from Concelho to the first non-NaN value in lapses
educacao_map = lapses.groupby("Concelho")["POP_emp_CAE16_educacao"].apply(
    lambda x: x.dropna().iloc[0] if not x.dropna().empty else np.nan
)

# Fill NaNs in lapses_pp_renov_period using the mapping
mask = lapses_pp_renov_period["POP_emp_CAE16_educacao"].isna() & lapses_pp_renov_period["Concelho"].notna()
lapses_pp_renov_period.loc[mask, "POP_emp_CAE16_educacao"] = lapses_pp_renov_period.loc[mask, "Concelho"].map(educacao_map)

In [ ]:
cols_to_fill = ['Freguesia_Final_Pos_RATF', 'POP_densidade_populacao',
       'POP_relacao_masculinidade', 'POP_indice_envelhecimento',
       'POP_indice_dependencia_idosos', 'POP_durac_media_mov_pendular',
       'POP_taxa_desemprego', 'POP_ES_completo', 'POP_residente_idade_media',
       'POP_ADP_dim_media', 'POP_AI_dim_media', 'POP_NclF_monoparentais',
       'POP_NclF_com_filhos', 'POP_meio_transp_total_abs',
       'POP_PMT_index_trafego_auto', 'POP_meio_transp_index_saudavel',
       'POP_index_trafg_dens_auto', 'POP_index_trafg_dens_temp_auto',
       'POP_index_trafg_dens_saudavel', 'POP_ix_trafg_dens_temp_saudavel',
       'POP_index_trafg_dens_motjov', 'POP_index_trafg_dens_temp_motjov',
       'POP_idade01_0_14anos', 'POP_idade02_15_19anos',
       'POP_idade03_20_24anos', 'POP_idade04_25_29anos',
       'POP_idade05_30_34anos', 'POP_idade06_35_39anos',
       'POP_idade07_40_44anos', 'POP_idade08_45_49anos',
       'POP_idade09_50_54anos', 'POP_idade10_55_59anos',
       'POP_idade11_60_64anos', 'POP_idade12_65_69anos',
       'POP_idade13_70_74anos', 'POP_idade14_75_mais_anos',
       'POP_idade15_residente_total_abs', 'POP_ativa_ecivil01_solteiro',
       'POP_ativa_ecivil02_casado', 'POP_ativa_ecivil03_viuvo',
       'POP_ativa_ecivil04_divorciado', 'POP_NclF_nfilhos00_0_filhosEnt',
       'POP_NclF_nfilhos01_1_filhosEnt', 'POP_NclF_nfilhos02_2_filhosEnt',
       'POP_NclF_nfilhos03_3_filhosEnt', 'POP_NclF_nfilhos04_4_m_filhosEnt',
       'POP_NclF_nfilhos05_tot_absoluto', 'POL_ESAE01_educacao',
       'POL_ESAE02_artes_humanidades', 'POL_ESAE03_cienc_soc_jornal_info',
       'POL_ESAE04_cienc_empr_admn_dir', 'POL_ESAE05_cienc_natur_mat_esta',
       'POL_ESAE06_tecnolog_info_com', 'POL_ESAE07_engen_industri_const',
       'POL_ESAE08_agric_sivic_pesca_vet', 'POL_ESAE09_saude_protecao_social','POL_ESAE10_servicos', 'POP_emp_CAE24_Total_absoluto',
       'POP_emp_CAE01_Agric_PA_C_Fl_Pesc', 'POP_emp_CAE02_industr_extrativas',
       'POP_emp_CAE03_indust_transformad', 'POP_emp_CAE04_eletr_G_V_AQF_AF',
       'POP_emp_CAE05_CTD_agua_SGRD', 'POP_emp_CAE06_construcao',
       'POP_emp_CAE07_comrc_GR_repar_VAM', 'POP_emp_CAE08_transpo_armazenag',
       'POP_emp_CAE09_aloj_resta_simil', 'POP_emp_CAE10_actv_info_comunic',
       'POP_emp_CAE11_actv_finan_seguros', 'POP_emp_CAE12_actv_imabiliarias',
       'POP_emp_CAE13_actv_consulto_CTS', 'POP_emp_CAE14_act_adm_serv_apoio',
       'POP_emp_CAE15_adm_publ_def_SSO', 'POP_emp_CAE16_educacao',
       'POP_emp_CAE17_actv_saud_hum_AS', 'POP_emp_CAE18_actv_artist_EDR',
       'POP_emp_CAE19_outr_act_servc', 'POP_emp_CAE20_ac_fam_emp_PDAPFUP',
       'POP_emp_CAE21_act_org_int_OIET', 'POP_GSE01_emprs_prof_intelect_CT',
       'POP_GSE02_quadros_intelectuais_C', 'POP_GSE03_forcas_armadas',
       'POP_GSE04_trab_n_qualif_SP', 'POP_GSE05_operar_n_qualif',
       'POP_GSE06_trab_adm_comerc_SNQ', 'POP_GSE07_assalariados_SP',
       'POP_GSE08_operar_qualificados_SQ', 'POP_GSE09_emp_admin_comerc_serv',
       'POP_GSE10_quadros_admin_interm', 'POP_GSE11_quadros_tecn_interm',
       'POP_GSE12_diretor_quadros_DEMGE', 'POP_GSE13_emprs_ind_com_serv',
       'POP_GSE14_trab_independent_SP', 'POP_GSE15_prestad_serv_CI',
       'POP_GSE16_trab_industr_AI', 'POP_GSE17_prof_tecn_interm_I',
       'POP_GSE18_prof_intelc_cientf', 'POP_GSE19_pequenos_patroes_SP',
       'POP_GSE20_pequenos_patroes_CS', 'POP_GSE21_pequenos_patroes_I',
       'POP_GSE22_pequenos_patroes_PTI', 'POP_GSE23_pequenos_patroes_PIC',
       'POP_GSE24_emprs_SP', 'POP_GSE25_outras_pess_atv_NE',
       'POP_GSE26_pessoas_inativas']  # Replace with your column names

for col in cols_to_fill:
    # Build mapping from Concelho to first non-NaN value in lapses for this column
    value_map = lapses.groupby("Concelho")[col].apply(lambda x: x.dropna().iloc[0] if not x.dropna().empty else np.nan)
    # Mask for rows in lapses_pp_renov_period where col is NaN and Concelho is not NaN
    mask = lapses_pp_renov_period[col].isna() & lapses_pp_renov_period["Concelho"].notna()
    # Fill NaNs in lapses_pp_renov_period using the mapping
    lapses_pp_renov_period.loc[mask, col] = lapses_pp_renov_period.loc[mask, "Concelho"].map(value_map)

In [ ]:
cols_to_fill = [ 'POP_densidade_populacao',
       'POP_relacao_masculinidade', 'POP_indice_envelhecimento',
       'POP_indice_dependencia_idosos', 'POP_durac_media_mov_pendular',
       'POP_taxa_desemprego', 'POP_ES_completo', 'POP_residente_idade_media',
       'POP_ADP_dim_media', 'POP_AI_dim_media', 'POP_NclF_monoparentais',
       'POP_NclF_com_filhos', 'POP_meio_transp_total_abs',
       'POP_PMT_index_trafego_auto', 'POP_meio_transp_index_saudavel',
       'POP_index_trafg_dens_auto', 'POP_index_trafg_dens_temp_auto',
       'POP_index_trafg_dens_saudavel', 'POP_ix_trafg_dens_temp_saudavel',
       'POP_index_trafg_dens_motjov', 'POP_index_trafg_dens_temp_motjov',
       'POP_idade01_0_14anos', 'POP_idade02_15_19anos',
       'POP_idade03_20_24anos', 'POP_idade04_25_29anos',
       'POP_idade05_30_34anos', 'POP_idade06_35_39anos',
       'POP_idade07_40_44anos', 'POP_idade08_45_49anos',
       'POP_idade09_50_54anos', 'POP_idade10_55_59anos',
       'POP_idade11_60_64anos', 'POP_idade12_65_69anos',
       'POP_idade13_70_74anos', 'POP_idade14_75_mais_anos',
       'POP_idade15_residente_total_abs', 'POP_ativa_ecivil01_solteiro',
       'POP_ativa_ecivil02_casado', 'POP_ativa_ecivil03_viuvo',
       'POP_ativa_ecivil04_divorciado', 'POP_NclF_nfilhos00_0_filhosEnt',
       'POP_NclF_nfilhos01_1_filhosEnt', 'POP_NclF_nfilhos02_2_filhosEnt',
       'POP_NclF_nfilhos03_3_filhosEnt', 'POP_NclF_nfilhos04_4_m_filhosEnt',
       'POP_NclF_nfilhos05_tot_absoluto', 'POL_ESAE01_educacao',
       'POL_ESAE02_artes_humanidades', 'POL_ESAE03_cienc_soc_jornal_info',
       'POL_ESAE04_cienc_empr_admn_dir', 'POL_ESAE05_cienc_natur_mat_esta',
       'POL_ESAE06_tecnolog_info_com', 'POL_ESAE07_engen_industri_const',
       'POL_ESAE08_agric_sivic_pesca_vet', 'POL_ESAE09_saude_protecao_social','POL_ESAE10_servicos', 'POP_emp_CAE24_Total_absoluto',
       'POP_emp_CAE01_Agric_PA_C_Fl_Pesc', 'POP_emp_CAE02_industr_extrativas',
       'POP_emp_CAE03_indust_transformad', 'POP_emp_CAE04_eletr_G_V_AQF_AF',
       'POP_emp_CAE05_CTD_agua_SGRD', 'POP_emp_CAE06_construcao',
       'POP_emp_CAE07_comrc_GR_repar_VAM', 'POP_emp_CAE08_transpo_armazenag',
       'POP_emp_CAE09_aloj_resta_simil', 'POP_emp_CAE10_actv_info_comunic',
       'POP_emp_CAE11_actv_finan_seguros', 'POP_emp_CAE12_actv_imabiliarias',
       'POP_emp_CAE13_actv_consulto_CTS', 'POP_emp_CAE14_act_adm_serv_apoio',
       'POP_emp_CAE15_adm_publ_def_SSO', 'POP_emp_CAE16_educacao',
       'POP_emp_CAE17_actv_saud_hum_AS', 'POP_emp_CAE18_actv_artist_EDR',
       'POP_emp_CAE19_outr_act_servc', 'POP_emp_CAE20_ac_fam_emp_PDAPFUP',
       'POP_emp_CAE21_act_org_int_OIET', 'POP_GSE01_emprs_prof_intelect_CT',
       'POP_GSE02_quadros_intelectuais_C', 'POP_GSE03_forcas_armadas',
       'POP_GSE04_trab_n_qualif_SP', 'POP_GSE05_operar_n_qualif',
       'POP_GSE06_trab_adm_comerc_SNQ', 'POP_GSE07_assalariados_SP',
       'POP_GSE08_operar_qualificados_SQ', 'POP_GSE09_emp_admin_comerc_serv',
       'POP_GSE10_quadros_admin_interm', 'POP_GSE11_quadros_tecn_interm',
       'POP_GSE12_diretor_quadros_DEMGE', 'POP_GSE13_emprs_ind_com_serv',
       'POP_GSE14_trab_independent_SP', 'POP_GSE15_prestad_serv_CI',
       'POP_GSE16_trab_industr_AI', 'POP_GSE17_prof_tecn_interm_I',
       'POP_GSE18_prof_intelc_cientf', 'POP_GSE19_pequenos_patroes_SP',
       'POP_GSE20_pequenos_patroes_CS', 'POP_GSE21_pequenos_patroes_I',
       'POP_GSE22_pequenos_patroes_PTI', 'POP_GSE23_pequenos_patroes_PIC',
       'POP_GSE24_emprs_SP', 'POP_GSE25_outras_pess_atv_NE',
       'POP_GSE26_pessoas_inativas']  # Replace with your column names

for col in cols_to_fill:
    # Build mapping from codpost_tom prefix (first 4 chars) to first non-NaN value in lapses
    prefix_map = lapses.groupby(lapses["codpost_tom"].astype(str).str[:4])[col].apply(
        lambda x: x.dropna().iloc[0] if not x.dropna().empty else np.nan
    )
    # Mask for rows in lapses_pp_renov_period where col is NaN and codpost_tom is not NaN
    mask = lapses_pp_renov_period[col].isna() & lapses_pp_renov_period["codpost_tom"].notna()
    # Get prefix for each row
    prefixes = lapses_pp_renov_period.loc[mask, "codpost_tom"].astype(str).str[:4]
    # Fill NaNs using the mapping
    lapses_pp_renov_period.loc[mask, col] = prefixes.map(prefix_map)

In [102]:
nan_percent_pp = (lapses_pp_renov_period.isna().mean() * 100).reset_index()
nan_percent_pp.columns = ['column', 'percent_nan']
nan_percent_pp['num_nan'] = lapses_pp_renov_period.isna().sum().values

In [103]:
unique_combinations = lapses[["DESCGARA", "garantia_label"]].drop_duplicates()
print(unique_combinations)

                   DESCGARA garantia_label
0                      Base         229-30
4                      Mais         229-35
82                    Extra         229-40
277                     NaN         229-35
1193    HOSPITALIZACAO BASE        229-451
1205        Standard [NOVO]        229-451
1562                    NaN         229-30
2166                    NaN         229-40
39365                   NaN        229-451
39366       Standard [NOVO]        229-452
59874                 Total         229-50
64273                   NaN         229-50
233798                Total         229-40


In [104]:
lapses_pp_renov_period.columns.get_loc("N_hospitais_PP2")

327

In [105]:
lapses_pp_renov_period.columns.get_loc("Social_susceptibility")

340

In [106]:
lapses_pp_renov_period[lapses_pp_renov_period["C_FVI"].isna()]["Concelho"].unique()

<ArrowStringArray>
[  'CAMARA DE LOBOS',           'FUNCHAL',        'SANTA CRUZ',
 'ANGRA DO HEROISMO',           'COIMBRA',     'RIBEIRA BRAVA',
       'SAO VICENTE',    'RIBEIRA GRANDE',           'CASCAIS',
      'PONTA DO SOL',   'FIGUEIRA DA FOZ',     'PONTA DELGADA',
       'PORTO SANTO',  'CALHETA (R.A.M.)',            'LISBOA',
           'ARGANIL',    'LAGOA (R.A.A.)',           'MACHICO']
Length: 18, dtype: str

In [107]:
lapses.columns.get_loc("N_hospitais_PP2")

327

In [108]:
lapses.columns.get_loc("Social_susceptibility")

340

In [109]:
lapses.columns[327:341]

Index(['N_hospitais_PP2', 'N_hospitais_Privado2', 'N_hospitais_Publico2',
       'N_hospitais_total2', 'N_Homens2', 'N_Mulheres2', 'N_pessoas_total2',
       'Prop_homens', 'Prop_mulheres', 'C_FVI', 'Exposure',
       'Physical_Susceptibility', 'Precipitation', 'Social_susceptibility'],
      dtype='str')

In [110]:
cols_to_fill = ['N_hospitais_PP2', 'N_hospitais_Privado2', 'N_hospitais_Publico2',
       'N_hospitais_total2', 'N_Homens2', 'N_Mulheres2', 'N_pessoas_total2',
       'Prop_homens', 'Prop_mulheres', 'C_FVI', 'Exposure',
       'Physical_Susceptibility', 'Precipitation', 'Social_susceptibility']  # Replace with your column names

for col in cols_to_fill:
    # Build mapping from codpost_tom prefix (first 4 chars) to first non-NaN value in lapses
    prefix_map = lapses.groupby(lapses["codpost_tom"].astype(str).str[:4])[col].apply(
        lambda x: x.dropna().iloc[0] if not x.dropna().empty else np.nan
    )
    # Mask for rows in lapses_pp_renov_period where col is NaN and codpost_tom is not NaN
    mask = lapses_pp_renov_period[col].isna() & lapses_pp_renov_period["codpost_tom"].notna()
    # Get prefix for each row
    prefixes = lapses_pp_renov_period.loc[mask, "codpost_tom"].astype(str).str[:4]
    # Fill NaNs using the mapping
    lapses_pp_renov_period.loc[mask, col] = prefixes.map(prefix_map)

In [111]:
cols_to_fill = ['N_hospitais_PP2', 'N_hospitais_Privado2', 'N_hospitais_Publico2',
       'N_hospitais_total2', 'N_Homens2', 'N_Mulheres2', 'N_pessoas_total2',
       'Prop_homens', 'Prop_mulheres', 'C_FVI', 'Exposure',
       'Physical_Susceptibility', 'Precipitation', 'Social_susceptibility']  # Replace with your column names

for col in cols_to_fill:
    # Build mapping from Concelho to first non-NaN value in lapses
    concelho_map = lapses.groupby("Concelho")[col].apply(
        lambda x: x.dropna().iloc[0] if not x.dropna().empty else np.nan
    )
    # Mask for rows in lapses_pp_renov_period where col is NaN and Concelho is not NaN
    mask = lapses_pp_renov_period[col].isna() & lapses_pp_renov_period["Concelho"].notna()
    # Fill NaNs using the mapping
    lapses_pp_renov_period.loc[mask, col] = lapses_pp_renov_period.loc[mask, "Concelho"].map(concelho_map)

In [112]:
cols_to_fill = ['N_hospitais_PP2', 'N_hospitais_Privado2', 'N_hospitais_Publico2',
       'N_hospitais_total2', 'N_Homens2', 'N_Mulheres2', 'N_pessoas_total2',
       'Prop_homens', 'Prop_mulheres', 'C_FVI', 'Exposure',
       'Physical_Susceptibility', 'Precipitation', 'Social_susceptibility']   # Replace with your column names

for col in cols_to_fill:
    for n in [3, 2, 1]:
        # Build mapping from codpost_tom prefix (first n chars) to first non-NaN value in lapses
        prefix_map = lapses.groupby(lapses["codpost_tom"].astype(str).str[:n])[col].apply(
            lambda x: x.dropna().iloc[0] if not x.dropna().empty else np.nan
        )
        # Mask for rows in lapses_pp_renov_period where col is NaN and codpost_tom is not NaN
        mask = lapses_pp_renov_period[col].isna() & lapses_pp_renov_period["codpost_tom"].notna()
        # Get prefix for each row
        prefixes = lapses_pp_renov_period.loc[mask, "codpost_tom"].astype(str).str[:n]
        # Fill NaNs using the mapping
        lapses_pp_renov_period.loc[mask, col] = prefixes.map(prefix_map)

In [113]:
#the ony nan values in the cols  C_FVI,Exposure,Physical_Susceptibility,Precipitation,Social_susceptibility are in the islands so i dont know whats the meaning  of C_FVI, or EXPOSURE the other cols i fill them with numbers that are enquadreted with the islands reality 

# Assign 5 to all NaN values in the Precipitation column because the only missing values are for the islands and the max value was 4 since the islands are the region with more rain i will input five
lapses_pp_renov_period["Precipitation"]=lapses_pp_renov_period['Precipitation'].fillna(5)
# this one is also only nan in the islands, so i took the max of all others and add one since for what i have researched the islands have btter health in general i have asigned 3
lapses_pp_renov_period['Physical_Susceptibility']=lapses_pp_renov_period['Physical_Susceptibility'].fillna(3)
#same case here Social_susceptibility measures stuf like how old the population is  and stuff like that since in the islands people tend to be younger than the older localities in portugal i saw the max and mean and took the midle value
lapses_pp_renov_period["Social_susceptibility"]=lapses_pp_renov_period["Social_susceptibility"].fillna(2)


In [114]:
lapses_pp_renov_period[["C_FVI","Exposure","Physical_Susceptibility","Precipitation","Social_susceptibility"]]
mean_value = lapses_pp_renov_period["C_FVI"].mean(skipna=True)
lapses_pp_renov_period["C_FVI"] = lapses_pp_renov_period["C_FVI"].fillna(mean_value)
mean_value = lapses_pp_renov_period["Exposure"].mean(skipna=True)
lapses_pp_renov_period["Exposure"] = lapses_pp_renov_period["Exposure"].fillna(mean_value)

In [115]:
nan_percent_pp = (lapses_pp_renov_period.isna().mean() * 100).reset_index()
nan_percent_pp.columns = ['column', 'percent_nan']
nan_percent_pp['num_nan'] = lapses_pp_renov_period.isna().sum().values

In [116]:
# there is a missmatch between the codes is garntia label and descgara, and since i do not want those cols in the model i will drop them 
#the cols Avg_Number_Floors ,Crime_rate_2011, Crime_rate_2012, Crime_rate_2013, Crime_rate_2014, Crime_rate_2015, Crime_rate_Avg have info related to 2011 to 2015 so if im analizing lapses of 2025 100 of data drift dont make sense 
#the cols freg_ratf_github has to detailed info if i imputted that as a cat col the model would ovrfit so i will retrive it, in addition the conc_ratf_github col is repeated because i already have that info
# the key_merge_comp does not make sense as well since it is a surrogatory key 

In [117]:
#removing columns that dont make sense to keep in the dataframe either because i have other cols with the sae info  or they are noot good  for the EDA
cols_to_drop = ['DESCGARA',"Avg_Number_Floors" ,"Crime_rate_2011", "Crime_rate_2012", "Crime_rate_2013", "Crime_rate_2014", "Crime_rate_2015", "Crime_rate_Avg","freg_ratf_github", 'conc_ratf_github','key_merge_comp']

lapses_pp_renov_period = lapses_pp_renov_period.drop(columns=cols_to_drop)


In [118]:
lapses_pp_renov_period["FECHA_PREEXISTENCIA"] = lapses_pp_renov_period["FECHA_PREEXISTENCIA"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)
lapses_pp_renov_period["data_pre_existencia"] = lapses_pp_renov_period["data_pre_existencia"].fillna(
    lapses_pp_renov_period["fefecto_poliza"]
)


In [119]:
lapses_pp_renov_period["PRIANUALAA"] = lapses_pp_renov_period["PRIANUALAA"].fillna(lapses_pp_renov_period["aux_prem_aa"])
lapses_pp_renov_period["PRIANUALAC"] = lapses_pp_renov_period["PRIANUALAC"].fillna(lapses_pp_renov_period["aux_prem_ac"])


In [120]:
mean_value = lapses_pp_renov_period["new_concentracao"].mean(skipna=True)
mask = lapses_pp_renov_period["new_concentracao"].isna() | (lapses_pp_renov_period["new_concentracao"] > 0.18)
lapses_pp_renov_period.loc[mask, "new_concentracao"] = mean_value

In [121]:
comparison = lapses_pp_renov_period["premio2"] == lapses_pp_renov_period["PRIANUALAC"]
comparison

0       True
1       True
2       True
3       True
4       True
        ... 
5397    True
5398    True
5399    True
5400    True
5401    True
Length: 5383, dtype: bool

In [ ]:
mask = lapses_pp_renov_period["FALTAASEG"].apply(lambda x: pd.to_datetime(x) > pd.to_datetime("2024-12-31"))
a=lapses_pp_renov_period[mask][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year","FALTAASEG","tenure","dt_start","dt_end","is_lapse","premio2"]]

In [123]:
renov_period

[datetime.date(2025, 4, 1), datetime.date(2025, 4, 30)]

In [124]:
#i still have some cases i need to deal with namelly the cases where the FALTASEG is in 2025 for this FVTOAC it does not make sense because they will not even have a total 9 months exposure (12-3) so i need to exclude the cases where (   FVTOAC - FALTASEG ) is less than 9 months 
# this is a better condition than just removing cases when Faltaseg is before 2024 because this automates the process when im chosing the renovation perioud 
mask = (pd.to_datetime(lapses_pp_renov_period["FVTO_AC"]) - pd.to_datetime(lapses_pp_renov_period["FALTAASEG"])).dt.days < 270
print(lapses_pp_renov_period[mask].shape)
lapses_pp_renov_period = lapses_pp_renov_period[~mask]


(29, 344)


In [125]:
lapses_pp_renov_period.to_parquet("lapses_pp_renov_period_before_removing_cols.parquet")

In [126]:
#lets keep removing cols 
cols_to_remove = ["FALTAASEG","FNACI8","POLIZAS_45","FECHA_PREEXISTENCIA","FCob","fefecto_poliza","RENOVACAO","INICIO","FIM","EXPOSICAO","fecha","aux_prem_aa","aux_prem_ac","aux_ren","aux_gar","aux_ren_apol","prem_final","prem_exp","Ren2","ren_apolice2","Ren_final",
                  "ren_inicial","premio2","data_inicio2","dt_antiga","data_pre_existencia","aux_ren2","dif_meses","dt_start","dt_end","n_claims_dia2","EXPOSICAO_v2","exp_Hosp_Cirurgia_e_Parto","exp_DENTAL","exp_assist_port","exp_Cob_Medica_Int","exp_Consultas","exp_Medicamentos","final_premium","new_concentracao",
                  "Nr_Line","key","marca_inicio_new","marca_fim_new","ICART_DIA","APLICA","idade_bin","canceled"]

lapses_pp_renov_period = lapses_pp_renov_period.drop(columns=cols_to_remove)


In [127]:
nan_percent_pp = (lapses_pp_renov_period.isna().mean() * 100).reset_index()
nan_percent_pp.columns = ['column', 'percent_nan']
nan_percent_pp['num_nan'] = lapses_pp_renov_period.isna().sum().values

In [128]:
#why is there people with 2 policies ususlly both indivudual it doesnt make sense 
has_duplicates = lapses_pp_renov_period["ASEGURADO"].duplicated(keep=False)
duplicate_rows = lapses_pp_renov_period[has_duplicates][["POLIZA", "ASEGURADO","fbajaaseg","IDADE","Tp_cliente"]].sort_values(by = "ASEGURADO")    
print(duplicate_rows)

Empty DataFrame
Columns: [POLIZA, ASEGURADO, fbajaaseg, IDADE, Tp_cliente]
Index: []


In [129]:
#still im gonna kepp both policies for the same pessoa segura and will do feature engenearing based on POLIZA and Assegurado 
lapses_pp_renov_period["key_PA"]= lapses_pp_renov_period["POLIZA"].astype(str) + "_" + lapses_pp_renov_period["ASEGURADO"].astype(str)+ "_" + lapses_pp_renov_period["NASEGURADO"].astype(str)

In [130]:
lapses_pp_renov_period.shape

(5354, 282)

# FEATURE ENGENIERING

In [131]:
# creating a dataframe with just the unique POLIZA and Pessoa Segura 
lapses_feature_eng =lapses_pp_renov_period[["key_PA","POLIZA","ASEGURADO","NASEGURADO","FVTO_AC"]].drop_duplicates().reset_index(drop=True)


In [132]:
lapses_feature_eng.FVTO_AC.dtype

dtype('O')

In [133]:
lapses_feature_eng.shape

(5354, 5)

In [134]:
lapses.FECHSTRO.dtype

dtype('O')

In [135]:
print(type(lapses_feature_eng['FVTO_AC'].iloc[0]))
print(type(lapses['FECHSTRO'].iloc[0]))

<class 'datetime.date'>
<class 'pandas.api.typing.NaTType'>


In [136]:
lapses["key_PA"] = lapses["POLIZA"].astype(str) + "_" + lapses["ASEGURADO"].astype(str)+ "_" + lapses["NASEGURADO"].astype(str)

In [137]:
print(lapses_feature_eng.columns)


Index(['key_PA', 'POLIZA', 'ASEGURADO', 'NASEGURADO', 'FVTO_AC'], dtype='str')


In [138]:
print(lapses_feature_eng[['key_PA', 'FVTO_AC']].head())

                  key_PA     FVTO_AC
0   204755164_11862752_1  2025-04-01
1  204755164_120807249_2  2025-04-01
2  204755164_120830196_3  2025-04-01
3  204755164_121548806_4  2025-04-01
4    204758776_4019200_1  2025-04-01


In [139]:
# Fast vectorized aggregation for lapses_feature_eng with correct tot_payments aggregation
filtered_lapses = lapses[lapses['key_PA'].isin(lapses_feature_eng['key_PA'])].copy()

# Ensure FVTO_AC and FECHSTRO are datetime
lapses_feature_eng['FVTO_AC'] = pd.to_datetime(lapses_feature_eng['FVTO_AC'])
filtered_lapses['FECHSTRO'] = pd.to_datetime(filtered_lapses['FECHSTRO'])

# Merge filtered_lapses with lapses_feature_eng to align FVTO_AC for each lapse
merged = filtered_lapses.merge(
    lapses_feature_eng[['key_PA', 'FVTO_AC']],
    on='key_PA',
    how='left'
)

# Use FVTO_AC_y for filtering due to column collision
merged = merged[merged['FECHSTRO'] <= merged['FVTO_AC_y']]

# Group and aggregate using FVTO_AC_y
agg = merged.groupby(['key_PA', 'FVTO_AC_y']).agg(
    tot_sinistros=('REFSTRO', 'nunique'),
    tot_amount_paid=('SINISTRALIDADE', 'sum'),
    tot_payments=('refstro2', 'nunique')  # Correct aggregation
).reset_index()

# Rename FVTO_AC_y to FVTO_AC for consistency
agg = agg.rename(columns={'FVTO_AC_y': 'FVTO_AC'})

# Merge back to lapses_feature_eng
lapses_feature_eng = lapses_feature_eng.merge(
    agg[['key_PA', 'FVTO_AC', 'tot_sinistros', 'tot_amount_paid', 'tot_payments']],
    on=['key_PA', 'FVTO_AC'],
    how='left'
)

# Fill NaN and convert types
lapses_feature_eng['tot_sinistros'] = lapses_feature_eng['tot_sinistros'].fillna(0).astype(int)
lapses_feature_eng['tot_amount_paid'] = lapses_feature_eng['tot_amount_paid'].fillna(0).astype(float).round(2)
lapses_feature_eng['tot_payments'] = lapses_feature_eng['tot_payments'].fillna(0).astype(int)



In [140]:
lapses_feature_eng["avg_payments_per_sinistro"] = (lapses_feature_eng["tot_payments"] / lapses_feature_eng["tot_sinistros"]).astype(float).round(2).replace([np.inf, -np.inf], 0).fillna(0)

In [141]:
lapses_feature_eng["avg_sinistralidade"] = (lapses_feature_eng["tot_amount_paid"] / lapses_feature_eng["tot_sinistros"]).astype(float).round(2).replace([np.inf, -np.inf], 0).fillna(0)

In [142]:
# List all objects in memory and their types/sizes
import sys
objs = [(k, type(v), sys.getsizeof(v)) for k, v in globals().items() if not k.startswith('__')]
objs_sorted = sorted(objs, key=lambda x: -x[2])
for name, typ, size in objs_sorted:
    print(f"{name:30} {str(typ):40} {size/1024/1024:.2f} MB")

lapses                         <class 'pandas.DataFrame'>               5894.15 MB
filtered_lapses                <class 'pandas.DataFrame'>               425.39 MB
merged                         <class 'pandas.DataFrame'>               285.76 MB
lapses_in_renov_period         <class 'pandas.DataFrame'>               120.53 MB
_75                            <class 'pandas.Series'>                  24.98 MB
lapses_pp_renov_period         <class 'pandas.DataFrame'>               13.16 MB
_45                            <class 'pandas.DataFrame'>               4.91 MB
scores                         <class 'pandas.DataFrame'>               3.82 MB
distrito_map                   <class 'pandas.Series'>                  2.06 MB
mask_concelho                  <class 'pandas.Series'>                  1.56 MB
mask_distrito                  <class 'pandas.Series'>                  1.56 MB
cond_lapse                     <class 'pandas.Series'>                  1.56 MB
cond_midterm                 

In [143]:
# Create a mask for FECHSTRO <= renov_period[1] (ensure types are compatible)
lapses['FECHSTRO'] = pd.to_datetime(lapses['FECHSTRO'])
mask_fechstro = lapses['FECHSTRO'] <= pd.Timestamp(renov_period[1])
mask_fechstro.sum(), mask_fechstro.shape

# Add REFSTRO notna condition to the mask
mask_fechstro_refstro = mask_fechstro & lapses['REFSTRO'].notna()
mask_fechstro_refstro.sum(), mask_fechstro_refstro.shape

(np.int64(1121312), (1638225,))

In [144]:
from tqdm import tqdm

# Ensure FECHSTRO is pandas datetime
lapses_prefiltered= lapses[mask_fechstro_refstro]

# Prepare columns
lapses_feature_eng['tot_sinistros_POLIZA'] = 0
lapses_feature_eng['tot_amount_paid_POLIZA'] = 0.0
lapses_feature_eng['tot_payments_POLIZA'] = 0

for idx, row in tqdm(lapses_feature_eng[['POLIZA', 'ASEGURADO', 'FVTO_AC']].drop_duplicates().iterrows(), total=lapses_feature_eng[['POLIZA', 'ASEGURADO', 'FVTO_AC']].drop_duplicates().shape[0]):
    fvto_ac_ts = pd.Timestamp(row['FVTO_AC'])
    mask = (
        (lapses_prefiltered['POLIZA'] == row['POLIZA']) &
        (lapses_prefiltered['FECHSTRO'] <= fvto_ac_ts)
    )
    subset = lapses_prefiltered[mask]
    lapses_feature_eng.loc[
        (lapses_feature_eng['POLIZA'] == row['POLIZA']) &
        (lapses_feature_eng['ASEGURADO'] == row['ASEGURADO']) &
        (lapses_feature_eng['FVTO_AC'] == row['FVTO_AC']),
        'tot_sinistros_POLIZA'] = subset['REFSTRO'].nunique()
    lapses_feature_eng.loc[
        (lapses_feature_eng['POLIZA'] == row['POLIZA']) &
        (lapses_feature_eng['ASEGURADO'] == row['ASEGURADO']) &
        (lapses_feature_eng['FVTO_AC'] == row['FVTO_AC']),
        'tot_amount_paid_POLIZA'] = subset['SINISTRALIDADE'].sum()
    lapses_feature_eng.loc[
        (lapses_feature_eng['POLIZA'] == row['POLIZA']) &
        (lapses_feature_eng['ASEGURADO'] == row['ASEGURADO']) &
        (lapses_feature_eng['FVTO_AC'] == row['FVTO_AC']),
        'tot_payments_POLIZA'] = subset['refstro2'].nunique()

# Fill NaN and convert types
lapses_feature_eng['tot_sinistros_POLIZA'] = lapses_feature_eng['tot_sinistros_POLIZA'].fillna(0).astype(int)
lapses_feature_eng['tot_amount_paid_POLIZA'] = lapses_feature_eng['tot_amount_paid_POLIZA'].fillna(0).astype(float).round(2)
lapses_feature_eng['tot_payments_POLIZA'] = lapses_feature_eng['tot_payments_POLIZA'].fillna(0).astype(int)

lapses_feature_eng[['POLIZA', 'ASEGURADO', 'FVTO_AC', 'tot_sinistros_POLIZA', 'tot_amount_paid_POLIZA', 'tot_payments_POLIZA']].head()

100%|██████████| 5354/5354 [01:58<00:00, 45.16it/s]


,POLIZA,ASEGURADO,FVTO_AC,tot_sinistros_POLIZA,tot_amount_paid_POLIZA,tot_payments_POLIZA
0,204755164,11862752,2025-04-01,94,7978.33,102
1,204755164,120807249,2025-04-01,94,7978.33,102
2,204755164,120830196,2025-04-01,94,7978.33,102
3,204755164,121548806,2025-04-01,94,7978.33,102
4,204758776,4019200,2025-04-01,83,6311.39,106


In [145]:
lapses_feature_eng.tot_sinistros_POLIZA.fillna(0).astype(int)
lapses_feature_eng.tot_payments_POLIZA.fillna(0).astype(int)
lapses_feature_eng.tot_amount_paid_POLIZA.fillna(0).astype(float).round(2)
lapses_feature_eng['tot_sinistros_POLIZA'] = lapses_feature_eng['tot_sinistros_POLIZA'].fillna(0).astype(int)
lapses_feature_eng['tot_payments_POLIZA'] = lapses_feature_eng['tot_payments_POLIZA'].fillna(0).astype(int)
lapses_feature_eng['tot_amount_paid_POLIZA'] = lapses_feature_eng['tot_amount_paid_POLIZA'].fillna(0).astype(float).round(2)    
lapses_feature_eng["avg_payments_per_sinistro_POLIZA"] = (lapses_feature_eng["tot_payments_POLIZA"] / lapses_feature_eng["tot_sinistros_POLIZA"]).astype(float).round(2).replace([np.inf, -np.inf], 0).fillna(0)
lapses_feature_eng["avg_sinistralidade_POLIZA"] = (lapses_feature_eng["tot_amount_paid_POLIZA"] / lapses_feature_eng["tot_sinistros_POLIZA"]).astype(float).round(2).replace([np.inf, -np.inf], 0).fillna(0)


In [146]:
def add_premio_per_exp_year(lapses_feature_eng, lapses):
    # Pivot lapses to get PRIANUALAC per p_exp_year for each (POLIZA, ASEGURADO)
    premio_pivot = (
        lapses.pivot_table(
            index=['POLIZA', 'ASEGURADO'],
            columns='p_exp_year',
            values='PRIANUALAC',
            aggfunc='first',  # or 'sum' if you want to sum values per year
            fill_value=0
        )
        .add_prefix('premio_')
        .reset_index()
    )
    # Exclude the last year column
    if lapses['p_exp_year'].notnull().any():
        max_year = lapses['p_exp_year'].max()
        col_to_drop = f'premio_{max_year}'
        if col_to_drop in premio_pivot.columns:
            premio_pivot = premio_pivot.drop(columns=[col_to_drop])

    # Merge with lapses_feature_eng
    result = lapses_feature_eng.merge(
        premio_pivot,
        on=['POLIZA', 'ASEGURADO'],
        how='left'
    )
    # Fill NaN with 0 for new premio columns
    premio_cols = [col for col in result.columns if col.startswith('premio_')]
    result[premio_cols] = result[premio_cols].fillna(0)
    return result

In [147]:
lapses_feature_eng = add_premio_per_exp_year(lapses_feature_eng, lapses)

In [148]:
lapses[lapses["ASEGURADO"]==11862752][["PRIANUALAC","p_exp_year"]].drop_duplicates().sort_values(by="p_exp_year")

,PRIANUALAC,p_exp_year
4989,385.42,2022
5005,439.47,2023
5013,496.34,2024
5039,532.55,2025


In [149]:
premio_cols = [col for col in lapses_feature_eng.columns if col.startswith('premio_')]
lapses_feature_eng["tot_prem_paid"] = lapses_feature_eng[premio_cols].sum(axis=1).astype(float).round(2)

In [150]:
lapses_feature_eng["LR"] = (  (lapses_feature_eng["tot_amount_paid"] / lapses_feature_eng["tot_prem_paid"])*100).astype(float).round(2).replace([np.inf, -np.inf], 0).fillna(0)

In [151]:


# Find all premio_xxxx columns and sort by year
premio_cols = sorted(
    [col for col in lapses_feature_eng.columns if re.match(r'premio_\\d{4}', col)],
    key=lambda x: int(x.split('_')[1])
)

# Calculate growth rates for consecutive years
for i in range(1, len(premio_cols)):
    prev_col = premio_cols[i-1]
    curr_col = premio_cols[i]
    growth_col = f'prem_inc_{curr_col.split("_")[1]}'
    lapses_feature_eng[growth_col] = (
        (lapses_feature_eng[curr_col] - lapses_feature_eng[prev_col]) /
        lapses_feature_eng[prev_col].replace(0, np.nan)
    ).replace([np.inf, -np.inf], 0).fillna(0).round(4)

In [152]:



# List of your premium columns in chronological order
premio_cols = [col for col in lapses_feature_eng.columns if col.startswith('premio_')]
premio_cols = sorted(premio_cols)  # Ensure columns are sorted by year

def calc_cagr(row):
    # Get nonzero premiums and their years
    premiums = row[premio_cols]
    nonzero = premiums[premiums != 0]
    if len(nonzero) < 2:
        return 0  # Not enough data to compute CAGR
    first_year = int(nonzero.index[0].split('_')[1])
    last_year = int(nonzero.index[-1].split('_')[1])
    n_years = last_year - first_year
    if n_years == 0 or nonzero.iloc[0] == 0:
        return 0
    cagr = (nonzero.iloc[-1] / nonzero.iloc[0]) ** (1 / n_years) - 1
    return round(cagr, 4)

lapses_feature_eng['cagr_premio'] = lapses_feature_eng.apply(calc_cagr, axis=1)

In [153]:
lapses_feature_eng[["POLIZA", "ASEGURADO","cagr_premio"]].sort_values(by="cagr_premio", ascending=True).head(20)

,POLIZA,ASEGURADO,cagr_premio
5348,207269988,120846441,0.0
4918,207196548,122716863,0.0
4949,207198243,121492053,0.0
5310,207220529,122755468,0.0
5309,207220529,121617039,0.0
5308,207220529,122702704,0.0
3027,206096663,122700395,0.0
5306,207220427,122727641,0.0
5305,207220181,122693519,0.0
5304,207220180,122508585,0.0


In [154]:
lapses[(lapses["POLIZA"]==207241391) &( lapses["ASEGURADO"]==122790793)][["POLIZA","ASEGURADO","PRIANUALAC","p_exp_year","DESCGARA","garantia_label","ICORR"]]

,POLIZA,ASEGURADO,PRIANUALAC,p_exp_year,DESCGARA,garantia_label,ICORR
1576861,207241391,122790793,120.63,2024,Base,229-30,100.0
1576862,207241391,122790793,57.52,2025,Base,229-30,104.0
1576863,207241391,122790793,57.52,2025,Base,229-30,104.0


In [155]:
#
# Find columns in lapses_feature_eng not in lapses_pp_renov_period (excluding 'key_PA')
new_cols = [col for col in lapses_feature_eng.columns if col not in lapses_pp_renov_period.columns and col != 'key_PA']

# Merge and add only those columns
lapses_pp_renov_period = lapses_pp_renov_period.merge(
    lapses_feature_eng[['key_PA'] + new_cols],
    on='key_PA',
    how='left'
)

In [156]:
# Count number of unique POLIZA per ASEGURADO
asegurado_counts = lapses_feature_eng.groupby("ASEGURADO")["POLIZA"].nunique()

# Map the count to the original DataFrame and flag as 1 if more than 1, else 0
lapses_feature_eng["more_then_1p"] = lapses_feature_eng["ASEGURADO"].map(lambda x: 1 if asegurado_counts[x] > 1 else 0)

In [157]:
#
# Find columns in lapses_feature_eng not in lapses_pp_renov_period (excluding 'key_PA')
new_cols = [col for col in lapses_feature_eng.columns if col not in lapses_pp_renov_period.columns and col != 'key_PA']

# Merge and add only those columns
lapses_pp_renov_period = lapses_pp_renov_period.merge(
    lapses_feature_eng[['key_PA'] + new_cols],
    on='key_PA',
    how='left'
)

In [158]:


# Ensure FECHSTRO and FVTO_AC are pandas Timestamps
lapses['FECHSTRO'] = pd.to_datetime(lapses['FECHSTRO'])
lapses_feature_eng['FVTO_AC'] = pd.to_datetime(lapses_feature_eng['FVTO_AC'])

# Identify all columns in lapses that start with 'cst_'
cst_cols = [col for col in lapses.columns if col.startswith('cst_')]

# Merge FVTO_AC into lapses for each POLIZA and ASEGURADO
merge_cols = ['POLIZA', 'ASEGURADO']
lapses_with_fvto = pd.merge(
    lapses,
    lapses_feature_eng[merge_cols + ['FVTO_AC']],
    on=merge_cols,
    how='left',
    suffixes=('', '_target')
)

# Filter rows where FECHSTRO is in [FVTO_AC - 1 year, FVTO_AC]
mask = (
    (lapses_with_fvto['FECHSTRO'] >= lapses_with_fvto['FVTO_AC'] - pd.DateOffset(years=1)) &
    (lapses_with_fvto['FECHSTRO'] <= lapses_with_fvto['FVTO_AC'])
)
filtered = lapses_with_fvto.loc[mask]

# Group by POLIZA, ASEGURADO, FVTO_AC and sum cst_ columns
agg = filtered.groupby(['POLIZA', 'ASEGURADO', 'FVTO_AC'])[cst_cols].sum().reset_index()

# Rename columns
agg = agg.rename(columns={col: f"{col}_tot_1_bef_ren" for col in cst_cols})

# Ensure FVTO_AC is datetime in both DataFrames before merging
lapses_feature_eng['FVTO_AC'] = pd.to_datetime(lapses_feature_eng['FVTO_AC'])
agg['FVTO_AC'] = pd.to_datetime(agg['FVTO_AC'])

# Merge back to lapses_feature_eng
lapses_feature_eng = lapses_feature_eng.merge(
    agg,
    on=['POLIZA', 'ASEGURADO', 'FVTO_AC'],
    how='left'
)

# Fill NaN with 0.0 if needed
for col in agg.columns:
    if col.endswith('_tot_1_bef_ren'):
        lapses_feature_eng[col] = lapses_feature_eng[col].fillna(0.0)

In [159]:
# Show cases where ASEGURADO appears more than once, sorted by ASEGURADO
cases = lapses_pp_renov_period[lapses_pp_renov_period['ASEGURADO'].duplicated(keep=False)]
cases = cases[['POLIZA', 'ASEGURADO', 'fbajaaseg', 'is_midterm', 'is_lapse', 'more_then_1p',"tenure","FVTO_AC"]].sort_values('ASEGURADO')
cases  # This will display the DataFrame in the notebook

,POLIZA,ASEGURADO,fbajaaseg,is_midterm,is_lapse,more_then_1p,tenure,FVTO_AC


In [160]:
#
# Find columns in lapses_feature_eng not in lapses_pp_renov_period (excluding 'key_PA')
new_cols = [col for col in lapses_feature_eng.columns if col not in lapses_pp_renov_period.columns and col != 'key_PA']

# Merge and add only those columns
lapses_pp_renov_period = lapses_pp_renov_period.merge(
    lapses_feature_eng[['key_PA'] + new_cols],
    on='key_PA',
    how='left'
)

# end of feature eng in the df lapses_eature_eng

In [161]:
lapses_pp_renov_period["proposed_prem_inc"]= (((lapses_pp_renov_period["PRIANUALAC"]/lapses_pp_renov_period["PRIANUALAA"])-1)*100).round(2).fillna(0).replace([np.inf, -np.inf], 0) 

In [162]:
lapses_pp_renov_period["PRIANUALAC_poliza"]= lapses_pp_renov_period.groupby("POLIZA")["PRIANUALAC"].transform('sum').round(2).fillna(0).replace([np.inf, -np.inf], 0) 
lapses_pp_renov_period["PRIANUALAA_poliza"]= lapses_pp_renov_period.groupby("POLIZA")["PRIANUALAA"].transform('sum').round(2).fillna(0).replace([np.inf, -np.inf], 0) 

In [163]:
lapses_pp_renov_period["proposed_prem_inc_poliza"]= (((lapses_pp_renov_period["PRIANUALAC_poliza"]/lapses_pp_renov_period["PRIANUALAA_poliza"])-1)*100).round(2).fillna(0).replace([np.inf, -np.inf], 0) 

In [164]:
# Assign Tp_familia using frozenset mapping for exact matches
mapping = {
    frozenset([0]): 0,
    frozenset([1]): 1,
    frozenset([3]): 2,
    frozenset([9]): 3,
    frozenset([0, 1]): 4,
    frozenset([0,3]): 5,
    frozenset([0, 9]): 6,
    frozenset([1,3]): 7,
    frozenset([1,9]): 8,
    frozenset([3,9]): 9,
    frozenset([0, 1, 3]): 10,
    frozenset([0,3,9]): 11,
    frozenset([0,1,9]): 12,
    frozenset([0,1,3,9]): 13
}

def get_tp_familia(parentescos):
    return mapping.get(frozenset(parentescos), -1)  # -1 for unmatched cases

lapses_pp_renov_period['Tp_familia'] = (
    lapses_pp_renov_period.groupby('POLIZA')['PARENTESCO']
    .transform(get_tp_familia)
)

In [165]:
cols= lapses_pp_renov_period.columns.tolist()




In [166]:
cols_to_drop = ['POLIZA',
 'NASEGURADO',
 'ASEGURADO',
 'FEFECTO',
 'TOMADOR',
 'FVTO_AC',
 'prem_inicial',
 'fbajaaseg',
 'p_exp_year',
 
 'Concelho',
 'Concentracao',
 'bin_Pre',
 'Bin_assist_port',
 'Bin_Cob_Medica_Int',
 'Bin_Consultas',
 'Bin_Medicamentos',
 'Freguesia_Final_Pos_RATF',
 'REFSTRO',
 'refstro2',
 'FECHSTRO',
 'SINISTRALIDADE',
 'cst_Ambul',
 'cst_Cesariana',
 'cst_Consulta_Domicilio',
 'cst_Consultas',
 'cst_Consultas_Urg',
 'cst_EAD_ECO',
 'cst_EAD_RM',
 'cst_EAD_TAC',
 'cst_EADs_Analises',
 'cst_EADs_AnatomiaPat',
 'cst_EADs_Endo',
 'cst_EADs_MedNuclear',
 'cst_EADs_Outros',
 'cst_EADs_RX',
 'cst_ERRO',
 'cst_Estomat',
 'cst_Estomat_Consult_e_Trat',
 'cst_Estomat_Prot',
 'cst_Fisio',
 'cst_Hosp_Cirurgia',
 'cst_Hosp_Cirurgia_e_Parto',
 'cst_Lesoes_Benig_Pele',
 'cst_Lesoes_Malig_Pele',
 'cst_Onc_Pacote_Total',
 'cst_Medicamentos',
 'cst_NOT_HOSP',
 'cst_Parto_Norm_e_IIG',
 'cst_Prot_Aros',
 'cst_Prot_Lentes',
 'cst_Prot_Lentes_contacto',
 'cst_Prot_Oculares',
 'cst_Prot_n_Oculares',
 'cst_Quimio_Radio',
 'cst_Subs_Desloc',
 'cst_Subs_Hosp',
 'cst_Tratamentos',
 'cont_Ambul',
 'cont_Cesariana',
 'cont_Consulta_Domicilio',
 'cont_Consultas',
 'cont_Consultas_Urg',
 'cont_EAD_ECO',
 'cont_EAD_RM',
 'cont_EAD_TAC',
 'cont_EADs_Analises',
 'cont_EADs_AnatomiaPat',
 'cont_EADs_Endo',
 'cont_EADs_MedNuclear',
 'cont_EADs_Outros',
 'cont_EADs_RX',
 'cont_ERRO',
 'cont_Estomat',
 'cont_Estomat_Consult_e_Trat',
 'cont_Estomat_Prot',
 'cont_Fisio',
 'cont_Hosp_Cirurgia',
 'cont_Hosp_Cirurgia_e_Parto',
 'cont_Lesoes_Benig_Pele',
 'cont_Lesoes_Malig_Pele',
 'cont_Medicamentos',
 'cont_NOT_HOSP',
 'cont_Parto_Norm_e_IIG',
 'cont_Prot_Aros',
 'cont_Prot_Lentes',
 'cont_Prot_Lentes_contacto',
 'cont_Prot_Oculares',
 'cont_Prot_n_Oculares',
 'cont_Quimio_Radio',
 'cont_Subs_Desloc',
 'cont_Subs_Hosp',
 'cont_Tratamentos',
 'n_claims_dia',
 'nclaim_total',
 'POLIZASNP',
 'new_nif_tomador',
 'is_midterm',
 'key_PA',
 'excess_Hosp_Cirurgia',
 'cst_Hosp_Cirurgia_v2',
 'cont_LL_Hosp',
 'cst_Estomat_Prot_total',
 'cst_Estomat_Prot_partial',
 'cont_total_est_prot',
 'cont_partial_est_prot',
 'garantia_label',
 'key_PA',"Bin_Hosp_Cirurgia_e_Parto",
 "Bin_DENTAL"
 
 ]

In [167]:
data= lapses_pp_renov_period.drop(columns=cols_to_drop)

In [168]:
#eleminate constant cols
const_cols = [col for col in data.columns if data[col].nunique() == 1]
data = data.drop(const_cols, axis=1)

In [169]:
data["proposed_prem_inc"] = ((data["PRIANUALAC"] / data["PRIANUALAA"] - 1) * 100).round(2)

In [170]:
cst_cols = [col for col in data.columns if col.startswith("cst_")]
data["total_cst_1_yb"] = data[cst_cols].sum(axis=1)

In [171]:
cols=data.columns

In [172]:
lapses[lapses["POLIZA"]==205020135][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year"]]

,POLIZA,NASEGURADO,ASEGURADO,PRIANUALAA,PRIANUALAC,p_exp_year
97404,205020135,1,120960624,744.75,1130.74,2022
97405,205020135,1,120960624,744.75,1130.74,2022
97406,205020135,1,120960624,744.75,1130.74,2022
97407,205020135,1,120960624,744.75,1130.74,2022
97408,205020135,1,120960624,744.75,1130.74,2022
97409,205020135,1,120960624,744.75,1130.74,2022
97410,205020135,1,120960624,744.75,1130.74,2022
97411,205020135,1,120960624,744.75,1130.74,2022
97412,205020135,1,120960624,744.75,1130.74,2022
97413,205020135,1,120960624,744.75,1130.74,2022


# EDA

as im going to the vars i will solve data inconsistencies in the cell bellow 

In [173]:
data["C_FVI"] = data["C_FVI"].astype(int)# the only non int instances where the ones that i substituted by the mean 
data["Exposure"] = data["Exposure"].astype(int)# the only non int instances where the ones that i substituted by the mean 
data["Physical_Susceptibility"] = data["Physical_Susceptibility"].astype(int) # its already int but its stored as a float 
data["Precipitation"] = data["Precipitation"].astype(int) # its already int but its stored as a float 
data["Social_susceptibility"] = data["Social_susceptibility"].astype(int) # its already int but its stored as a float 

In [174]:
data.shape

(5354, 221)

in the data df i have 230 cols i have variables that contain info about the individual characteristics of  the pessoa segura,  the characteristics of the policy at individual and group level, i have a lot of socio demographic variables, but i do doubt about the veracity of some of those in the future i might try to generate some  socioeconomical variables to complete that, since i have so many of those i will try to do PCA on all of them and reduce them into a smaller number of principal components, to ease out model calculations 

In [175]:
# total number of lapses 
total_lapses= data['is_lapse'].sum()
total_policies= data.shape[0]
lapse_rate = (total_lapses / total_policies) * 100
print(f"Total Lapses: {total_lapses}")
print(f"Total Policies: {total_policies}")  
print(f"Lapse Rate: {lapse_rate:.2f}%")

Total Lapses: 316
Total Policies: 5354
Lapse Rate: 5.90%


stil 230 cols is a lot before encoding so i need to reduce the number of features to do that i will 
drop : Near-Constant or Zero-Variance Features
drop the premium history since premiums( nominal ) are highly correlated trughout years so i will hust leve the proposed premium, and the increase rates as well as the CAGR the thing is since the majority of policies only have 1 policy holder i dont know  if its better to leave both the premium per pessoa segura and poliza or leave just the pessoa segura because they will be extremelly correlated

now im gonna separate into categorical and numerical variables and plot the lase rate for each variable 


In [176]:
num_cols = (["PESO","ICORR","PRIANUALAC","PRIANUALAA","IDADE","households","inhabitants","purch_power_Euro","N_Clientes","N_Prestadores","N_Domicilios"] 
+data.columns[46:150].to_list() +
["new_estatura","IMC", "N_Homens2","N_Mulheres2","N_pessoas_total2","Prop_homens","Prop_mulheres",
"tot_sins","tot_payments","tot_amount_paid","avg_payments_per_sinistro","avg_sinistralidade", "tot_sins_POLIZA","tot_payments_POLIZA","tot_amount_paid_POLIZA","avg_payments_per_sinistro_POLIZA",
"avg_sinistralidade_POLIZA","premio_2022","premio_2023","premio_2024","tot_prem_paid","LR","cagr_premio"] +
data.columns[188:223].to_list() +["PRIANUALAC_poliza","PRIANUALAA_poliza","proposed_prem_inc_poliza","proposed_prem_inc","total_cst_1_yb"])



                                                                                                                                                                     
                                                                                                                                                                     
                                                                                                                                                                     
                                                                                                                                                                     
                                                                                                                                                                     
                                                                                                                                                                     
                                                                                                                                                                     
                                                                                                                                                                     
                                                                                                                                                                     
                                                                                                                                                                     




cat_cols = ["SEXO","PARENTESCO","TRANSF_SEGURO_INT", "TRANSF_SEGURO_EXT","FP","n_pessoas_ren2","tenure","new_ant_pessoa","Distrito","Chapter_6","Chapter_17","Chapter_13","Chapter_7","Chapter_10","Chapter_16","Chapter_9","Chapter_12","Chapter_8","Chapter_3","Chapter_2","Chapter_4","Chapter_14","Chapter_5"
,"Chapter_1","Chapter_11","Capital_Hosp_Cirurgia_e_Parto","Capital_Parto","Capital_Hosp_Cirurgia","Capital_Prot_n_Oculares","Capital_Prot_Oculares","Capital_Cob_Medica_Int","Capital_Consultas","Capital_Medicamentos","Capital_Estomat_Consult_e_Trat","Cap_Oncologia","Tp_cliente",
"N_hospitais_PP2","N_hospitais_Privado2","N_hospitais_Publico2","N_hospitais_total2","C_FVI","Exposure","Physical_Susceptibility","Precipitation","Social_susceptibility","score_final","ICART_MES","is_tomador","more_then_1p","Tp_familia"]

# seing the cases with no PRIANUALAA


In [177]:
pp_no_PRIANUALAA= lapses_pp_renov_period[lapses_pp_renov_period["PRIANUALAA"]== 0][["POLIZA","ASEGURADO"]]
# Get unique (POLIZA, ASEGURADO) pairs as a set of tuples
# Get unique (POLIZA, ASEGURADO) pairs
unique_combinations = pp_no_PRIANUALAA[["POLIZA", "ASEGURADO"]].drop_duplicates()

# Merge to filter lapses
filtered_lapses = lapses.merge(unique_combinations, on=["POLIZA", "ASEGURADO"], how="inner")


#i checked that for these cases where PRIANUALAA is 0 i relly do not have information so even in the lapses df i  only have 0 in the PRIANUALAA col so to deal with that i will substitute the premium increaase col into the average of the premium increase for the lapse category 

In [178]:
lapses[(lapses["POLIZA"]==205020135) ][["POLIZA","NASEGURADO","ASEGURADO","PRIANUALAA","PRIANUALAC","p_exp_year","FALTAASEG","tenure","dt_start","dt_end","is_lapse","premio2"]]

,POLIZA,NASEGURADO,ASEGURADO,PRIANUALAA,PRIANUALAC,p_exp_year,FALTAASEG,tenure,dt_start,dt_end,is_lapse,premio2
97404,205020135,1,120960624,744.75,1130.74,2022,2018-09-14,3,2022-01-01,2022-06-09,0,744.75
97405,205020135,1,120960624,744.75,1130.74,2022,2018-09-14,3,2022-06-10,2022-06-24,0,744.75
97406,205020135,1,120960624,744.75,1130.74,2022,2018-09-14,3,2022-06-25,2022-06-28,0,744.75
97407,205020135,1,120960624,744.75,1130.74,2022,2018-09-14,3,2022-06-28,2022-06-28,0,744.75
97408,205020135,1,120960624,744.75,1130.74,2022,2018-09-14,3,2022-06-28,2022-06-28,0,744.75
97409,205020135,1,120960624,744.75,1130.74,2022,2018-09-14,3,2022-06-29,2022-08-31,0,744.75
97410,205020135,1,120960624,744.75,1130.74,2022,2018-09-14,4,2022-09-01,2022-09-08,0,1130.74
97411,205020135,1,120960624,744.75,1130.74,2022,2018-09-14,4,2022-09-09,2022-09-16,0,1130.74
97412,205020135,1,120960624,744.75,1130.74,2022,2018-09-14,4,2022-09-16,2022-09-16,0,1130.74
97413,205020135,1,120960624,744.75,1130.74,2022,2018-09-14,4,2022-09-16,2022-09-16,0,1130.74


In [179]:
data.Distrito.value_counts()

Distrito
LISBOA                 1718
PORTO                   640
SETUBAL                 536
FARO                    500
BRAGA                   410
AVEIRO                  367
LEIRIA                  212
SANTAREM                173
VISEU                   157
VIANA_DO_CASTELO        131
COIMBRA                 114
ILHA_DA_MADEIRA          99
BEJA                     65
EVORA                    61
GUARDA                   55
VILA_REAL                39
CASTELO_BRANCO           38
PORTALEGRE               16
ILHA_DE_SÃO_MIGUEL      12
BRAGANÃA                 7
ILHA_TERCEIRA             3
ILHA_DE_PORTO_SANTO       1
Name: count, dtype: int64

In [180]:
# Create a single mapping dictionary im gonna do this because it doest make sense to have all these categoris if there are ctegories with so few obs 
distrito_map = {
    "ILHA_DO_PICO": "AÇORES",
    "ILHA_GRACIOSA": "AÇORES",
    "ILHA_TERCEIRA": "AÇORES",
    "ILHA_DO_FAIAL": "AÇORES",
    "ILHA_DE_SÃO_MIGUEL": "AÇORES",
    "ILHA_DE_PORTO_SANTO": "ILHA_DA_MADEIRA",
    "BRAGANÃA": "VILA_REAL",
    "PORTALEGRE": "CASTELO_BRANCO"
}

# Apply the mapping
data['Distrito'] = data['Distrito'].replace(distrito_map)

In [181]:
data.Distrito.value_counts()

Distrito
LISBOA              1718
PORTO                640
SETUBAL              536
FARO                 500
BRAGA                410
AVEIRO               367
LEIRIA               212
SANTAREM             173
VISEU                157
VIANA_DO_CASTELO     131
COIMBRA              114
ILHA_DA_MADEIRA      100
BEJA                  65
EVORA                 61
GUARDA                55
CASTELO_BRANCO        54
VILA_REAL             46
AÇORES                15
Name: count, dtype: int64

In [182]:
data.Tp_familia.value_counts()


Tp_familia
0     2353
10    1455
5      959
4      552
2       15
6       10
12       3
11       3
7        2
1        2
Name: count, dtype: int64

In [183]:
# Value counts for Tp_familia
value_counts = data.Tp_familia.value_counts()

# Number of lapses per Tp_familia
lapses_count = data[data.is_lapse == 1].Tp_familia.value_counts()

# Percentage of lapses per Tp_familia
lapses_percentage = (lapses_count / value_counts * 100).round(2)

# Combine results into a DataFrame
result = pd.DataFrame({
    'Total': value_counts,
    'Lapses': lapses_count,
    'Lapse %': lapses_percentage
}).fillna(0)

result

,Total,Lapses,Lapse %
Tp_familia,,,
0,2353,128.0,5.44
1,2,0.0,0.00
2,15,0.0,0.00
4,552,20.0,3.62
5,959,57.0,5.94
6,10,0.0,0.00
7,2,0.0,0.00
10,1455,111.0,7.63
11,3,0.0,0.00


 # im also gonna convert this to have less categories 
# categories
0 = only the tomador
1= only conjuge
2= only the kids
3= other person not tomador conj or kids
4= couple(tomador + conj)
5=tomador + kids
6= tomador + other person
7= conj + kids
8= conj + other person
9= kids + other person
10= tomador+ conj+kids
11= tomador + kids+ other person
12= tomador+ conj + other person
13 = tudo 

im gonna join the categories 3 + 6 +11 +12 + 13 because these cat have low occurence and no lapses  then im also gonna group the cat 1+2+7

to do this for each group of agregation i will use the smalest number within the group as the new label

In [184]:
mask1 = data["Tp_familia"].isin([3, 6, 11, 12, 13])
mask2 = data["Tp_familia"].isin([1,2,7])

data.loc[mask1, "Tp_familia"] = 3
data.loc[mask2, "Tp_familia"] = 1

In [185]:
# this code will automatically detect all the diferent combinations of coverages create a dictionary with keys from 1 to lent of the number of dif combinstions and values a string with all the coverages then apply the lable to each row this way i will  escape from de redundancy and poor dat inserction of DESCGARA and the garantia lable and the model will prioritize the amount of the coverages



# Get all unique combinations
capital_combinations = data[[
    "Capital_Hosp_Cirurgia_e_Parto", "Capital_Parto", "Capital_Hosp_Cirurgia",
    "Capital_Prot_n_Oculares", "Capital_Prot_Oculares", "Capital_Cob_Medica_Int",
    "Capital_Consultas", "Capital_Medicamentos", "Capital_Estomat_Consult_e_Trat",
    "Cap_Oncologia"
]].drop_duplicates().reset_index(drop=True)

# Create dictionary: key = number, value = description
combination_dict = {}
for idx, row in capital_combinations.iterrows():
    desc = ', '.join([f"{col}: {row[col]}" for col in capital_combinations.columns])
    combination_dict[idx + 1] = desc

# Map each row in data to its combination number
def get_combination_number(row):
    for idx, comb_row in capital_combinations.iterrows():
        if all(row[col] == comb_row[col] for col in capital_combinations.columns):
            return idx + 1
    return None

data['type_coverage'] = data.apply(get_combination_number, axis=1)

# Show the dictionary and the updated DataFrame
combination_dict
data.head()

,SEXO,PARENTESCO,PESO,TRANSF_SEGURO_INT,TRANSF_SEGURO_EXT,ICORR,PRIANUALAC,PRIANUALAA,FP,IDADE,...,cst_Subs_Hosp_tot_1_bef_ren,cst_Tratamentos_tot_1_bef_ren,cst_Hosp_Cirurgia_v2_tot_1_bef_ren,proposed_prem_inc,PRIANUALAC_poliza,PRIANUALAA_poliza,proposed_prem_inc_poliza,Tp_familia,total_cst_1_yb,type_coverage
0,H,0,72.0,N,S,118.0,532.55,496.34,6,31,...,0.0,0.0,643.7,7.30,2015.58,1903.78,5.87,10,1514.25,1
1,V,1,60.0,N,N,120.0,541.58,505.48,6,33,...,0.0,0.0,0.0,7.14,2015.58,1903.78,5.87,10,0.00,1
2,V,3,15.0,N,S,104.0,442.62,425.60,6,9,...,0.0,0.0,0.0,4.00,2015.58,1903.78,5.87,10,21.50,2
3,H,3,3.0,N,N,111.0,498.83,476.36,6,3,...,0.0,0.0,0.0,4.72,2015.58,1903.78,5.87,10,40.10,2
4,H,0,51.0,N,S,106.0,1228.41,1170.88,6,46,...,0.0,0.0,0.0,4.91,1228.41,1170.88,4.91,0,317.06,2


In [186]:
#still ledt with 5 cases of pessoas seguras with no npessoas ren 2 
#lets drop them 
data = data.dropna(subset=['n_pessoas_ren2'])

In [187]:
#lets check the percentage of lapses in the dataset

amout_lapses = data["is_lapse"].sum()
percentage_lapses = amout_lapses / len(data) * 100  
nolapses = len(data) - amout_lapses
print(f"Total Lapses: {amout_lapses}")
print(f"Total No Lapses: {nolapses}")       
print(f"Percentage of Lapses: {percentage_lapses:.2f}%")

Total Lapses: 316
Total No Lapses: 5038
Percentage of Lapses: 5.90%


In [188]:
nan_percent_data = (data.isna().mean() * 100).reset_index()
nan_percent_data.columns = ['column', 'percent_nan']
nan_percent_data['num_nan'] = data.isna().sum().values

# saving the data df into  a parquet file

In [189]:
data.to_parquet("renovaçoes_dia_31_05_2025.parquet")